<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/09_SPP_GAN_Statistical_Guidance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==================================================================================================
# NOTEBOOK 09 — SPP-GAN STATISTICAL GUIDANCE
# ==================================================================================================
#
# Research:
#   A Unified Privacy-Preserving Framework for High-Fidelity Synthetic Data Generation
#   Using Statistical and Machine Learning Models
#
# Framework:
#   SPP-GAN — Statistical-Guided Privacy-Preserving GAN
#
# Purpose:
#   Define, validate, serialize, and test the differentiable statistical-guidance module
#   used by the SPP-GAN generator objective.
#
# IMPORTANT:
#   - This notebook DOES NOT train SPP-GAN.
#   - This notebook DOES NOT enable Differential Privacy.
#   - This notebook DOES NOT generate final synthetic datasets.
#   - Training is performed later in Notebook 12.
#
# Frozen dependencies:
#   Notebook 00 — Environment, Configuration & Reproducibility
#   Notebook 02 — Preprocessing, Encoding & Data Splits
#   Notebook 03 — Statistical & Data Characterization
#   Notebook 08 — SPP-GAN Architecture
#
# Downstream:
#   Notebook 12 — SPP-GAN Training
#
# ==================================================================================================

In [ ]:
# ==================================================================================================
# 1. HEADER & SCOPE
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import json
import math
import hashlib
import random
import warnings
import traceback

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

print("=" * 100)
print("NOTEBOOK 09 — SPP-GAN STATISTICAL GUIDANCE")
print("=" * 100)

NOTEBOOK_ID = "09"
NOTEBOOK_NAME = "SPP-GAN Statistical Guidance"
FRAMEWORK_NAME = "SPP-GAN"

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

NB02_ROOT = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "notebook_02"
)

NB03_ROOT = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "notebook_03"
)

NB08_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_08"
)

NB09_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_09"
)

DIRS = {
    "root": NB09_ROOT,
    "guidance": NB09_ROOT / "guidance",
    "loss": NB09_ROOT / "loss",
    "configuration": NB09_ROOT / "configuration",
    "validation": NB09_ROOT / "validation",
    "metadata": NB09_ROOT / "metadata",
    "models": NB09_ROOT / "models",
}

for path in DIRS.values():
    path.mkdir(
        parents=True,
        exist_ok=True
    )

print(f"Project root : {PROJECT_ROOT}")
print(f"Notebook root: {NB09_ROOT}")
print()
print("Scope:")
print("  Statistical guidance definition : YES")
print("  Statistical loss definition     : YES")
print("  Gradient validation             : YES")
print("  SPP-GAN training                : NO")
print("  Differential Privacy            : NO")
print("  Synthetic dataset generation    : NO")

NOTEBOOK 09 — SPP-GAN STATISTICAL GUIDANCE
Project root : /content/drive/MyDrive/SPP_GAN_Research
Notebook root: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09

Scope:
  Statistical guidance definition : YES
  Statistical loss definition     : YES
  Gradient validation             : YES
  SPP-GAN training                : NO
  Differential Privacy            : NO
  Synthetic dataset generation    : NO


In [ ]:
# ==================================================================================================
# NOTEBOOK 09 — GOOGLE DRIVE INITIALIZATION
# ==================================================================================================

from pathlib import Path
import os
import shutil

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"

print("=" * 100)
print("NOTEBOOK 09 — GOOGLE DRIVE INITIALIZATION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Ensure /content/drive is a clean mountpoint
# --------------------------------------------------------------------------------------------------

if DRIVE_ROOT.exists():

    # If MyDrive is already visible, Drive is likely mounted.
    if MYDRIVE_ROOT.exists():

        print("\n✓ /content/drive/MyDrive already exists.")

    else:

        # A non-empty /content/drive without MyDrive is not a valid
        # Google Drive mount.
        existing_items = list(DRIVE_ROOT.iterdir())

        if existing_items:

            print("\n⚠ /content/drive exists but is not a valid Drive mount.")
            print("  Cleaning local mountpoint...")

            for item in existing_items:
                if item.is_dir() and not item.is_symlink():
                    shutil.rmtree(item)
                else:
                    item.unlink()

            print("✓ Local mountpoint cleaned.")

else:

    DRIVE_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    print("\n✓ Created clean /content/drive mountpoint.")

# --------------------------------------------------------------------------------------------------
# 2. Mount Google Drive
# --------------------------------------------------------------------------------------------------

from google.colab import drive

if not MYDRIVE_ROOT.exists():

    print("\nMounting Google Drive...")

    drive.mount(
        "/content/drive",
        force_remount=False
    )

else:

    print("\n✓ Google Drive already mounted.")

# --------------------------------------------------------------------------------------------------
# 3. Verify actual MyDrive
# --------------------------------------------------------------------------------------------------

if not MYDRIVE_ROOT.exists():

    raise RuntimeError(
        "Google Drive initialization failed: "
        "/content/drive/MyDrive is not visible."
    )

print("\n✓ Google Drive mounted successfully.")
print(f"  Drive root : {DRIVE_ROOT}")
print(f"  MyDrive    : {MYDRIVE_ROOT}")

# --------------------------------------------------------------------------------------------------
# 4. Verify canonical project root
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = MYDRIVE_ROOT / "SPP_GAN_Research"

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        f"Canonical SPP-GAN project root is missing:\n"
        f"{PROJECT_ROOT}"
    )

print(f"\n✓ Canonical project root verified:")
print(f"  {PROJECT_ROOT}")

print("\n" + "=" * 100)
print("NOTEBOOK 09 DRIVE INITIALIZATION: PASS")
print("=" * 100)

NOTEBOOK 09 — GOOGLE DRIVE INITIALIZATION

✓ /content/drive/MyDrive already exists.

✓ Google Drive already mounted.

✓ Google Drive mounted successfully.
  Drive root : /content/drive
  MyDrive    : /content/drive/MyDrive

✓ Canonical project root verified:
  /content/drive/MyDrive/SPP_GAN_Research

NOTEBOOK 09 DRIVE INITIALIZATION: PASS


In [ ]:
# ==================================================================================================
# 2. LOAD CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 100)
print("2. LOAD CONFIGURATION")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# 1. Reproducibility Configuration
# -----------------------------------------------------------------------------------------------

MASTER_SEED = 2025
DETERMINISTIC = True

random.seed(MASTER_SEED)
np.random.seed(MASTER_SEED)
torch.manual_seed(MASTER_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MASTER_SEED)

if DETERMINISTIC:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Master seed : {MASTER_SEED}")
print(f"Deterministic: {DETERMINISTIC}")
print(f"Device      : {DEVICE}")

# -----------------------------------------------------------------------------------------------
# 2. Dataset Registry
#    Inherited from the frozen Notebook 00 configuration.
# -----------------------------------------------------------------------------------------------

DATASET_REGISTRY = {
    "adult_income": {
        "target": "income",
        "identifier_columns": [],
    },
    "bank_marketing": {
        "target": "y",
        "identifier_columns": [],
    },
    "diabetes_130us": {
        "target": "readmitted",
        "identifier_columns": [
            "encounter_id",
            "patient_nbr",
        ],
    },
}

DATASET_IDS = list(
    DATASET_REGISTRY.keys()
)

EXPECTED_DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

if DATASET_IDS != EXPECTED_DATASET_IDS:
    raise ValueError(
        "Dataset registry does not match the frozen Notebook 00 registry.\n"
        f"Expected: {EXPECTED_DATASET_IDS}\n"
        f"Found   : {DATASET_IDS}"
    )

print(
    f"Datasets registered : {len(DATASET_IDS)}"
)

# -----------------------------------------------------------------------------------------------
# 3. Frozen Notebook 08 SPP-GAN Architecture Configuration
# -----------------------------------------------------------------------------------------------

SPPGAN_CONFIG = {
    "latent_dim": 128,
    "generator_hidden_dims": [256, 256],
    "critic_hidden_dims": [256, 256],
    "lambda_stat": 1.0,
    "numerical_activation": "identity",
    "categorical_training_activation": "gumbel_softmax",
    "categorical_probability_mapping": "softmax",
    "hard_decoding": "Notebook_13",
    "critic_output": "scalar",
}

# -----------------------------------------------------------------------------------------------
# 4. Statistical Guidance Configuration
# -----------------------------------------------------------------------------------------------

GUIDANCE_CONFIG = {

    "version":
        "SPP-GAN-STAT-GUIDANCE-1.0",

    "distribution": {
        "enabled": True,
        "method": "rbf_mmd",
        "sigma": 1.0,
    },

    "numerical": {
        "enabled": True,
        "components": [
            "mean",
            "std",
            "min",
            "max",
        ],
    },

    "categorical": {
        "enabled": True,
        "components": [
            "marginal_probability",
        ],
    },

    "dependency": {
        "enabled": True,
        "method": "covariance_frobenius",
    },

    "correlation": {
        "enabled": True,
        "method": "correlation_frobenius",
    },

    "normalization": {
        "enabled": True,
        "epsilon": 1e-8,
    },

    "loss": {
        "lambda_m": 0.25,
        "lambda_mom": 0.25,
        "lambda_d": 0.25,
        "lambda_c": 0.25,
    },

    "training": {
        "lambda_stat": SPPGAN_CONFIG["lambda_stat"],
    },

    "testing": {
        "batch_size": 64,
        "mmd_max_samples": 64,
    },
}

# -----------------------------------------------------------------------------------------------
# 5. Validate Statistical Guidance Weights
# -----------------------------------------------------------------------------------------------

GUIDANCE_WEIGHTS = {
    key: float(value)
    for key, value in GUIDANCE_CONFIG["loss"].items()
}

if any(
    value < 0
    for value in GUIDANCE_WEIGHTS.values()
):
    raise ValueError(
        "Statistical guidance weights cannot be negative."
    )

WEIGHT_SUM = sum(
    GUIDANCE_WEIGHTS.values()
)

if not np.isclose(
    WEIGHT_SUM,
    1.0,
    atol=1e-12,
):
    raise ValueError(
        "Statistical guidance weights must sum to 1.0.\n"
        f"Observed sum: {WEIGHT_SUM}"
    )

# -----------------------------------------------------------------------------------------------
# 6. Validate Frozen SPP-GAN Architecture Parameters
# -----------------------------------------------------------------------------------------------

if SPPGAN_CONFIG["latent_dim"] != 128:
    raise ValueError(
        "latent_dim does not match the frozen Notebook 08 configuration."
    )

if SPPGAN_CONFIG["generator_hidden_dims"] != [256, 256]:
    raise ValueError(
        "Generator hidden dimensions do not match Notebook 08."
    )

if SPPGAN_CONFIG["critic_hidden_dims"] != [256, 256]:
    raise ValueError(
        "Critic hidden dimensions do not match Notebook 08."
    )

if SPPGAN_CONFIG["lambda_stat"] != 1.0:
    raise ValueError(
        "lambda_stat does not match the frozen Notebook 08 configuration."
    )

if SPPGAN_CONFIG["critic_output"] != "scalar":
    raise ValueError(
        "SPP-GAN critic must use the frozen scalar output."
    )

# -----------------------------------------------------------------------------------------------
# 7. Configuration Summary
# -----------------------------------------------------------------------------------------------

print(
    f"Latent dimension    : "
    f"{SPPGAN_CONFIG['latent_dim']}"
)

print(
    f"λ_stat              : "
    f"{SPPGAN_CONFIG['lambda_stat']}"
)

print(
    f"Guidance version    : "
    f"{GUIDANCE_CONFIG['version']}"
)

print(
    f"Guidance weight sum : "
    f"{WEIGHT_SUM:.6f}"
)

print(
    "Statistical guidance: ENABLED"
)

print(
    "Training            : NOT PERFORMED"
)

print(
    "Differential Privacy: NOT PERFORMED"
)

print(
    "Synthetic generation: NOT PERFORMED"
)

print()
print("✓ Configuration loaded and validated.")


2. LOAD CONFIGURATION
Master seed : 2025
Deterministic: True
Device      : cpu
Datasets registered : 3
Latent dimension    : 128
λ_stat              : 1.0
Guidance version    : SPP-GAN-STAT-GUIDANCE-1.0
Guidance weight sum : 1.000000
Statistical guidance: ENABLED
Training            : NOT PERFORMED
Differential Privacy: NOT PERFORMED
Synthetic generation: NOT PERFORMED

✓ Configuration loaded and validated.


In [ ]:
# ==================================================================================================
# 3. LOAD STATISTICAL PROFILES
# ==================================================================================================

print("\n" + "=" * 100)
print("3. LOAD STATISTICAL PROFILES")
print("=" * 100)

from pathlib import Path
import json
import hashlib
import pandas as pd

# --------------------------------------------------------------------------------------------------
# 0. Ensure Google Drive Is Properly Available
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"

print("\n" + "-" * 100)
print("GOOGLE DRIVE INITIALIZATION")
print("-" * 100)

# -----------------------------------------------------------------------------------------------
# 0.1 Mount Google Drive when MyDrive is not visible
# -----------------------------------------------------------------------------------------------

if not MYDRIVE_ROOT.exists():

    print("Google Drive MyDrive is not visible.")
    print("Attempting to mount Google Drive...")

    from google.colab import drive

    try:
        drive.mount(
            "/content/drive",
            force_remount=False
        )

    except ValueError as exc:

        raise RuntimeError(
            "Google Drive mountpoint is not clean.\n"
            f"Mountpoint: {DRIVE_ROOT}\n\n"
            "The current runtime contains a local /content/drive directory "
            "that is preventing Google Drive from mounting.\n"
            "Do not delete Google Drive data. Restart the Colab runtime "
            "or clean the local mountpoint before rerunning Notebook 09."
        ) from exc

# -----------------------------------------------------------------------------------------------
# 0.2 Verify MyDrive
# -----------------------------------------------------------------------------------------------

if not MYDRIVE_ROOT.exists():
    raise RuntimeError(
        "Google Drive is not mounted correctly.\n"
        f"Expected MyDrive path: {MYDRIVE_ROOT}"
    )

if not MYDRIVE_ROOT.is_dir():
    raise RuntimeError(
        "Expected MyDrive path is not a directory.\n"
        f"Path: {MYDRIVE_ROOT}"
    )

print(f"✓ Google Drive available: {DRIVE_ROOT}")
print(f"✓ MyDrive available     : {MYDRIVE_ROOT}")


# --------------------------------------------------------------------------------------------------
# 1. Canonical Project Root
# --------------------------------------------------------------------------------------------------

CANONICAL_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

if not CANONICAL_PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Canonical SPP-GAN project root is unavailable.\n"
        f"Expected path: {CANONICAL_PROJECT_ROOT}\n\n"
        "Ensure Google Drive is mounted and the canonical project "
        "directory exists before running Notebook 09."
    )

if not CANONICAL_PROJECT_ROOT.is_dir():
    raise RuntimeError(
        "Canonical project root exists but is not a directory.\n"
        f"Path: {CANONICAL_PROJECT_ROOT}"
    )

PROJECT_ROOT = CANONICAL_PROJECT_ROOT

print()
print(f"✓ Project root: {PROJECT_ROOT}")


# --------------------------------------------------------------------------------------------------
# 2. Canonical Notebook 03 Artifact Directories
# --------------------------------------------------------------------------------------------------

NB03_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_03"
)

NB03_REFERENCE_ROOT = (
    NB03_ROOT
    / "reference"
)

NB03_GUIDANCE_ROOT = (
    NB03_ROOT
    / "guidance"
)

print()
print("Canonical Notebook 03 artifact root:")
print(f"  {NB03_ROOT}")

print()
print("Canonical reference directory:")
print(f"  {NB03_REFERENCE_ROOT}")

print()
print("Canonical guidance directory:")
print(f"  {NB03_GUIDANCE_ROOT}")


# --------------------------------------------------------------------------------------------------
# 3. Runtime Visibility Check
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("NOTEBOOK 03 RUNTIME VISIBILITY CHECK")
print("-" * 100)

print(f"NB03_ROOT exists : {NB03_ROOT.exists()}")
print(f"NB03_ROOT is_dir : {NB03_ROOT.is_dir()}")

if NB03_ROOT.exists():

    print()
    print("Notebook 03 directories/files visible to Notebook 09:")

    for path in sorted(
        NB03_ROOT.iterdir(),
        key=lambda p: p.name.lower()
    ):

        item_type = "DIR " if path.is_dir() else "FILE"

        print(
            f"  [{item_type}] {path.name}"
        )


# --------------------------------------------------------------------------------------------------
# 4. Verify Canonical Notebook 03 Directories
# --------------------------------------------------------------------------------------------------

REQUIRED_NB03_DIRECTORIES = {
    "notebook_03_root": NB03_ROOT,
    "reference": NB03_REFERENCE_ROOT,
    "guidance": NB03_GUIDANCE_ROOT,
}

for directory_name, directory_path in (
    REQUIRED_NB03_DIRECTORIES.items()
):

    if not directory_path.exists():

        raise FileNotFoundError(
            "Required Notebook 03 directory is missing.\n"
            f"Directory type : {directory_name}\n"
            f"Expected path  : {directory_path}\n\n"
            "Notebook 09 must consume the persisted Notebook 03 "
            "artifacts and must not reconstruct or fabricate them."
        )

    if not directory_path.is_dir():

        raise RuntimeError(
            "Expected Notebook 03 path is not a directory.\n"
            f"Directory type : {directory_name}\n"
            f"Path           : {directory_path}"
        )

print()
print("✓ Canonical Notebook 03 directories verified.")


# --------------------------------------------------------------------------------------------------
# 5. Expected Dataset-Specific Artifact Names
# --------------------------------------------------------------------------------------------------

EXPECTED_GUIDANCE_FILES = {
    dataset_id:
        f"{dataset_id}_spp_gan_statistical_guidance.json"
    for dataset_id in DATASET_IDS
}

EXPECTED_REFERENCE_FILES = {
    dataset_id:
        f"{dataset_id}_spp_gan_statistical_reference.json"
    for dataset_id in DATASET_IDS
}

print()
print("Expected Notebook 03 statistical artifacts:")

for dataset_id in DATASET_IDS:

    print(
        f"  {dataset_id}"
    )

    print(
        f"    Guidance : "
        f"{EXPECTED_GUIDANCE_FILES[dataset_id]}"
    )

    print(
        f"    Reference: "
        f"{EXPECTED_REFERENCE_FILES[dataset_id]}"
    )


# --------------------------------------------------------------------------------------------------
# 6. Resolve and Validate Canonical Artifact Paths
# --------------------------------------------------------------------------------------------------

GUIDANCE_PATHS = {}
REFERENCE_PATHS = {}

ARTIFACT_DISCOVERY = []

for dataset_id in DATASET_IDS:

    guidance_name = EXPECTED_GUIDANCE_FILES[dataset_id]

    reference_name = EXPECTED_REFERENCE_FILES[dataset_id]

    canonical_guidance_path = (
        NB03_GUIDANCE_ROOT
        / guidance_name
    )

    canonical_reference_path = (
        NB03_REFERENCE_ROOT
        / reference_name
    )

    # ----------------------------------------------------------------------------------------------
    # Guidance artifact
    # ----------------------------------------------------------------------------------------------

    guidance_exists = (
        canonical_guidance_path.exists()
        and canonical_guidance_path.is_file()
        and canonical_guidance_path.stat().st_size > 0
    )

    # ----------------------------------------------------------------------------------------------
    # Reference artifact
    # ----------------------------------------------------------------------------------------------

    reference_exists = (
        canonical_reference_path.exists()
        and canonical_reference_path.is_file()
        and canonical_reference_path.stat().st_size > 0
    )

    # ----------------------------------------------------------------------------------------------
    # Exact-name duplicate detection
    # ----------------------------------------------------------------------------------------------

    guidance_matches = sorted(
        NB03_GUIDANCE_ROOT.glob(
            guidance_name
        )
    )

    reference_matches = sorted(
        NB03_REFERENCE_ROOT.glob(
            reference_name
        )
    )

    guidance_match_count = len(
        guidance_matches
    )

    reference_match_count = len(
        reference_matches
    )

    guidance_status = (
        "FOUND"
        if (
            guidance_exists
            and guidance_match_count == 1
        )
        else
        "MISSING"
        if guidance_match_count == 0
        else
        "DUPLICATE"
    )

    reference_status = (
        "FOUND"
        if (
            reference_exists
            and reference_match_count == 1
        )
        else
        "MISSING"
        if reference_match_count == 0
        else
        "DUPLICATE"
    )

    if guidance_status == "FOUND":

        GUIDANCE_PATHS[dataset_id] = (
            canonical_guidance_path
        )

    if reference_status == "FOUND":

        REFERENCE_PATHS[dataset_id] = (
            canonical_reference_path
        )

    ARTIFACT_DISCOVERY.append(
        {
            "dataset_id": dataset_id,

            "guidance_filename": guidance_name,
            "guidance_match_count": guidance_match_count,
            "guidance_status": guidance_status,
            "guidance_path": (
                str(canonical_guidance_path)
                if guidance_status == "FOUND"
                else None
            ),
            "guidance_size_bytes": (
                int(canonical_guidance_path.stat().st_size)
                if guidance_exists
                else None
            ),

            "reference_filename": reference_name,
            "reference_match_count": reference_match_count,
            "reference_status": reference_status,
            "reference_path": (
                str(canonical_reference_path)
                if reference_status == "FOUND"
                else None
            ),
            "reference_size_bytes": (
                int(canonical_reference_path.stat().st_size)
                if reference_exists
                else None
            ),
        }
    )


# --------------------------------------------------------------------------------------------------
# 7. Discovery DataFrame
# --------------------------------------------------------------------------------------------------

DISCOVERY_DF = pd.DataFrame(
    ARTIFACT_DISCOVERY
)

print()
print("-" * 100)
print("NOTEBOOK 03 CANONICAL ARTIFACT DISCOVERY")
print("-" * 100)

for row in ARTIFACT_DISCOVERY:

    print(
        f"{row['dataset_id']:<20} "
        f"guidance={row['guidance_status']:<9} "
        f"reference={row['reference_status']:<9}"
    )


# --------------------------------------------------------------------------------------------------
# 8. Persist Discovery Diagnostic
# --------------------------------------------------------------------------------------------------

DISCOVERY_DIR = (
    NB09_ROOT
    / "metadata"
)

DISCOVERY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DISCOVERY_PATH = (
    DISCOVERY_DIR
    / "notebook_03_artifact_discovery.csv"
)

DISCOVERY_DF.to_csv(
    DISCOVERY_PATH,
    index=False
)

if not DISCOVERY_PATH.exists():

    raise RuntimeError(
        "Notebook 03 artifact discovery report was not persisted."
    )

print()
print(
    f"✓ Discovery report saved: "
    f"{DISCOVERY_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 9. Validate Artifact Availability
# --------------------------------------------------------------------------------------------------

MISSING_GUIDANCE = [
    dataset_id
    for dataset_id in DATASET_IDS
    if dataset_id not in GUIDANCE_PATHS
]

MISSING_REFERENCE = [
    dataset_id
    for dataset_id in DATASET_IDS
    if dataset_id not in REFERENCE_PATHS
]

DUPLICATE_GUIDANCE = [
    row["dataset_id"]
    for row in ARTIFACT_DISCOVERY
    if row["guidance_status"] == "DUPLICATE"
]

DUPLICATE_REFERENCE = [
    row["dataset_id"]
    for row in ARTIFACT_DISCOVERY
    if row["reference_status"] == "DUPLICATE"
]

if MISSING_GUIDANCE or MISSING_REFERENCE:

    print()
    print("!" * 100)
    print("NOTEBOOK 03 CANONICAL ARTIFACTS ARE NOT AVAILABLE")
    print("!" * 100)

    if MISSING_GUIDANCE:

        print()
        print("Missing guidance artifacts:")

        for dataset_id in MISSING_GUIDANCE:

            print(
                f"  ✗ {dataset_id}: "
                f"{NB03_GUIDANCE_ROOT / EXPECTED_GUIDANCE_FILES[dataset_id]}"
            )

    if MISSING_REFERENCE:

        print()
        print("Missing reference artifacts:")

        for dataset_id in MISSING_REFERENCE:

            print(
                f"  ✗ {dataset_id}: "
                f"{NB03_REFERENCE_ROOT / EXPECTED_REFERENCE_FILES[dataset_id]}"
            )

    raise FileNotFoundError(
        "Notebook 03 statistical guidance/reference artifacts "
        "are missing from the canonical persistence directories."
    )


if DUPLICATE_GUIDANCE or DUPLICATE_REFERENCE:

    print()
    print("!" * 100)
    print("DUPLICATE NOTEBOOK 03 ARTIFACTS DETECTED")
    print("!" * 100)

    if DUPLICATE_GUIDANCE:

        print(
            f"Duplicate guidance artifacts: "
            f"{DUPLICATE_GUIDANCE}"
        )

    if DUPLICATE_REFERENCE:

        print(
            f"Duplicate reference artifacts: "
            f"{DUPLICATE_REFERENCE}"
        )

    raise RuntimeError(
        "Duplicate Notebook 03 statistical artifacts detected. "
        "Resolve duplicates before continuing."
    )


print()
print(
    "✓ All six canonical Notebook 03 statistical artifacts are available."
)


# --------------------------------------------------------------------------------------------------
# 10. Required Notebook 03 Guidance Schema
# --------------------------------------------------------------------------------------------------

REQUIRED_GUIDANCE_KEYS = {
    "guidance_version",
    "guidance_type",
    "source_reference_version",
    "source_reference_type",
    "feature_schema",
    "numeric_feature_guidance",
    "categorical_feature_guidance",
    "strongest_numeric_pearson_dependencies",
    "strongest_numeric_spearman_dependencies",
    "strongest_categorical_dependencies",
    "target_policy",
    "identifier_policy",
    "provenance_policy",
    "evidence_policy",
}


# --------------------------------------------------------------------------------------------------
# 11. Required Notebook 03 Reference Schema
# --------------------------------------------------------------------------------------------------

REQUIRED_REFERENCE_KEYS = {
    "reference_version",
    "reference_type",
    "dataset_id",
    "creation_timestamp_utc",
    "random_seed",
    "fit_policy",
    "schema_policy",
    "dataset_profile",
    "feature_schema",
    "feature_profiles",
    "pearson_correlation",
    "spearman_correlation",
    "categorical_dependency",
}


# --------------------------------------------------------------------------------------------------
# 12. Load and Validate Persisted Statistical Profiles
# --------------------------------------------------------------------------------------------------

STATISTICAL_PROFILES = {}

STATISTICAL_PROFILE_HASHES = {}


def _sha256_file(path):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            digest.update(chunk)

    return digest.hexdigest()


for dataset_id in DATASET_IDS:

    guidance_path = GUIDANCE_PATHS[dataset_id]

    reference_path = REFERENCE_PATHS[dataset_id]

    # ----------------------------------------------------------------------------------------------
    # Load guidance
    # ----------------------------------------------------------------------------------------------

    with open(
        guidance_path,
        "r",
        encoding="utf-8",
    ) as f:

        guidance = json.load(f)

    if not isinstance(
        guidance,
        dict,
    ):

        raise ValueError(
            f"Guidance artifact is not a JSON object:\n"
            f"{guidance_path}"
        )

    missing_guidance_keys = (
        REQUIRED_GUIDANCE_KEYS
        -
        set(guidance.keys())
    )

    if missing_guidance_keys:

        raise ValueError(
            f"Notebook 03 guidance schema validation failed "
            f"for {dataset_id}.\n"
            f"Missing keys: "
            f"{sorted(missing_guidance_keys)}"
        )

    # ----------------------------------------------------------------------------------------------
    # Load reference
    # ----------------------------------------------------------------------------------------------

    with open(
        reference_path,
        "r",
        encoding="utf-8",
    ) as f:

        reference = json.load(f)

    if not isinstance(
        reference,
        dict,
    ):

        raise ValueError(
            f"Reference artifact is not a JSON object:\n"
            f"{reference_path}"
        )

    missing_reference_keys = (
        REQUIRED_REFERENCE_KEYS
        -
        set(reference.keys())
    )

    if missing_reference_keys:

        raise ValueError(
            f"Notebook 03 reference schema validation failed "
            f"for {dataset_id}.\n"
            f"Missing keys: "
            f"{sorted(missing_reference_keys)}"
        )

    # ----------------------------------------------------------------------------------------------
    # Dataset identity
    # ----------------------------------------------------------------------------------------------

    if reference.get("dataset_id") != dataset_id:

        raise ValueError(
            "Notebook 03 dataset identity mismatch.\n"
            f"Expected: {dataset_id}\n"
            f"Found   : {reference.get('dataset_id')}\n"
            f"Reference: {reference_path}"
        )

    # ----------------------------------------------------------------------------------------------
    # Guidance/reference version linkage
    # ----------------------------------------------------------------------------------------------

    guidance_reference_version = (
        guidance.get(
            "source_reference_version"
        )
    )

    reference_version = (
        reference.get(
            "reference_version"
        )
    )

    if (
        guidance_reference_version is None
        or reference_version is None
    ):

        raise ValueError(
            f"Missing reference-version linkage for {dataset_id}."
        )

    if guidance_reference_version != reference_version:

        raise ValueError(
            f"Guidance/reference version mismatch for {dataset_id}.\n"
            f"Guidance source version : "
            f"{guidance_reference_version}\n"
            f"Reference version       : "
            f"{reference_version}"
        )

    # ----------------------------------------------------------------------------------------------
    # Reference type linkage
    # ----------------------------------------------------------------------------------------------

    if reference.get(
        "reference_type"
    ) != "SPP-GAN_statistical_reference":

        raise ValueError(
            f"Unexpected Notebook 03 reference_type for "
            f"{dataset_id}.\n"
            f"Found: {reference.get('reference_type')}"
        )

    # ----------------------------------------------------------------------------------------------
    # Guidance source-reference type linkage
    # ----------------------------------------------------------------------------------------------

    if guidance.get(
        "source_reference_type"
    ) != reference.get(
        "reference_type"
    ):

        raise ValueError(
            f"Guidance source_reference_type mismatch for "
            f"{dataset_id}.\n"
            f"Guidance: "
            f"{guidance.get('source_reference_type')}\n"
            f"Reference: "
            f"{reference.get('reference_type')}"
        )

    # ----------------------------------------------------------------------------------------------
    # Train-only provenance
    # ----------------------------------------------------------------------------------------------

    fit_policy = reference.get(
        "fit_policy"
    )

    if not isinstance(
        fit_policy,
        dict,
    ):

        raise ValueError(
            f"Invalid fit_policy structure for {dataset_id}."
        )

    if fit_policy.get(
        "source_split"
    ) != "train_only":

        raise ValueError(
            f"Notebook 03 statistical reference for "
            f"{dataset_id} is not marked train_only.\n"
            f"Found source_split: "
            f"{fit_policy.get('source_split')}"
        )

    if bool(
        fit_policy.get(
            "validation_used_for_reference",
            True,
        )
    ):

        raise ValueError(
            f"Validation data is marked as used for the "
            f"Notebook 03 reference for {dataset_id}."
        )

    if bool(
        fit_policy.get(
            "test_used_for_reference",
            True,
        )
    ):

        raise ValueError(
            f"Test data is marked as used for the "
            f"Notebook 03 reference for {dataset_id}."
        )

    if bool(
        fit_policy.get(
            "synthetic_data_used_for_reference",
            True,
        )
    ):

        raise ValueError(
            f"Synthetic data is marked as used for the "
            f"Notebook 03 reference for {dataset_id}."
        )

    # ----------------------------------------------------------------------------------------------
    # Schema policy
    # ----------------------------------------------------------------------------------------------

    schema_policy = reference.get(
        "schema_policy"
    )

    if not isinstance(
        schema_policy,
        dict,
    ):

        raise ValueError(
            f"Invalid schema_policy structure for {dataset_id}."
        )

    if not bool(
        schema_policy.get(
            "target_retained_in_generative_schema",
            False,
        )
    ):

        raise ValueError(
            f"Target is not marked as retained in the "
            f"generative schema for {dataset_id}."
        )

    if not bool(
        schema_policy.get(
            "target_excluded_from_preprocessing",
            False,
        )
    ):

        raise ValueError(
            f"Target preprocessing exclusion policy failed "
            f"for {dataset_id}."
        )

    if not bool(
        schema_policy.get(
            "target_excluded_from_transformed_features",
            False,
        )
    ):

        raise ValueError(
            f"Target transformed-feature exclusion policy failed "
            f"for {dataset_id}."
        )

    if not bool(
        schema_policy.get(
            "identifiers_excluded_from_generative_data",
            False,
        )
    ):

        raise ValueError(
            f"Identifier exclusion policy failed "
            f"for {dataset_id}."
        )

    # ----------------------------------------------------------------------------------------------
    # Guidance evidence policy
    # ----------------------------------------------------------------------------------------------

    evidence_policy = guidance.get(
        "evidence_policy"
    )

    if not isinstance(
        evidence_policy,
        dict,
    ):

        raise ValueError(
            f"Invalid evidence_policy structure for "
            f"{dataset_id}."
        )

    # ----------------------------------------------------------------------------------------------
    # Guidance provenance, when explicitly available
    # ----------------------------------------------------------------------------------------------

    guidance_fit_policy = guidance.get(
        "fit_policy"
    )

    if guidance_fit_policy is None:
        guidance_fit_policy = guidance.get(
            "provenance_policy"
        )

    if guidance_fit_policy is not None:

        if not isinstance(
            guidance_fit_policy,
            dict,
        ):

            raise ValueError(
                f"Invalid guidance provenance structure for "
                f"{dataset_id}."
            )

        guidance_source_split = (
            guidance_fit_policy.get(
                "source_split"
            )
        )

        if (
            guidance_source_split is not None
            and guidance_source_split
            not in [
                "train_only",
                "train",
            ]
        ):

            raise ValueError(
                f"Unexpected guidance source split for "
                f"{dataset_id}: "
                f"{guidance_source_split}"
            )

    # ----------------------------------------------------------------------------------------------
    # SHA-256 integrity
    # ----------------------------------------------------------------------------------------------

    guidance_sha256 = _sha256_file(
        guidance_path
    )

    reference_sha256 = _sha256_file(
        reference_path
    )

    # ----------------------------------------------------------------------------------------------
    # Store validated profile
    # ----------------------------------------------------------------------------------------------

    STATISTICAL_PROFILES[dataset_id] = {
        "guidance": guidance,
        "reference": reference,
        "guidance_path": guidance_path,
        "reference_path": reference_path,
        "guidance_sha256": guidance_sha256,
        "reference_sha256": reference_sha256,
    }

    STATISTICAL_PROFILE_HASHES[dataset_id] = {
        "guidance_sha256": guidance_sha256,
        "reference_sha256": reference_sha256,
    }

    print(
        f"✓ {dataset_id:<20} "
        f"guidance={guidance_path.name} | "
        f"reference={reference_path.name}"
    )


# --------------------------------------------------------------------------------------------------
# 13. Final Coverage Validation
# --------------------------------------------------------------------------------------------------

if set(
    STATISTICAL_PROFILES.keys()
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Notebook 03 statistical profile coverage does not "
        "match the registered dataset registry."
    )

if len(
    STATISTICAL_PROFILES
) != len(
    DATASET_IDS
):

    raise RuntimeError(
        "Notebook 03 statistical profile count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 14. Final Integrity Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("NOTEBOOK 03 STATISTICAL PROFILE LOAD COMPLETE")
print("=" * 100)

print()
print(
    f"Validated datasets: "
    f"{len(STATISTICAL_PROFILES)}"
)

print(
    "✓ Six canonical Notebook 03 artifacts loaded."
)

print(
    "✓ Guidance/reference schemas validated."
)

print(
    "✓ Dataset identity validated."
)

print(
    "✓ Guidance/reference version linkage validated."
)

print(
    "✓ Guidance/reference type linkage validated."
)

print(
    "✓ Train-only reference provenance validated."
)

print(
    "✓ Validation/test exclusion validated."
)

print(
    "✓ Synthetic-data exclusion from reference validated."
)

print(
    "✓ Target/schema policy validated."
)

print(
    "✓ Identifier exclusion policy validated."
)

print(
    "✓ SHA-256 integrity hashes calculated."
)

print(
    "✓ Notebook 03 statistical reference/guidance artifacts "
    "successfully loaded from canonical persisted paths."
)

print()
print("=" * 100)
print("SECTION 3 STATUS: PASS")
print("=" * 100)


3. LOAD STATISTICAL PROFILES

----------------------------------------------------------------------------------------------------
GOOGLE DRIVE INITIALIZATION
----------------------------------------------------------------------------------------------------
✓ Google Drive available: /content/drive
✓ MyDrive available     : /content/drive/MyDrive

✓ Project root: /content/drive/MyDrive/SPP_GAN_Research

Canonical Notebook 03 artifact root:
  /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03

Canonical reference directory:
  /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03/reference

Canonical guidance directory:
  /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03/guidance

----------------------------------------------------------------------------------------------------
NOTEBOOK 03 RUNTIME VISIBILITY CHECK
----------------------------------------------------------------------------------------------------
NB03_ROOT exists : True
N

In [21]:
# ==================================================================================================
# SECTION 4 — LOAD AND VALIDATE SPP-GAN ARCHITECTURE
# ==================================================================================================

print("\n" + "=" * 100)
print("4. LOAD AND VALIDATE SPP-GAN ARCHITECTURE")
print("=" * 100)

from pathlib import Path
import json
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 0. Validate Notebook 08 Root
# --------------------------------------------------------------------------------------------------

if "NB08_ROOT" not in globals():
    raise RuntimeError(
        "NB08_ROOT is not defined.\n"
        "Notebook 08 canonical root must be initialized before Section 4."
    )

if not isinstance(NB08_ROOT, Path):
    NB08_ROOT = Path(NB08_ROOT)

if not NB08_ROOT.exists():
    raise FileNotFoundError(
        "Notebook 08 root does not exist.\n"
        f"Expected path: {NB08_ROOT}"
    )

if not NB08_ROOT.is_dir():
    raise RuntimeError(
        "Notebook 08 root is not a directory.\n"
        f"Path: {NB08_ROOT}"
    )

print()
print(f"✓ Notebook 08 root: {NB08_ROOT}")


# --------------------------------------------------------------------------------------------------
# 1. Canonical Notebook 08 Architecture Artifacts
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_SUMMARY_PATH = (
    NB08_ROOT
    / "architecture"
    / "sppgan_architecture_summary.csv"
)

PARAMETER_COUNT_PATH = (
    NB08_ROOT
    / "metadata"
    / "sppgan_parameter_count.csv"
)

MODEL_CONFIG_PATH = (
    NB08_ROOT
    / "config"
    / "sppgan_model_configuration.json"
)

ARCHITECTURE_REGISTRY_PATH = (
    NB08_ROOT
    / "metadata"
    / "sppgan_architecture_artifacts.csv"
)

REQUIRED_ARCHITECTURE_FILES = {
    "architecture_summary": ARCHITECTURE_SUMMARY_PATH,
    "parameter_count": PARAMETER_COUNT_PATH,
    "model_configuration": MODEL_CONFIG_PATH,
    "architecture_registry": ARCHITECTURE_REGISTRY_PATH,
}

print()
print("-" * 100)
print("NOTEBOOK 08 ARCHITECTURE ARTIFACT VERIFICATION")
print("-" * 100)

for artifact_name, path in REQUIRED_ARCHITECTURE_FILES.items():

    if not path.exists():
        raise FileNotFoundError(
            "Required Notebook 08 artifact is missing.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {path}"
        )

    if not path.is_file():
        raise RuntimeError(
            "Notebook 08 artifact is not a file.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            "Notebook 08 artifact is empty.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {path}"
        )

    print(
        f"✓ {artifact_name:<24} "
        f"{path}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Load Architecture Artifacts
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_SUMMARY_DF = pd.read_csv(
    ARCHITECTURE_SUMMARY_PATH
)

PARAMETER_COUNT_DF = pd.read_csv(
    PARAMETER_COUNT_PATH
)

with open(
    MODEL_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as f:
    MODEL_CONFIGURATION = json.load(f)

ARCHITECTURE_REGISTRY_DF = pd.read_csv(
    ARCHITECTURE_REGISTRY_PATH
)

if not isinstance(
    MODEL_CONFIGURATION,
    dict,
):
    raise ValueError(
        "SPP-GAN model configuration must be a JSON object."
    )

print()
print("-" * 100)
print("LOADED ARCHITECTURE ARTIFACTS")
print("-" * 100)

print(
    f"✓ Architecture summary : "
    f"{ARCHITECTURE_SUMMARY_DF.shape}"
)

print(
    f"✓ Parameter count       : "
    f"{PARAMETER_COUNT_DF.shape}"
)

print(
    f"✓ Architecture registry : "
    f"{ARCHITECTURE_REGISTRY_DF.shape}"
)

print(
    "✓ Model configuration   : loaded"
)


# --------------------------------------------------------------------------------------------------
# 3. Required Architecture Summary Columns
# --------------------------------------------------------------------------------------------------

REQUIRED_ARCHITECTURE_COLUMNS = {
    "dataset",
    "target",
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
    "latent_dim",
    "generator_hidden_1",
    "generator_hidden_2",
    "critic_hidden_1",
    "critic_hidden_2",
    "total_trainable_parameters",
    "numerical_activation",
    "categorical_training_activation",
    "categorical_probability_mapping",
    "hard_decoding",
    "critic_output",
    "statistical_guidance",
    "differential_privacy",
    "privacy_accounting",
    "training",
}

missing_columns = (
    REQUIRED_ARCHITECTURE_COLUMNS
    - set(ARCHITECTURE_SUMMARY_DF.columns)
)

if missing_columns:
    raise ValueError(
        "Notebook 08 architecture summary schema is incomplete.\n"
        f"Missing columns: {sorted(missing_columns)}"
    )

print()
print("✓ Architecture summary schema validated.")


# --------------------------------------------------------------------------------------------------
# 4. Dataset Registry Consistency
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASETS = set(
    DATASET_IDS
)

ARCHITECTURE_DATASETS = set(
    ARCHITECTURE_SUMMARY_DF["dataset"].astype(str)
)

if ARCHITECTURE_DATASETS != EXPECTED_DATASETS:
    raise ValueError(
        "Notebook 08 architecture dataset registry mismatch.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found   : {sorted(ARCHITECTURE_DATASETS)}"
    )

if len(
    ARCHITECTURE_SUMMARY_DF
) != len(
    DATASET_IDS
):
    raise ValueError(
        "Notebook 08 architecture summary row count does not "
        "match the registered dataset count."
    )

print(
    "✓ Dataset registry consistency validated."
)


# --------------------------------------------------------------------------------------------------
# 5. Unique Dataset Validation
# --------------------------------------------------------------------------------------------------

if (
    ARCHITECTURE_SUMMARY_DF["dataset"]
    .duplicated()
    .any()
):

    duplicates = (
        ARCHITECTURE_SUMMARY_DF.loc[
            ARCHITECTURE_SUMMARY_DF["dataset"].duplicated(
                keep=False
            ),
            "dataset",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    raise ValueError(
        "Duplicate dataset architecture records detected.\n"
        f"Datasets: {duplicates}"
    )

print(
    "✓ Dataset architecture uniqueness validated."
)


# --------------------------------------------------------------------------------------------------
# 6. Numeric Architecture Validation
# --------------------------------------------------------------------------------------------------

NUMERIC_ARCHITECTURE_COLUMNS = [
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
    "latent_dim",
    "generator_hidden_1",
    "generator_hidden_2",
    "critic_hidden_1",
    "critic_hidden_2",
    "total_trainable_parameters",
]

for column in NUMERIC_ARCHITECTURE_COLUMNS:

    values = pd.to_numeric(
        ARCHITECTURE_SUMMARY_DF[column],
        errors="coerce",
    )

    if values.isna().any():
        raise ValueError(
            "Architecture column contains non-numeric values.\n"
            f"Column: {column}"
        )

    if (
        values <= 0
    ).any():
        raise ValueError(
            "Architecture column contains non-positive values.\n"
            f"Column: {column}"
        )

print(
    "✓ Numeric architecture parameters validated."
)


# --------------------------------------------------------------------------------------------------
# 7. Generative Dimension Consistency
# --------------------------------------------------------------------------------------------------

for _, row in ARCHITECTURE_SUMMARY_DF.iterrows():

    dataset_id = str(
        row["dataset"]
    )

    expected_dimension = (
        int(row["numerical_features"])
        + int(row["categorical_features"])
        + 1
    )

    actual_dimension = int(
        row["generative_dimension"]
    )

    if actual_dimension != expected_dimension:
        raise ValueError(
            f"Generative dimension mismatch for {dataset_id}.\n"
            f"Expected from schema components: {expected_dimension}\n"
            f"Architecture summary           : {actual_dimension}"
        )

print(
    "✓ Generative-dimension consistency validated."
)


# --------------------------------------------------------------------------------------------------
# 8. Statistical Profile Dataset Linkage
# --------------------------------------------------------------------------------------------------

if "STATISTICAL_PROFILES" not in globals():
    raise RuntimeError(
        "STATISTICAL_PROFILES is unavailable.\n"
        "Section 3 must successfully load Notebook 03 statistical "
        "profiles before Section 4."
    )

if set(
    STATISTICAL_PROFILES.keys()
) != EXPECTED_DATASETS:
    raise ValueError(
        "Statistical profile dataset coverage does not match "
        "the architecture dataset registry."
    )

print(
    "✓ Notebook 03 statistical-profile linkage validated."
)


# --------------------------------------------------------------------------------------------------
# 9. Target Consistency
# --------------------------------------------------------------------------------------------------

for _, row in ARCHITECTURE_SUMMARY_DF.iterrows():

    dataset_id = str(
        row["dataset"]
    )

    architecture_target = str(
        row["target"]
    )

    reference = STATISTICAL_PROFILES[
        dataset_id
    ]["reference"]

    reference_target = str(
        reference.get(
            "target",
            architecture_target,
        )
    )

    if architecture_target != reference_target:
        raise ValueError(
            f"Target mismatch between Notebook 08 architecture "
            f"and Notebook 03 reference for {dataset_id}.\n"
            f"Notebook 08: {architecture_target}\n"
            f"Notebook 03: {reference_target}"
        )

    reference_schema_policy = reference.get(
        "schema_policy",
        {}
    )

    if not bool(
        reference_schema_policy.get(
            "target_retained_in_generative_schema",
            False,
        )
    ):
        raise ValueError(
            f"Notebook 03 reference does not retain the target "
            f"in the generative schema for {dataset_id}."
        )

print(
    "✓ Target-policy linkage validated."
)


# --------------------------------------------------------------------------------------------------
# 10. Architectural Component Contract
# --------------------------------------------------------------------------------------------------

EXPECTED_ARCHITECTURAL_VALUES = {
    "critic_output": "scalar",
    "statistical_guidance": "Notebook_09",
    "differential_privacy": "Notebook_10",
    "privacy_accounting": "Notebook_11",
    "training": "Notebook_12",
    "hard_decoding": "Notebook_13",
}

for column, expected_value in (
    EXPECTED_ARCHITECTURAL_VALUES.items()
):

    values = (
        ARCHITECTURE_SUMMARY_DF[column]
        .astype(str)
        .unique()
        .tolist()
    )

    if values != [expected_value]:
        raise ValueError(
            f"Unexpected architectural contract in '{column}'.\n"
            f"Expected only: {expected_value}\n"
            f"Found         : {values}"
        )

print(
    "✓ SPP-GAN architectural component contract validated."
)


# --------------------------------------------------------------------------------------------------
# 11. Categorical Output Contract
# --------------------------------------------------------------------------------------------------

categorical_activation_values = (
    ARCHITECTURE_SUMMARY_DF[
        "categorical_training_activation"
    ]
    .astype(str)
    .unique()
    .tolist()
)

categorical_mapping_values = (
    ARCHITECTURE_SUMMARY_DF[
        "categorical_probability_mapping"
    ]
    .astype(str)
    .unique()
    .tolist()
)

if categorical_activation_values != [
    "gumbel_softmax"
]:
    raise ValueError(
        "Unexpected categorical training activation.\n"
        f"Found: {categorical_activation_values}"
    )

if categorical_mapping_values != [
    "softmax"
]:
    raise ValueError(
        "Unexpected categorical probability mapping.\n"
        f"Found: {categorical_mapping_values}"
    )

print(
    "✓ Categorical-output contract validated."
)


# --------------------------------------------------------------------------------------------------
# 12. Numerical Activation — Authoritative Contract Validation
# --------------------------------------------------------------------------------------------------

numerical_activation_values = (
    ARCHITECTURE_SUMMARY_DF[
        "numerical_activation"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)

print()
print(
    "Numerical activation declared by Notebook 08:"
)

print(
    f"  {numerical_activation_values}"
)


# --------------------------------------------------------------------------------------------------
# 12A. Identify Notebook 02 Numerical Transformation Metadata
# --------------------------------------------------------------------------------------------------

NB02_NUMERICAL_CONTRACT = None
NB02_CONTRACT_SOURCE = None

NB02_CANDIDATE_OBJECTS = [
    "PREPROCESSING_METADATA",
    "PREPROCESSING_SCHEMAS",
    "DATASET_METADATA",
    "SCHEMA_METADATA",
    "FEATURE_METADATA",
    "NB02_METADATA",
    "NOTEBOOK_02_METADATA",
]

for candidate_name in NB02_CANDIDATE_OBJECTS:

    if candidate_name in globals():

        candidate_object = globals()[candidate_name]

        if isinstance(
            candidate_object,
            dict,
        ):
            NB02_NUMERICAL_CONTRACT = candidate_object
            NB02_CONTRACT_SOURCE = candidate_name
            break


# --------------------------------------------------------------------------------------------------
# 12B. Validate Numerical Activation Without Inventing a Transformation
# --------------------------------------------------------------------------------------------------

if NB02_NUMERICAL_CONTRACT is None:

    print(
        "⚠ Notebook 02 numerical-transformation metadata is not "
        "available in the current runtime."
    )

    print(
        "✓ Numerical activation was not overridden or fabricated."
    )

    print(
        "✓ Notebook 08 numerical activation will remain "
        "authoritative until the Notebook 02 transformation "
        "contract is explicitly available."
    )

else:

    print()
    print(
        f"✓ Notebook 02 numerical metadata detected: "
        f"{NB02_CONTRACT_SOURCE}"
    )

    # ------------------------------------------------------------------
    # Attempt to identify explicit numerical transformation descriptors
    # ------------------------------------------------------------------

    numerical_contract_terms = []

    def _collect_numerical_terms(
        obj,
        path="",
        depth=0,
        max_depth=4,
    ):
        if depth > max_depth:
            return

        if isinstance(obj, dict):

            for key, value in obj.items():

                key_text = str(key).lower()

                if any(
                    token in key_text
                    for token in [
                        "numeric",
                        "continuous",
                        "scal",
                        "transform",
                        "activation",
                        "range",
                        "bound",
                    ]
                ):

                    numerical_contract_terms.append(
                        (
                            f"{path}.{key}" if path else str(key),
                            value,
                        )
                    )

                _collect_numerical_terms(
                    value,
                    f"{path}.{key}" if path else str(key),
                    depth + 1,
                    max_depth,
                )

        elif isinstance(obj, list):

            for index, value in enumerate(obj):

                _collect_numerical_terms(
                    value,
                    f"{path}[{index}]",
                    depth + 1,
                    max_depth,
                )

    _collect_numerical_terms(
        NB02_NUMERICAL_CONTRACT
    )

    if numerical_contract_terms:

        print(
            f"✓ Notebook 02 numerical contract evidence found: "
            f"{len(numerical_contract_terms)} item(s)"
        )

    else:

        print(
            "⚠ Notebook 02 metadata is present but does not expose "
            "an explicit numerical transformation descriptor."
        )

        print(
            "✓ No unsupported transformation assumption was made."
        )


# --------------------------------------------------------------------------------------------------
# 12C. Explicit Methodology Boundary
# --------------------------------------------------------------------------------------------------

print()
print(
    "Numerical-output methodology boundary:"
)

print(
    "  Notebook 08 records the generator output mechanism."
)

print(
    "  Notebook 02 remains authoritative for the transformed "
    "numerical data domain."
)

print(
    "  No activation change is introduced by Notebook 09."
)

print(
    "  No numerical transformation is fabricated at this stage."
)


# --------------------------------------------------------------------------------------------------
# 13. Parameter Count Schema
# --------------------------------------------------------------------------------------------------

if PARAMETER_COUNT_DF.empty:
    raise ValueError(
        "Notebook 08 parameter-count artifact is empty."
    )

if "dataset" not in PARAMETER_COUNT_DF.columns:
    raise ValueError(
        "Notebook 08 parameter-count artifact lacks the "
        "'dataset' column."
    )

if PARAMETER_COUNT_DF["dataset"].duplicated().any():
    raise ValueError(
        "Duplicate dataset entries detected in parameter-count artifact."
    )

parameter_datasets = set(
    PARAMETER_COUNT_DF["dataset"].astype(str)
)

if parameter_datasets != EXPECTED_DATASETS:
    raise ValueError(
        "Parameter-count dataset coverage mismatch.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found   : {sorted(parameter_datasets)}"
    )

print(
    "✓ Parameter-count artifact coverage validated."
)


# --------------------------------------------------------------------------------------------------
# 14. Architecture Registry Coverage
# --------------------------------------------------------------------------------------------------

if ARCHITECTURE_REGISTRY_DF.empty:
    raise ValueError(
        "Notebook 08 architecture registry is empty."
    )

if "dataset" not in ARCHITECTURE_REGISTRY_DF.columns:
    raise ValueError(
        "Notebook 08 architecture registry lacks the "
        "'dataset' column."
    )

registry_datasets = set(
    ARCHITECTURE_REGISTRY_DF["dataset"].astype(str)
)

if registry_datasets != EXPECTED_DATASETS:
    raise ValueError(
        "Architecture registry dataset coverage mismatch.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found   : {sorted(registry_datasets)}"
    )

if (
    ARCHITECTURE_REGISTRY_DF["dataset"]
    .duplicated()
    .any()
):
    raise ValueError(
        "Duplicate dataset entries detected in architecture registry."
    )

print(
    "✓ Architecture registry coverage validated."
)


# --------------------------------------------------------------------------------------------------
# 15. Create Validated Architecture Registry
# --------------------------------------------------------------------------------------------------

SPPGAN_ARCHITECTURE = {}

for _, row in ARCHITECTURE_SUMMARY_DF.iterrows():

    dataset_id = str(
        row["dataset"]
    )

    SPPGAN_ARCHITECTURE[dataset_id] = {
        column: row[column]
        for column in ARCHITECTURE_SUMMARY_DF.columns
    }


# --------------------------------------------------------------------------------------------------
# 16. Final Architecture Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SPP-GAN ARCHITECTURE LOAD AND VALIDATION COMPLETE")
print("=" * 100)

print()
print(
    f"Validated datasets: "
    f"{len(SPPGAN_ARCHITECTURE)}"
)

for dataset_id in DATASET_IDS:

    architecture = SPPGAN_ARCHITECTURE[
        dataset_id
    ]

    print()
    print(
        f"{dataset_id}"
    )

    print(
        f"  Generative dimension   : "
        f"{int(architecture['generative_dimension'])}"
    )

    print(
        f"  Transformed dimension  : "
        f"{int(architecture['transformed_dimension'])}"
    )

    print(
        f"  Numerical features     : "
        f"{int(architecture['numerical_features'])}"
    )

    print(
        f"  Categorical features   : "
        f"{int(architecture['categorical_features'])}"
    )

    print(
        f"  Latent dimension       : "
        f"{int(architecture['latent_dim'])}"
    )

    print(
        f"  Numerical activation   : "
        f"{architecture['numerical_activation']}"
    )

    print(
        f"  Total trainable params : "
        f"{int(architecture['total_trainable_parameters']):,}"
    )


# --------------------------------------------------------------------------------------------------
# 17. Section-Level Validation Registry
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_VALIDATION_STATUS = {
    "architecture_artifacts": True,
    "architecture_schema": True,
    "dataset_registry": True,
    "dataset_uniqueness": True,
    "numeric_parameters": True,
    "generative_dimension": True,
    "statistical_profile_linkage": True,
    "target_policy": True,
    "architectural_contract": True,
    "categorical_output_contract": True,
    "numerical_activation_not_fabricated": True,
    "parameter_count_coverage": True,
    "architecture_registry_coverage": True,
}

if not all(
    ARCHITECTURE_VALIDATION_STATUS.values()
):
    failed_checks = [
        key
        for key, value in ARCHITECTURE_VALIDATION_STATUS.items()
        if not value
    ]

    raise RuntimeError(
        "Notebook 08 architecture validation failed.\n"
        f"Failed checks: {failed_checks}"
    )


# --------------------------------------------------------------------------------------------------
# 18. Final Section Status
# --------------------------------------------------------------------------------------------------

print()
print(
    "✓ Notebook 08 architecture artifacts loaded."
)

print(
    "✓ Dataset/schema consistency validated."
)

print(
    "✓ Statistical-profile linkage validated."
)

print(
    "✓ Architectural component contract validated."
)

print(
    "✓ Categorical-output contract validated."
)

print(
    "✓ Numerical activation handled without unsupported assumptions."
)

print(
    "✓ Parameter-count coverage validated."
)

print(
    "✓ Architecture registry coverage validated."
)

print()
print("=" * 100)
print("SECTION 4 STATUS: PASS")
print("=" * 100)


4. LOAD AND VALIDATE SPP-GAN ARCHITECTURE

✓ Notebook 08 root: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08

----------------------------------------------------------------------------------------------------
NOTEBOOK 08 ARCHITECTURE ARTIFACT VERIFICATION
----------------------------------------------------------------------------------------------------
✓ architecture_summary     /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/architecture/sppgan_architecture_summary.csv
✓ parameter_count          /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/metadata/sppgan_parameter_count.csv
✓ model_configuration      /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/config/sppgan_model_configuration.json
✓ architecture_registry    /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/metadata/sppgan_architecture_artifacts.csv

----------------------------------------------------------------

In [24]:
# ==================================================================================================
# 5. VALIDATE STATISTICAL INPUTS
# ==================================================================================================

print("\n" + "=" * 100)
print("5. VALIDATE STATISTICAL INPUTS")
print("=" * 100)

import math
import numbers
import json
from collections.abc import Mapping


# --------------------------------------------------------------------------------------------------
# 5.0 Required Keys
# --------------------------------------------------------------------------------------------------

REQUIRED_GUIDANCE_KEYS = [
    "guidance_version",
    "guidance_type",
    "source_reference_version",
    "source_reference_type",
    "feature_schema",
    "numeric_feature_guidance",
    "categorical_feature_guidance",
    "strongest_numeric_pearson_dependencies",
    "strongest_numeric_spearman_dependencies",
    "strongest_categorical_dependencies",
    "target_policy",
    "identifier_policy",
    "provenance_policy",
    "evidence_policy",
]

REQUIRED_REFERENCE_KEYS = [
    "reference_version",
    "reference_type",
    "dataset_id",
    "creation_timestamp_utc",
    "random_seed",
    "fit_policy",
    "schema_policy",
    "dataset_profile",
    "feature_schema",
    "feature_profiles",
    "pearson_correlation",
    "spearman_correlation",
    "categorical_dependency",
]

STATISTICAL_INPUT_VALIDATION = []


# --------------------------------------------------------------------------------------------------
# 5.1 Helper Functions
# --------------------------------------------------------------------------------------------------

def is_finite_numeric(value):
    """
    Return True only for finite numeric scalar values.
    """
    if isinstance(value, bool):
        return False

    if not isinstance(value, numbers.Number):
        return False

    try:
        return math.isfinite(float(value))
    except (TypeError, ValueError):
        return False


def contains_nonfinite_numeric(value):
    """
    Recursively detect NaN / +inf / -inf in nested
    dictionaries, lists, tuples, arrays and numeric scalars.
    """

    if isinstance(value, Mapping):

        return any(
            contains_nonfinite_numeric(v)
            for v in value.values()
        )

    if isinstance(value, (list, tuple, set)):

        return any(
            contains_nonfinite_numeric(v)
            for v in value
        )

    if hasattr(value, "tolist"):

        try:
            converted = value.tolist()

            if converted is not value:
                return contains_nonfinite_numeric(
                    converted
                )

        except Exception:
            pass

    if isinstance(value, bool):
        return False

    if isinstance(value, numbers.Number):
        return not is_finite_numeric(value)

    return False


def validate_nonempty_container(
    value,
    field_name,
    dataset_id,
):
    """
    Validate that a required statistical object exists and
    has a supported non-empty container/scalar representation.

    Important:
    This function intentionally does NOT assume that every
    Notebook 03 artifact is a dictionary.
    """

    if value is None:
        raise ValueError(
            f"{field_name} is missing for '{dataset_id}'."
        )

    if isinstance(
        value,
        (Mapping, list, tuple, set),
    ):

        if len(value) == 0:
            raise ValueError(
                f"{field_name} is empty for '{dataset_id}'."
            )

        return True

    if hasattr(value, "shape"):

        try:
            if value.size == 0:
                raise ValueError(
                    f"{field_name} is empty for '{dataset_id}'."
                )
        except AttributeError:
            pass

        return True

    if isinstance(
        value,
        (str, numbers.Number),
    ):

        if isinstance(value, str) and not value.strip():
            raise ValueError(
                f"{field_name} is empty for '{dataset_id}'."
            )

        return True

    raise TypeError(
        f"{field_name} has unsupported structure for "
        f"'{dataset_id}': {type(value).__name__}"
    )


def extract_feature_names(
    schema,
    feature_type,
):
    """
    Extract feature names only when Notebook 03 exposes them
    through a recognized schema field.

    No artificial schema is created.
    """

    if not isinstance(
        schema,
        Mapping,
    ):
        return []

    if feature_type == "numeric":

        candidate_keys = [
            "numeric_features",
            "numerical_features",
            "numeric",
            "numerical",
        ]

    else:

        candidate_keys = [
            "categorical_features",
            "categorical",
            "cat_features",
        ]

    for key in candidate_keys:

        if key not in schema:
            continue

        value = schema[key]

        if isinstance(
            value,
            Mapping,
        ):
            return [
                str(k)
                for k in value.keys()
            ]

        if isinstance(
            value,
            (list, tuple, set),
        ):
            return [
                str(v)
                for v in value
            ]

    return []


def policy_contains_train_only(
    policy,
):
    """
    Check explicit textual evidence of a training-only policy
    without assuming a fixed Notebook 03 policy schema.
    """

    if policy is None:
        return False

    try:
        text = json.dumps(
            policy,
            sort_keys=True,
            default=str,
        ).lower()
    except Exception:
        text = str(policy).lower()

    indicators = [
        "train_only",
        "training_only",
        "training data only",
        "fit on training",
        "fit using training",
        "train data only",
        "training split",
    ]

    return any(
        indicator in text
        for indicator in indicators
    )


# --------------------------------------------------------------------------------------------------
# 5.2 Validate Dataset Coverage
# --------------------------------------------------------------------------------------------------

if "STATISTICAL_PROFILES" not in globals():
    raise RuntimeError(
        "STATISTICAL_PROFILES is not available."
    )

if set(STATISTICAL_PROFILES.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "STATISTICAL_PROFILES dataset coverage does not match "
        "DATASET_IDS.\n"
        f"Expected: {sorted(DATASET_IDS)}\n"
        f"Found   : {sorted(STATISTICAL_PROFILES.keys())}"
    )

print()
print(
    "✓ Notebook 03 statistical-profile dataset coverage validated."
)


# --------------------------------------------------------------------------------------------------
# 5.3 Dataset-by-Dataset Validation
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(
        f"VALIDATING STATISTICAL INPUTS: {dataset_id}"
    )
    print("-" * 100)

    profile_bundle = STATISTICAL_PROFILES[
        dataset_id
    ]

    if not isinstance(
        profile_bundle,
        Mapping,
    ):
        raise TypeError(
            f"STATISTICAL_PROFILES['{dataset_id}'] "
            "must be a mapping."
        )

    if "guidance" not in profile_bundle:
        raise KeyError(
            f"Guidance artifact missing for '{dataset_id}'."
        )

    if "reference" not in profile_bundle:
        raise KeyError(
            f"Reference artifact missing for '{dataset_id}'."
        )

    guidance = profile_bundle[
        "guidance"
    ]

    reference = profile_bundle[
        "reference"
    ]


    # ----------------------------------------------------------------------------------------------
    # A. Top-Level Object Validation
    # ----------------------------------------------------------------------------------------------

    if not isinstance(
        guidance,
        Mapping,
    ):
        raise TypeError(
            f"Guidance artifact for '{dataset_id}' "
            "is not a mapping."
        )

    if not isinstance(
        reference,
        Mapping,
    ):
        raise TypeError(
            f"Reference artifact for '{dataset_id}' "
            "is not a mapping."
        )


    # ----------------------------------------------------------------------------------------------
    # B. Required Guidance Keys
    # ----------------------------------------------------------------------------------------------

    missing_guidance = [
        key
        for key in REQUIRED_GUIDANCE_KEYS
        if key not in guidance
    ]

    if missing_guidance:
        raise RuntimeError(
            f"Missing required guidance keys for "
            f"'{dataset_id}': {missing_guidance}"
        )

    guidance_keys_valid = True


    # ----------------------------------------------------------------------------------------------
    # C. Required Reference Keys
    # ----------------------------------------------------------------------------------------------

    missing_reference = [
        key
        for key in REQUIRED_REFERENCE_KEYS
        if key not in reference
    ]

    if missing_reference:
        raise RuntimeError(
            f"Missing required reference keys for "
            f"'{dataset_id}': {missing_reference}"
        )

    reference_keys_valid = True


    # ----------------------------------------------------------------------------------------------
    # D. Dataset Identity
    # ----------------------------------------------------------------------------------------------

    reference_dataset_id = str(
        reference["dataset_id"]
    )

    dataset_id_valid = (
        reference_dataset_id
        == dataset_id
    )

    if not dataset_id_valid:
        raise ValueError(
            f"Dataset identity mismatch for '{dataset_id}'.\n"
            f"Expected: {dataset_id}\n"
            f"Found   : {reference_dataset_id}"
        )


    # ----------------------------------------------------------------------------------------------
    # E. Guidance-to-Reference Version Linkage
    # ----------------------------------------------------------------------------------------------

    reference_version = str(
        reference["reference_version"]
    )

    source_reference_version = str(
        guidance["source_reference_version"]
    )

    version_link_valid = (
        source_reference_version
        == reference_version
    )

    if not version_link_valid:
        raise ValueError(
            f"Guidance/reference version mismatch for "
            f"'{dataset_id}'.\n"
            f"Guidance source version: "
            f"{source_reference_version}\n"
            f"Reference version: "
            f"{reference_version}"
        )


    # ----------------------------------------------------------------------------------------------
    # F. Guidance-to-Reference Type Linkage
    # ----------------------------------------------------------------------------------------------

    reference_type = str(
        reference["reference_type"]
    )

    source_reference_type = str(
        guidance["source_reference_type"]
    )

    reference_type_link_valid = (
        source_reference_type
        == reference_type
    )

    if not reference_type_link_valid:
        raise ValueError(
            f"Guidance/reference type mismatch for "
            f"'{dataset_id}'.\n"
            f"Guidance source type: "
            f"{source_reference_type}\n"
            f"Reference type: "
            f"{reference_type}"
        )


    # ----------------------------------------------------------------------------------------------
    # G. Guidance / Reference Types
    # ----------------------------------------------------------------------------------------------

    guidance_type = str(
        guidance["guidance_type"]
    )

    if not guidance_type.strip():
        raise ValueError(
            f"Empty guidance_type for '{dataset_id}'."
        )

    if not reference_type.strip():
        raise ValueError(
            f"Empty reference_type for '{dataset_id}'."
        )


    # ----------------------------------------------------------------------------------------------
    # H. Feature Schema Validation
    # ----------------------------------------------------------------------------------------------

    guidance_schema = guidance[
        "feature_schema"
    ]

    reference_schema = reference[
        "feature_schema"
    ]

    if not isinstance(
        guidance_schema,
        Mapping,
    ):
        raise TypeError(
            f"Guidance feature_schema is not a mapping "
            f"for '{dataset_id}'."
        )

    if not isinstance(
        reference_schema,
        Mapping,
    ):
        raise TypeError(
            f"Reference feature_schema is not a mapping "
            f"for '{dataset_id}'."
        )

    guidance_schema_valid = True
    reference_schema_valid = True


    # ----------------------------------------------------------------------------------------------
    # I. Architecture-Level Feature Counts
    # ----------------------------------------------------------------------------------------------

    architecture = SPPGAN_ARCHITECTURE[
        dataset_id
    ]

    expected_numeric_count = int(
        architecture[
            "numerical_features"
        ]
    )

    expected_categorical_count = int(
        architecture[
            "categorical_features"
        ]
    )

    reference_numeric_features = extract_feature_names(
        reference_schema,
        "numeric",
    )

    reference_categorical_features = extract_feature_names(
        reference_schema,
        "categorical",
    )

    guidance_numeric_features = extract_feature_names(
        guidance_schema,
        "numeric",
    )

    guidance_categorical_features = extract_feature_names(
        guidance_schema,
        "categorical",
    )


    # Only enforce counts when the schema explicitly exposes
    # corresponding feature lists.

    if reference_numeric_features:

        if len(reference_numeric_features) != expected_numeric_count:
            raise ValueError(
                f"Reference numerical feature count mismatch "
                f"for '{dataset_id}'.\n"
                f"Expected: {expected_numeric_count}\n"
                f"Found   : {len(reference_numeric_features)}"
            )

    if reference_categorical_features:

        if len(reference_categorical_features) != expected_categorical_count:
            raise ValueError(
                f"Reference categorical feature count mismatch "
                f"for '{dataset_id}'.\n"
                f"Expected: {expected_categorical_count}\n"
                f"Found   : {len(reference_categorical_features)}"
            )

    if guidance_numeric_features:

        if len(guidance_numeric_features) != expected_numeric_count:
            raise ValueError(
                f"Guidance numerical feature count mismatch "
                f"for '{dataset_id}'.\n"
                f"Expected: {expected_numeric_count}\n"
                f"Found   : {len(guidance_numeric_features)}"
            )

    if guidance_categorical_features:

        if len(guidance_categorical_features) != expected_categorical_count:
            raise ValueError(
                f"Guidance categorical feature count mismatch "
                f"for '{dataset_id}'.\n"
                f"Expected: {expected_categorical_count}\n"
                f"Found   : {len(guidance_categorical_features)}"
            )

    numeric_schema_valid = True
    categorical_schema_valid = True


    # ----------------------------------------------------------------------------------------------
    # J. Statistical Guidance Components
    # ----------------------------------------------------------------------------------------------

    numeric_guidance = guidance[
        "numeric_feature_guidance"
    ]

    categorical_guidance = guidance[
        "categorical_feature_guidance"
    ]

    pearson_guidance = guidance[
        "strongest_numeric_pearson_dependencies"
    ]

    spearman_guidance = guidance[
        "strongest_numeric_spearman_dependencies"
    ]

    categorical_dependency_guidance = guidance[
        "strongest_categorical_dependencies"
    ]

    numeric_guidance_valid = validate_nonempty_container(
        numeric_guidance,
        "numeric_feature_guidance",
        dataset_id,
    )

    categorical_guidance_valid = validate_nonempty_container(
        categorical_guidance,
        "categorical_feature_guidance",
        dataset_id,
    )

    pearson_guidance_valid = validate_nonempty_container(
        pearson_guidance,
        "strongest_numeric_pearson_dependencies",
        dataset_id,
    )

    spearman_guidance_valid = validate_nonempty_container(
        spearman_guidance,
        "strongest_numeric_spearman_dependencies",
        dataset_id,
    )

    categorical_dependency_guidance_valid = validate_nonempty_container(
        categorical_dependency_guidance,
        "strongest_categorical_dependencies",
        dataset_id,
    )


    # ----------------------------------------------------------------------------------------------
    # K. Reference Statistical Components
    # ----------------------------------------------------------------------------------------------

    pearson_reference = reference[
        "pearson_correlation"
    ]

    spearman_reference = reference[
        "spearman_correlation"
    ]

    categorical_reference = reference[
        "categorical_dependency"
    ]

    pearson_reference_valid = validate_nonempty_container(
        pearson_reference,
        "pearson_correlation",
        dataset_id,
    )

    spearman_reference_valid = validate_nonempty_container(
        spearman_reference,
        "spearman_correlation",
        dataset_id,
    )

    categorical_reference_valid = validate_nonempty_container(
        categorical_reference,
        "categorical_dependency",
        dataset_id,
    )


    # ----------------------------------------------------------------------------------------------
    # L. Dataset Profile Validation
    # ----------------------------------------------------------------------------------------------

    dataset_profile = reference[
        "dataset_profile"
    ]

    dataset_profile_valid = validate_nonempty_container(
        dataset_profile,
        "dataset_profile",
        dataset_id,
    )


    # ----------------------------------------------------------------------------------------------
    # M. Feature Profile Validation
    # ----------------------------------------------------------------------------------------------

    feature_profiles = reference[
        "feature_profiles"
    ]

    feature_profiles_valid = validate_nonempty_container(
        feature_profiles,
        "feature_profiles",
        dataset_id,
    )


    # ----------------------------------------------------------------------------------------------
    # N. Target / Identifier / Provenance Policies
    # ----------------------------------------------------------------------------------------------

    target_policy = guidance[
        "target_policy"
    ]

    identifier_policy = guidance[
        "identifier_policy"
    ]

    provenance_policy = guidance[
        "provenance_policy"
    ]

    target_policy_valid = (
        target_policy is not None
    )

    identifier_policy_valid = (
        identifier_policy is not None
    )

    provenance_policy_valid = (
        provenance_policy is not None
    )

    if not target_policy_valid:
        raise ValueError(
            f"target_policy is missing for '{dataset_id}'."
        )

    if not identifier_policy_valid:
        raise ValueError(
            f"identifier_policy is missing for '{dataset_id}'."
        )

    if not provenance_policy_valid:
        raise ValueError(
            f"provenance_policy is missing for '{dataset_id}'."
        )


    # ----------------------------------------------------------------------------------------------
    # O. Evidence Policy
    # ----------------------------------------------------------------------------------------------

    evidence_policy = guidance[
        "evidence_policy"
    ]

    evidence_policy_valid = validate_nonempty_container(
        evidence_policy,
        "evidence_policy",
        dataset_id,
    )


    # ----------------------------------------------------------------------------------------------
    # P. Train-Only Provenance
    # ----------------------------------------------------------------------------------------------

    fit_policy = reference[
        "fit_policy"
    ]

    schema_policy = reference[
        "schema_policy"
    ]

    fit_policy_valid = validate_nonempty_container(
        fit_policy,
        "fit_policy",
        dataset_id,
    )

    schema_policy_valid = validate_nonempty_container(
        schema_policy,
        "schema_policy",
        dataset_id,
    )

    train_only_policy_valid = (
        policy_contains_train_only(
            fit_policy
        )
        or
        policy_contains_train_only(
            schema_policy
        )
    )

    if not train_only_policy_valid:
        raise ValueError(
            f"Explicit train-only provenance evidence was not "
            f"found for '{dataset_id}'."
        )


    # ----------------------------------------------------------------------------------------------
    # Q. Finite-Value Validation
    # ----------------------------------------------------------------------------------------------

    guidance_nonfinite = contains_nonfinite_numeric(
        guidance
    )

    reference_nonfinite = contains_nonfinite_numeric(
        reference
    )

    finite_values_valid = (
        not guidance_nonfinite
        and not reference_nonfinite
    )

    if not finite_values_valid:
        raise ValueError(
            f"Non-finite numerical value detected in statistical "
            f"inputs for '{dataset_id}'."
        )


    # ----------------------------------------------------------------------------------------------
    # R. Dataset Validation Result
    # ----------------------------------------------------------------------------------------------

    validation_pass = all([
        guidance_keys_valid,
        reference_keys_valid,
        dataset_id_valid,
        version_link_valid,
        reference_type_link_valid,
        guidance_schema_valid,
        reference_schema_valid,
        numeric_schema_valid,
        categorical_schema_valid,
        numeric_guidance_valid,
        categorical_guidance_valid,
        pearson_guidance_valid,
        spearman_guidance_valid,
        categorical_dependency_guidance_valid,
        pearson_reference_valid,
        spearman_reference_valid,
        categorical_reference_valid,
        dataset_profile_valid,
        feature_profiles_valid,
        target_policy_valid,
        identifier_policy_valid,
        provenance_policy_valid,
        evidence_policy_valid,
        fit_policy_valid,
        schema_policy_valid,
        train_only_policy_valid,
        finite_values_valid,
    ])

    STATISTICAL_INPUT_VALIDATION.append({
        "dataset_id":
            dataset_id,

        "guidance_keys_valid":
            guidance_keys_valid,

        "reference_keys_valid":
            reference_keys_valid,

        "dataset_id_valid":
            dataset_id_valid,

        "version_link_valid":
            version_link_valid,

        "reference_type_link_valid":
            reference_type_link_valid,

        "guidance_schema_valid":
            guidance_schema_valid,

        "reference_schema_valid":
            reference_schema_valid,

        "numeric_schema_valid":
            numeric_schema_valid,

        "categorical_schema_valid":
            categorical_schema_valid,

        "numeric_guidance_valid":
            numeric_guidance_valid,

        "categorical_guidance_valid":
            categorical_guidance_valid,

        "pearson_guidance_valid":
            pearson_guidance_valid,

        "spearman_guidance_valid":
            spearman_guidance_valid,

        "categorical_dependency_guidance_valid":
            categorical_dependency_guidance_valid,

        "pearson_reference_valid":
            pearson_reference_valid,

        "spearman_reference_valid":
            spearman_reference_valid,

        "categorical_reference_valid":
            categorical_reference_valid,

        "dataset_profile_valid":
            dataset_profile_valid,

        "feature_profiles_valid":
            feature_profiles_valid,

        "target_policy_valid":
            target_policy_valid,

        "identifier_policy_valid":
            identifier_policy_valid,

        "provenance_policy_valid":
            provenance_policy_valid,

        "evidence_policy_valid":
            evidence_policy_valid,

        "train_only_policy_valid":
            train_only_policy_valid,

        "finite_values_valid":
            finite_values_valid,

        "status":
            "PASS"
            if validation_pass
            else "FAIL",
    })

    print(
        f"✓ {dataset_id} statistical inputs validated."
    )


# --------------------------------------------------------------------------------------------------
# 5.4 Validation DataFrame
# --------------------------------------------------------------------------------------------------

STATISTICAL_INPUT_VALIDATION_DF = pd.DataFrame(
    STATISTICAL_INPUT_VALIDATION
)

display(
    STATISTICAL_INPUT_VALIDATION_DF
)


# --------------------------------------------------------------------------------------------------
# 5.5 Hard Failure
# --------------------------------------------------------------------------------------------------

if not (
    STATISTICAL_INPUT_VALIDATION_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):

    failed_datasets = (
        STATISTICAL_INPUT_VALIDATION_DF.loc[
            STATISTICAL_INPUT_VALIDATION_DF[
                "status"
            ]
            != "PASS",
            "dataset_id",
        ]
        .astype(str)
        .tolist()
    )

    raise RuntimeError(
        "Notebook 03 statistical input validation failed.\n"
        f"Failed datasets: {failed_datasets}"
    )


# --------------------------------------------------------------------------------------------------
# 5.6 Coverage Summary
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("STATISTICAL INPUT COVERAGE SUMMARY")
print("-" * 100)

print(
    f"✓ Datasets validated                 : "
    f"{len(STATISTICAL_INPUT_VALIDATION_DF)}"
)

print(
    f"✓ Guidance artifacts validated       : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['guidance_keys_valid'].sum())}"
)

print(
    f"✓ Reference artifacts validated      : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['reference_keys_valid'].sum())}"
)

print(
    f"✓ Version linkages validated         : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['version_link_valid'].sum())}"
)

print(
    f"✓ Reference-type linkages validated : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['reference_type_link_valid'].sum())}"
)

print(
    f"✓ Numeric guidance validated         : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['numeric_guidance_valid'].sum())}"
)

print(
    f"✓ Categorical guidance validated     : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['categorical_guidance_valid'].sum())}"
)

print(
    f"✓ Pearson guidance validated         : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['pearson_guidance_valid'].sum())}"
)

print(
    f"✓ Spearman guidance validated        : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['spearman_guidance_valid'].sum())}"
)

print(
    f"✓ Categorical dependency validated   : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['categorical_dependency_guidance_valid'].sum())}"
)

print(
    f"✓ Reference statistical structures   : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['pearson_reference_valid'].sum())}"
)

print(
    f"✓ Feature profiles validated         : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['feature_profiles_valid'].sum())}"
)

print(
    f"✓ Train-only policy validated        : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['train_only_policy_valid'].sum())}"
)

print(
    f"✓ Finite statistical inputs          : "
    f"{int(STATISTICAL_INPUT_VALIDATION_DF['finite_values_valid'].sum())}"
)


# --------------------------------------------------------------------------------------------------
# 5.7 Persist Validation Artifact
# --------------------------------------------------------------------------------------------------

if "NB09_ROOT" not in globals():
    raise RuntimeError(
        "NB09_ROOT is not defined."
    )

NB09_METADATA_ROOT = (
    NB09_ROOT
    / "metadata"
)

NB09_METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

STATISTICAL_INPUT_VALIDATION_PATH = (
    NB09_METADATA_ROOT
    / "statistical_input_validation.csv"
)

STATISTICAL_INPUT_VALIDATION_DF.to_csv(
    STATISTICAL_INPUT_VALIDATION_PATH,
    index=False,
)

if not STATISTICAL_INPUT_VALIDATION_PATH.exists():
    raise RuntimeError(
        "Statistical input validation artifact was not persisted."
    )

if STATISTICAL_INPUT_VALIDATION_PATH.stat().st_size == 0:
    raise RuntimeError(
        "Statistical input validation artifact is empty."
    )

print()
print(
    f"✓ Validation artifact saved:\n"
    f"  {STATISTICAL_INPUT_VALIDATION_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 5.8 Final Status
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 5 STATUS: PASS")
print("=" * 100)


5. VALIDATE STATISTICAL INPUTS

✓ Notebook 03 statistical-profile dataset coverage validated.

----------------------------------------------------------------------------------------------------
VALIDATING STATISTICAL INPUTS: adult_income
----------------------------------------------------------------------------------------------------
✓ adult_income statistical inputs validated.

----------------------------------------------------------------------------------------------------
VALIDATING STATISTICAL INPUTS: bank_marketing
----------------------------------------------------------------------------------------------------
✓ bank_marketing statistical inputs validated.

----------------------------------------------------------------------------------------------------
VALIDATING STATISTICAL INPUTS: diabetes_130us
----------------------------------------------------------------------------------------------------
✓ diabetes_130us statistical inputs validated.


,dataset_id,guidance_keys_valid,reference_keys_valid,dataset_id_valid,version_link_valid,reference_type_link_valid,guidance_schema_valid,reference_schema_valid,numeric_schema_valid,categorical_schema_valid,...,categorical_reference_valid,dataset_profile_valid,feature_profiles_valid,target_policy_valid,identifier_policy_valid,provenance_policy_valid,evidence_policy_valid,train_only_policy_valid,finite_values_valid,status
0,adult_income,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,PASS
1,bank_marketing,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,PASS
2,diabetes_130us,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,PASS



----------------------------------------------------------------------------------------------------
STATISTICAL INPUT COVERAGE SUMMARY
----------------------------------------------------------------------------------------------------
✓ Datasets validated                 : 3
✓ Guidance artifacts validated       : 3
✓ Reference artifacts validated      : 3
✓ Version linkages validated         : 3
✓ Reference-type linkages validated : 3
✓ Numeric guidance validated         : 3
✓ Categorical guidance validated     : 3
✓ Pearson guidance validated         : 3
✓ Spearman guidance validated        : 3
✓ Categorical dependency validated   : 3
✓ Reference statistical structures   : 3
✓ Feature profiles validated         : 3
✓ Train-only policy validated        : 3
✓ Finite statistical inputs          : 3

✓ Validation artifact saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09/metadata/statistical_input_validation.csv

SECTION 5 STATUS: PASS


In [26]:
# ==================================================================================================
# 6. DEFINE DISTRIBUTION GUIDANCE
# ==================================================================================================

print("\n" + "=" * 100)
print("6. DEFINE DISTRIBUTION GUIDANCE")
print("=" * 100)

import math
import torch
import numpy as np


# --------------------------------------------------------------------------------------------------
# 6.0 Configuration
# --------------------------------------------------------------------------------------------------

if "GUIDANCE_CONFIG" not in globals():
    raise RuntimeError(
        "GUIDANCE_CONFIG is not available."
    )

if "STATISTICAL_PROFILES" not in globals():
    raise RuntimeError(
        "STATISTICAL_PROFILES is not available."
    )

if "SPPGAN_ARCHITECTURE" not in globals():
    raise RuntimeError(
        "SPPGAN_ARCHITECTURE is not available."
    )

EPSILON = float(
    GUIDANCE_CONFIG[
        "normalization"
    ][
        "epsilon"
    ]
)

if not math.isfinite(EPSILON):
    raise ValueError(
        "GUIDANCE_CONFIG normalization epsilon must be finite."
    )

if EPSILON <= 0:
    raise ValueError(
        "GUIDANCE_CONFIG normalization epsilon must be > 0."
    )


# --------------------------------------------------------------------------------------------------
# 6.1 Safe Tensor Conversion
# --------------------------------------------------------------------------------------------------

def _safe_tensor(
    value,
    device,
    dtype=torch.float32,
):
    """
    Convert an input into a tensor while rejecting non-finite values.

    Statistical artifacts are validated in Section 5.
    Therefore invalid numerical values must not be silently replaced.
    """

    tensor = torch.as_tensor(
        value,
        dtype=dtype,
        device=device,
    )

    if not torch.is_floating_point(
        tensor
    ):
        tensor = tensor.to(
            dtype=dtype
        )

    if not torch.isfinite(
        tensor
    ).all():

        raise ValueError(
            "Non-finite value detected while converting "
            "statistical guidance input to tensor."
        )

    return tensor


# --------------------------------------------------------------------------------------------------
# 6.2 Tensor Shape Validation
# --------------------------------------------------------------------------------------------------

def _validate_feature_batch(
    tensor,
    name,
):
    """
    Validate a two-dimensional feature batch.
    """

    if not isinstance(
        tensor,
        torch.Tensor,
    ):
        raise TypeError(
            f"{name} must be a torch.Tensor."
        )

    if tensor.ndim != 2:
        raise ValueError(
            f"{name} must have shape [batch, features]. "
            f"Received shape: {tuple(tensor.shape)}"
        )

    if tensor.shape[0] < 1:
        raise ValueError(
            f"{name} must contain at least one sample."
        )

    if tensor.shape[1] < 1:
        raise ValueError(
            f"{name} must contain at least one feature."
        )

    if not torch.isfinite(
        tensor
    ).all():

        raise ValueError(
            f"{name} contains non-finite values."
        )

    return True


# --------------------------------------------------------------------------------------------------
# 6.3 Pairwise Squared Euclidean Distance
# --------------------------------------------------------------------------------------------------

def pairwise_squared_distance(
    x,
    y,
):
    """
    Compute pairwise squared Euclidean distances.

    x: [n, d]
    y: [m, d]

    Returns:
        [n, m]
    """

    _validate_feature_batch(
        x,
        "x",
    )

    _validate_feature_batch(
        y,
        "y",
    )

    if x.shape[1] != y.shape[1]:
        raise ValueError(
            "Pairwise distance feature dimensions do not match.\n"
            f"x: {x.shape[1]}\n"
            f"y: {y.shape[1]}"
        )

    x_norm = (
        x.pow(2)
        .sum(
            dim=1,
            keepdim=True,
        )
    )

    y_norm = (
        y.pow(2)
        .sum(
            dim=1,
            keepdim=True,
        )
        .transpose(
            0,
            1,
        )
    )

    distance = (
        x_norm
        + y_norm
        - 2.0
        * torch.matmul(
            x,
            y.transpose(
                0,
                1,
            ),
        )
    )

    distance = torch.clamp(
        distance,
        min=0.0,
    )

    if not torch.isfinite(
        distance
    ).all():

        raise FloatingPointError(
            "Pairwise squared distance produced "
            "non-finite values."
        )

    return distance


# --------------------------------------------------------------------------------------------------
# 6.4 RBF Kernel
# --------------------------------------------------------------------------------------------------

def rbf_kernel(
    x,
    y,
    sigma=1.0,
):
    """
    Radial Basis Function kernel.

    K(x,y) = exp(-||x-y||² / (2*sigma²))
    """

    if not math.isfinite(
        float(sigma)
    ):
        raise ValueError(
            "RBF sigma must be finite."
        )

    if float(sigma) <= 0:
        raise ValueError(
            "RBF sigma must be > 0."
        )

    distance = pairwise_squared_distance(
        x,
        y,
    )

    sigma_squared = max(
        float(sigma) ** 2,
        EPSILON,
    )

    kernel = torch.exp(
        -distance
        / (
            2.0
            * sigma_squared
        )
    )

    if not torch.isfinite(
        kernel
    ).all():

        raise FloatingPointError(
            "RBF kernel produced non-finite values."
        )

    return kernel


# --------------------------------------------------------------------------------------------------
# 6.5 Differentiable MMD²
# --------------------------------------------------------------------------------------------------

def differentiable_mmd(
    real,
    synthetic,
    sigma=1.0,
):
    """
    Biased differentiable MMD² using an RBF kernel.

    Both inputs:
        [batch, feature_dimension]

    The function is differentiable with respect to the
    synthetic representation.
    """

    _validate_feature_batch(
        real,
        "real",
    )

    _validate_feature_batch(
        synthetic,
        "synthetic",
    )

    if real.shape[1] != synthetic.shape[1]:
        raise ValueError(
            "MMD feature dimensions do not match.\n"
            f"Real      : {real.shape[1]}\n"
            f"Synthetic : {synthetic.shape[1]}"
        )

    k_rr = rbf_kernel(
        real,
        real,
        sigma=sigma,
    )

    k_ss = rbf_kernel(
        synthetic,
        synthetic,
        sigma=sigma,
    )

    k_rs = rbf_kernel(
        real,
        synthetic,
        sigma=sigma,
    )

    mmd_squared = (
        k_rr.mean()
        + k_ss.mean()
        - 2.0
        * k_rs.mean()
    )

    if not torch.isfinite(
        mmd_squared
    ):
        raise FloatingPointError(
            "MMD² produced a non-finite value."
        )

    return mmd_squared


# --------------------------------------------------------------------------------------------------
# 6.6 Robust RBF Bandwidth
# --------------------------------------------------------------------------------------------------

def estimate_rbf_sigma(
    real_features,
    min_sigma=None,
):
    """
    Estimate a reproducible RBF bandwidth from the real feature
    representation using the median pairwise distance.

    The synthetic batch is not used to determine sigma.

    This prevents the generator from influencing the kernel
    bandwidth itself.
    """

    _validate_feature_batch(
        real_features,
        "real_features",
    )

    distance = pairwise_squared_distance(
        real_features,
        real_features,
    )

    nonzero_distance = distance[
        distance > 0
    ]

    if nonzero_distance.numel() == 0:

        sigma = 1.0

    else:

        median_distance = torch.median(
            nonzero_distance
        )

        sigma = torch.sqrt(
            torch.clamp(
                median_distance,
                min=EPSILON,
            )
            / 2.0
        )

        sigma = float(
            sigma.detach().cpu().item()
        )

    if min_sigma is None:
        min_sigma = max(
            math.sqrt(EPSILON),
            1e-6,
        )

    sigma = max(
        float(sigma),
        float(min_sigma),
    )

    if not math.isfinite(
        sigma
    ) or sigma <= 0:

        raise ValueError(
            "Estimated RBF bandwidth is invalid."
        )

    return sigma


# --------------------------------------------------------------------------------------------------
# 6.7 Distribution Guidance
# --------------------------------------------------------------------------------------------------

def distribution_guidance(
    real_features,
    synthetic_features,
    sigma=None,
):
    """
    Compute differentiable marginal-distribution discrepancy.

    The representation supplied to this function must already
    be in the same transformed feature space for both real and
    synthetic observations.

    Notebook 02 remains authoritative for the transformed
    numerical/categorical representation.
    """

    _validate_feature_batch(
        real_features,
        "real_features",
    )

    _validate_feature_batch(
        synthetic_features,
        "synthetic_features",
    )

    if (
        real_features.shape[1]
        != synthetic_features.shape[1]
    ):
        raise ValueError(
            "Real and synthetic distribution representations "
            "must have identical feature dimensions.\n"
            f"Real      : {real_features.shape[1]}\n"
            f"Synthetic : {synthetic_features.shape[1]}"
        )

    if sigma is None:

        sigma = estimate_rbf_sigma(
            real_features
        )

    loss = differentiable_mmd(
        real_features,
        synthetic_features,
        sigma=sigma,
    )

    if not torch.isfinite(
        loss
    ):
        raise FloatingPointError(
            "Distribution guidance produced a non-finite loss."
        )

    return loss


# --------------------------------------------------------------------------------------------------
# 6.8 Configuration Validation
# --------------------------------------------------------------------------------------------------

distribution_config = GUIDANCE_CONFIG.get(
    "distribution",
    {}
)

if not isinstance(
    distribution_config,
    dict,
):
    raise TypeError(
        "GUIDANCE_CONFIG['distribution'] must be a dictionary."
    )

configured_sigma = distribution_config.get(
    "sigma",
    None,
)

if configured_sigma is not None:

    if not math.isfinite(
        float(configured_sigma)
    ):
        raise ValueError(
            "Configured distribution sigma must be finite."
        )

    if float(configured_sigma) <= 0:
        raise ValueError(
            "Configured distribution sigma must be > 0."
        )


# --------------------------------------------------------------------------------------------------
# 6.9 Differentiability Smoke Test
# --------------------------------------------------------------------------------------------------

_test_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

_test_real = torch.randn(
    8,
    4,
    device=_test_device,
    dtype=torch.float32,
)

_test_synthetic = torch.randn(
    8,
    4,
    device=_test_device,
    dtype=torch.float32,
    requires_grad=True,
)

_test_loss = distribution_guidance(
    _test_real,
    _test_synthetic,
    sigma=configured_sigma,
)

if _test_loss.ndim != 0:
    raise RuntimeError(
        "Distribution guidance must return a scalar loss."
    )

if not torch.isfinite(
    _test_loss
):
    raise RuntimeError(
        "Distribution guidance smoke test produced "
        "a non-finite loss."
    )

if not _test_loss.requires_grad:
    raise RuntimeError(
        "Distribution guidance is not differentiable "
        "with respect to synthetic features."
    )

_test_loss.backward()

if (
    _test_synthetic.grad is None
):
    raise RuntimeError(
        "Distribution guidance did not propagate "
        "gradient to synthetic features."
    )

if not torch.isfinite(
    _test_synthetic.grad
).all():

    raise RuntimeError(
        "Distribution guidance produced non-finite "
        "synthetic gradients."
    )

if float(
    _test_synthetic.grad.abs().sum().item()
) <= 0:

    raise RuntimeError(
        "Distribution guidance produced zero "
        "synthetic gradient."
    )


# --------------------------------------------------------------------------------------------------
# 6.10 Cleanup Smoke-Test Graph
# --------------------------------------------------------------------------------------------------

del _test_real
del _test_synthetic
del _test_loss


# --------------------------------------------------------------------------------------------------
# 6.11 Final Section Output
# --------------------------------------------------------------------------------------------------

print()
print("✓ Distribution guidance defined.")
print("  Component : marginal-distribution guidance")
print("  Method    : differentiable RBF-MMD²")
print("  Kernel    : RBF")
print("  Bandwidth : configured or real-batch-derived")
print("  Gradient  : synthetic representation ✓")
print("  Non-finite handling : explicit validation")
print("  Statistical source  : Notebook 03")
print("  Representation source: Notebook 02")


6. DEFINE DISTRIBUTION GUIDANCE

✓ Distribution guidance defined.
  Component : marginal-distribution guidance
  Method    : differentiable RBF-MMD²
  Kernel    : RBF
  Bandwidth : configured or real-batch-derived
  Gradient  : synthetic representation ✓
  Non-finite handling : explicit validation
  Statistical source  : Notebook 03
  Representation source: Notebook 02


In [28]:
# ==================================================================================================
# 7. DEFINE NUMERICAL GUIDANCE
# ==================================================================================================

print("\n" + "=" * 100)
print("7. DEFINE NUMERICAL GUIDANCE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 7.0 Numerical Moment Guidance
# --------------------------------------------------------------------------------------------------

def numerical_moment_guidance(
    real_numeric,
    synthetic_numeric,
):
    """
    Differentiable numerical moment guidance.

    Compares the first two moments of each numerical feature:

        1. Mean
        2. Standard deviation

    The formulation follows the SPP-GAN statistical-guidance
    methodology:

        L_mom =
            mean(
                |mu_real - mu_syn|
                +
                |sigma_real - sigma_syn|
            )

    Both real and synthetic inputs must be represented in the
    same transformed numerical space.

    Gradient flow is required only through synthetic_numeric.
    """

    # ----------------------------------------------------------------------------------------------
    # A. Tensor Validation
    # ----------------------------------------------------------------------------------------------

    if not isinstance(
        real_numeric,
        torch.Tensor,
    ):
        raise TypeError(
            "real_numeric must be a torch.Tensor."
        )

    if not isinstance(
        synthetic_numeric,
        torch.Tensor,
    ):
        raise TypeError(
            "synthetic_numeric must be a torch.Tensor."
        )

    if real_numeric.ndim != 2:
        raise ValueError(
            "real_numeric must have shape [batch, features]. "
            f"Received: {tuple(real_numeric.shape)}"
        )

    if synthetic_numeric.ndim != 2:
        raise ValueError(
            "synthetic_numeric must have shape [batch, features]. "
            f"Received: {tuple(synthetic_numeric.shape)}"
        )

    if real_numeric.shape[1] != synthetic_numeric.shape[1]:
        raise ValueError(
            "Real and synthetic numerical feature dimensions "
            "must match.\n"
            f"Real      : {real_numeric.shape[1]}\n"
            f"Synthetic : {synthetic_numeric.shape[1]}"
        )

    if real_numeric.shape[0] < 1:
        raise ValueError(
            "real_numeric must contain at least one sample."
        )

    if synthetic_numeric.shape[0] < 1:
        raise ValueError(
            "synthetic_numeric must contain at least one sample."
        )

    if not torch.isfinite(
        real_numeric
    ).all():

        raise ValueError(
            "real_numeric contains non-finite values."
        )

    if not torch.isfinite(
        synthetic_numeric
    ).all():

        raise ValueError(
            "synthetic_numeric contains non-finite values."
        )


    # ----------------------------------------------------------------------------------------------
    # B. No Numerical Features
    # ----------------------------------------------------------------------------------------------

    if real_numeric.shape[1] == 0:

        return torch.zeros(
            (),
            device=synthetic_numeric.device,
            dtype=synthetic_numeric.dtype,
        )


    # ----------------------------------------------------------------------------------------------
    # C. Ensure Compatible Device and Dtype
    # ----------------------------------------------------------------------------------------------

    if real_numeric.device != synthetic_numeric.device:

        real_numeric = real_numeric.to(
            device=synthetic_numeric.device
        )

    if real_numeric.dtype != synthetic_numeric.dtype:

        real_numeric = real_numeric.to(
            dtype=synthetic_numeric.dtype
        )


    # ----------------------------------------------------------------------------------------------
    # D. Real and Synthetic First Moments
    # ----------------------------------------------------------------------------------------------

    real_mean = real_numeric.mean(
        dim=0
    )

    synthetic_mean = synthetic_numeric.mean(
        dim=0
    )


    # ----------------------------------------------------------------------------------------------
    # E. Real and Synthetic Second Central Moments
    # ----------------------------------------------------------------------------------------------

    real_std = real_numeric.std(
        dim=0,
        unbiased=False,
    )

    synthetic_std = synthetic_numeric.std(
        dim=0,
        unbiased=False,
    )


    # ----------------------------------------------------------------------------------------------
    # F. Mean Discrepancy
    # ----------------------------------------------------------------------------------------------

    mean_loss = torch.mean(
        torch.abs(
            synthetic_mean
            - real_mean
        )
    )


    # ----------------------------------------------------------------------------------------------
    # G. Standard-Deviation Discrepancy
    # ----------------------------------------------------------------------------------------------

    std_loss = torch.mean(
        torch.abs(
            synthetic_std
            - real_std
        )
    )


    # ----------------------------------------------------------------------------------------------
    # H. Moment Guidance
    # ----------------------------------------------------------------------------------------------

    moment_loss = (
        mean_loss
        + std_loss
    ) / 2.0


    # ----------------------------------------------------------------------------------------------
    # I. Numerical Stability Validation
    # ----------------------------------------------------------------------------------------------

    if not torch.isfinite(
        mean_loss
    ):
        raise FloatingPointError(
            "Numerical mean discrepancy is non-finite."
        )

    if not torch.isfinite(
        std_loss
    ):
        raise FloatingPointError(
            "Numerical standard-deviation discrepancy is non-finite."
        )

    if not torch.isfinite(
        moment_loss
    ):
        raise FloatingPointError(
            "Numerical moment guidance produced a non-finite loss."
        )

    if moment_loss.ndim != 0:
        raise RuntimeError(
            "Numerical moment guidance must return a scalar."
        )


    return moment_loss


# --------------------------------------------------------------------------------------------------
# 7.1 Differentiability Smoke Test
# --------------------------------------------------------------------------------------------------

_test_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

_test_real_numeric = torch.randn(
    16,
    4,
    device=_test_device,
    dtype=torch.float32,
)

_test_synthetic_numeric = torch.randn(
    16,
    4,
    device=_test_device,
    dtype=torch.float32,
    requires_grad=True,
)

_test_moment_loss = numerical_moment_guidance(
    _test_real_numeric,
    _test_synthetic_numeric,
)


# --------------------------------------------------------------------------------------------------
# 7.2 Validate Scalar Differentiable Loss
# --------------------------------------------------------------------------------------------------

if _test_moment_loss.ndim != 0:
    raise RuntimeError(
        "Numerical moment guidance did not return a scalar."
    )

if not torch.isfinite(
    _test_moment_loss
):
    raise RuntimeError(
        "Numerical moment guidance produced "
        "a non-finite loss."
    )

if not _test_moment_loss.requires_grad:
    raise RuntimeError(
        "Numerical moment guidance is not differentiable "
        "with respect to synthetic numerical features."
    )


# --------------------------------------------------------------------------------------------------
# 7.3 Backward Test
# --------------------------------------------------------------------------------------------------

_test_moment_loss.backward()

if _test_synthetic_numeric.grad is None:
    raise RuntimeError(
        "Numerical moment guidance did not propagate "
        "gradient to synthetic numerical features."
    )

if not torch.isfinite(
    _test_synthetic_numeric.grad
).all():

    raise RuntimeError(
        "Numerical moment guidance produced "
        "non-finite synthetic gradients."
    )

if float(
    _test_synthetic_numeric.grad.abs().sum().item()
) <= 0:

    raise RuntimeError(
        "Numerical moment guidance produced "
        "zero synthetic gradient."
    )


# --------------------------------------------------------------------------------------------------
# 7.4 Cleanup
# --------------------------------------------------------------------------------------------------

del _test_real_numeric
del _test_synthetic_numeric
del _test_moment_loss


# --------------------------------------------------------------------------------------------------
# 7.5 Final Output
# --------------------------------------------------------------------------------------------------

print(
    "✓ Numerical guidance defined."
)

print(
    "  Components : mean, standard deviation"
)

print(
    "  Loss       : mean absolute moment discrepancy"
)

print(
    "  Statistics : first two moments"
)

print(
    "  Gradient   : synthetic numerical features ✓"
)

print(
    "  Stability  : finite-value validation ✓"
)

print(
    "  Min/Max    : excluded from L_mom to match methodology"
)


7. DEFINE NUMERICAL GUIDANCE
✓ Numerical guidance defined.
  Components : mean, standard deviation
  Loss       : mean absolute moment discrepancy
  Statistics : first two moments
  Gradient   : synthetic numerical features ✓
  Stability  : finite-value validation ✓
  Min/Max    : excluded from L_mom to match methodology


In [30]:
# ==================================================================================================
# 8. DEFINE CATEGORICAL GUIDANCE
# ==================================================================================================

print("\n" + "=" * 100)
print("8. DEFINE CATEGORICAL GUIDANCE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 8.0 Configuration
# --------------------------------------------------------------------------------------------------

if "EPSILON" not in globals():
    raise RuntimeError(
        "EPSILON is not defined."
    )

if EPSILON <= 0:
    raise ValueError(
        "EPSILON must be > 0."
    )

CATEGORY_SIMPLEX_TOLERANCE = 1e-5


# --------------------------------------------------------------------------------------------------
# 8.1 Probability Tensor Validation
# --------------------------------------------------------------------------------------------------

def _validate_probability_vector(
    probability,
    name,
):
    """
    Validate a categorical probability vector.

    Required properties:

        p_k >= 0
        sum(p_k) ~= 1
        all values finite
    """

    if not isinstance(
        probability,
        torch.Tensor,
    ):
        raise TypeError(
            f"{name} must be a torch.Tensor."
        )

    if probability.ndim != 1:
        raise ValueError(
            f"{name} must be one-dimensional. "
            f"Received shape: {tuple(probability.shape)}"
        )

    if probability.numel() < 1:
        raise ValueError(
            f"{name} must contain at least one category."
        )

    if not torch.isfinite(
        probability
    ).all():

        raise ValueError(
            f"{name} contains non-finite values."
        )

    if (
        probability < -CATEGORY_SIMPLEX_TOLERANCE
    ).any():

        raise ValueError(
            f"{name} contains negative probabilities."
        )

    probability_sum = probability.sum()

    if not torch.isfinite(
        probability_sum
    ):

        raise ValueError(
            f"{name} has a non-finite probability sum."
        )

    if not torch.isclose(
        probability_sum,
        torch.tensor(
            1.0,
            device=probability.device,
            dtype=probability.dtype,
        ),
        atol=CATEGORY_SIMPLEX_TOLERANCE,
        rtol=CATEGORY_SIMPLEX_TOLERANCE,
    ):

        raise ValueError(
            f"{name} does not lie on the probability simplex.\n"
            f"Sum = {float(probability_sum.detach().cpu())}"
        )

    return True


# --------------------------------------------------------------------------------------------------
# 8.2 Categorical Probability Guidance
# --------------------------------------------------------------------------------------------------

def categorical_probability_guidance(
    real_probabilities,
    synthetic_probabilities,
):
    """
    Differentiable categorical marginal-distribution guidance.

    For every categorical feature j:

        L_cat,j =
            mean_k |p_real(j,k) - p_syn(j,k)|

    Overall categorical guidance:

        L_cat =
            mean_j L_cat,j

    The synthetic probability vectors remain differentiable
    so that the loss can propagate to the generator's
    categorical Gumbel-Softmax outputs.

    Inputs:
        real_probabilities:
            list of probability vectors from the real training
            distribution.

        synthetic_probabilities:
            list of differentiable probability vectors produced
            by the generator.
    """

    # ----------------------------------------------------------------------------------------------
    # A. Input Container Validation
    # ----------------------------------------------------------------------------------------------

    if real_probabilities is None:
        raise ValueError(
            "real_probabilities cannot be None."
        )

    if synthetic_probabilities is None:
        raise ValueError(
            "synthetic_probabilities cannot be None."
        )

    if not isinstance(
        real_probabilities,
        (list, tuple),
    ):
        raise TypeError(
            "real_probabilities must be a list or tuple."
        )

    if not isinstance(
        synthetic_probabilities,
        (list, tuple),
    ):
        raise TypeError(
            "synthetic_probabilities must be a list or tuple."
        )

    if len(real_probabilities) != len(
        synthetic_probabilities
    ):
        raise ValueError(
            "Real and synthetic categorical feature counts "
            "must match.\n"
            f"Real      : {len(real_probabilities)}\n"
            f"Synthetic : {len(synthetic_probabilities)}"
        )

    # ----------------------------------------------------------------------------------------------
    # B. No Categorical Features
    # ----------------------------------------------------------------------------------------------

    if len(real_probabilities) == 0:

        if len(synthetic_probabilities) != 0:
            raise ValueError(
                "Real categorical feature count is zero but "
                "synthetic categorical features are present."
            )

        return torch.zeros(
            (),
            device=DEVICE,
            dtype=torch.float32,
        )

    # ----------------------------------------------------------------------------------------------
    # C. Feature-Level Probability Validation
    # ----------------------------------------------------------------------------------------------

    losses = []

    for feature_index, (
        real_prob,
        synthetic_prob,
    ) in enumerate(
        zip(
            real_probabilities,
            synthetic_probabilities,
        )
    ):

        # ------------------------------------------------------------------------------------------
        # Convert real probability vector
        # ------------------------------------------------------------------------------------------

        if not isinstance(
            real_prob,
            torch.Tensor,
        ):

            real_prob = torch.as_tensor(
                real_prob,
                dtype=torch.float32,
                device=synthetic_prob.device,
            )

        # ------------------------------------------------------------------------------------------
        # Synthetic tensor validation
        # ------------------------------------------------------------------------------------------

        if not isinstance(
            synthetic_prob,
            torch.Tensor,
        ):
            raise TypeError(
                f"Synthetic categorical probability vector "
                f"{feature_index} must be a torch.Tensor."
            )

        # ------------------------------------------------------------------------------------------
        # Match device and dtype
        # ------------------------------------------------------------------------------------------

        real_prob = real_prob.to(
            device=synthetic_prob.device,
            dtype=synthetic_prob.dtype,
        )

        # ------------------------------------------------------------------------------------------
        # Validate cardinality
        # ------------------------------------------------------------------------------------------

        if real_prob.numel() != synthetic_prob.numel():

            raise ValueError(
                "Categorical cardinality mismatch.\n"
                f"Feature index : {feature_index}\n"
                f"Real categories      : {real_prob.numel()}\n"
                f"Synthetic categories : {synthetic_prob.numel()}"
            )

        # ------------------------------------------------------------------------------------------
        # Validate probability vectors
        # ------------------------------------------------------------------------------------------

        _validate_probability_vector(
            real_prob,
            f"real_probabilities[{feature_index}]",
        )

        _validate_probability_vector(
            synthetic_prob,
            f"synthetic_probabilities[{feature_index}]",
        )

        # ------------------------------------------------------------------------------------------
        # Feature-Level Probability Discrepancy
        # ------------------------------------------------------------------------------------------

        feature_loss = torch.mean(
            torch.abs(
                synthetic_prob
                - real_prob
            )
        )

        if not torch.isfinite(
            feature_loss
        ):

            raise FloatingPointError(
                f"Categorical guidance produced a non-finite "
                f"loss for feature {feature_index}."
            )

        losses.append(
            feature_loss
        )

    # ----------------------------------------------------------------------------------------------
    # D. Aggregate Categorical Guidance
    # ----------------------------------------------------------------------------------------------

    categorical_loss = torch.stack(
        losses
    ).mean()

    if categorical_loss.ndim != 0:
        raise RuntimeError(
            "Categorical guidance must return a scalar."
        )

    if not torch.isfinite(
        categorical_loss
    ):

        raise FloatingPointError(
            "Categorical guidance produced a non-finite loss."
        )

    return categorical_loss


# --------------------------------------------------------------------------------------------------
# 8.3 Differentiability Smoke Test
# --------------------------------------------------------------------------------------------------

_test_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

_test_real_probabilities = [
    torch.tensor(
        [0.20, 0.50, 0.30],
        device=_test_device,
        dtype=torch.float32,
    ),
    torch.tensor(
        [0.70, 0.30],
        device=_test_device,
        dtype=torch.float32,
    ),
    torch.tensor(
        [0.10, 0.20, 0.40, 0.30],
        device=_test_device,
        dtype=torch.float32,
    ),
]

_test_synthetic_probabilities = [
    torch.tensor(
        [0.25, 0.45, 0.30],
        device=_test_device,
        dtype=torch.float32,
        requires_grad=True,
    ),
    torch.tensor(
        [0.60, 0.40],
        device=_test_device,
        dtype=torch.float32,
        requires_grad=True,
    ),
    torch.tensor(
        [0.15, 0.25, 0.35, 0.25],
        device=_test_device,
        dtype=torch.float32,
        requires_grad=True,
    ),
]


# --------------------------------------------------------------------------------------------------
# 8.4 Forward Test
# --------------------------------------------------------------------------------------------------

_test_categorical_loss = categorical_probability_guidance(
    _test_real_probabilities,
    _test_synthetic_probabilities,
)

if _test_categorical_loss.ndim != 0:
    raise RuntimeError(
        "Categorical guidance did not return a scalar."
    )

if not torch.isfinite(
    _test_categorical_loss
):

    raise RuntimeError(
        "Categorical guidance produced a non-finite "
        "smoke-test loss."
    )

if not _test_categorical_loss.requires_grad:
    raise RuntimeError(
        "Categorical guidance is not differentiable "
        "with respect to synthetic probabilities."
    )


# --------------------------------------------------------------------------------------------------
# 8.5 Backward Test
# --------------------------------------------------------------------------------------------------

_test_categorical_loss.backward()

for feature_index, synthetic_prob in enumerate(
    _test_synthetic_probabilities
):

    if synthetic_prob.grad is None:

        raise RuntimeError(
            "Categorical guidance did not propagate gradient "
            f"to synthetic categorical feature {feature_index}."
        )

    if not torch.isfinite(
        synthetic_prob.grad
    ).all():

        raise RuntimeError(
            "Categorical guidance produced non-finite gradient "
            f"for synthetic categorical feature {feature_index}."
        )

    if float(
        synthetic_prob.grad.abs().sum().item()
    ) <= 0:

        raise RuntimeError(
            "Categorical guidance produced zero gradient "
            f"for synthetic categorical feature {feature_index}."
        )


# --------------------------------------------------------------------------------------------------
# 8.6 Cleanup
# --------------------------------------------------------------------------------------------------

del _test_real_probabilities
del _test_synthetic_probabilities
del _test_categorical_loss


# --------------------------------------------------------------------------------------------------
# 8.7 Final Output
# --------------------------------------------------------------------------------------------------

print(
    "✓ Categorical guidance defined."
)

print(
    "  Method : marginal category probability discrepancy"
)

print(
    "  Distance : mean absolute probability discrepancy"
)

print(
    "  Cardinality matching : validated"
)

print(
    "  Probability simplex  : validated"
)

print(
    "  Non-finite handling  : explicit rejection"
)

print(
    "  Gradient propagation : synthetic probabilities ✓"
)

print(
    "  Gumbel-Softmax compatibility : validated"
)


8. DEFINE CATEGORICAL GUIDANCE
✓ Categorical guidance defined.
  Method : marginal category probability discrepancy
  Distance : mean absolute probability discrepancy
  Cardinality matching : validated
  Probability simplex  : validated
  Non-finite handling  : explicit rejection
  Gradient propagation : synthetic probabilities ✓
  Gumbel-Softmax compatibility : validated


In [33]:
# ==================================================================================================
# 9. DEFINE DEPENDENCY GUIDANCE
# ==================================================================================================

print("\n" + "=" * 100)
print("9. DEFINE DEPENDENCY GUIDANCE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 9.0 Configuration
# --------------------------------------------------------------------------------------------------

DEPENDENCY_TOLERANCE = 1e-8
DEPENDENCY_GRADIENT_TOLERANCE = 1e-12
DEPENDENCY_SIMPLEX_TOLERANCE = 1e-5


# --------------------------------------------------------------------------------------------------
# 9.1 Validate Dependency Batch
# --------------------------------------------------------------------------------------------------

def _validate_dependency_batch(
    x,
    name,
):
    """
    Validate a dependency-analysis tensor.

    Expected shape:
        [batch, features]

    Requirements:
        - torch.Tensor
        - two-dimensional
        - at least two observations
        - finite values
    """

    if not isinstance(
        x,
        torch.Tensor,
    ):
        raise TypeError(
            f"{name} must be a torch.Tensor."
        )

    if x.ndim != 2:
        raise ValueError(
            f"{name} must be two-dimensional. "
            f"Received shape: {tuple(x.shape)}"
        )

    if x.shape[0] < 2:
        raise ValueError(
            f"{name} requires at least two observations."
        )

    if not torch.isfinite(
        x
    ).all():

        raise ValueError(
            f"{name} contains non-finite values."
        )

    return True


# --------------------------------------------------------------------------------------------------
# 9.2 Differentiable Pearson Correlation Matrix
# --------------------------------------------------------------------------------------------------

def differentiable_pearson_matrix(
    x,
):
    """
    Compute a differentiable Pearson correlation matrix.

    Input:
        x:
            Tensor of shape [batch, features]

    Output:
        Tensor of shape [features, features]

    Constant features are assigned zero dependency
    contribution because Pearson correlation is undefined
    for zero-variance features.
    """

    _validate_dependency_batch(
        x,
        "x",
    )

    n_features = x.shape[1]

    if n_features == 0:
        return x.new_zeros(
            (0, 0)
        )

    # ----------------------------------------------------------------------------------------------
    # Center features
    # ----------------------------------------------------------------------------------------------

    centered = (
        x
        - x.mean(
            dim=0,
            keepdim=True,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Sample covariance
    # ----------------------------------------------------------------------------------------------

    covariance = (
        centered.transpose(0, 1)
        @ centered
    ) / (
        x.shape[0] - 1
    )

    # ----------------------------------------------------------------------------------------------
    # Feature variances
    # ----------------------------------------------------------------------------------------------

    variance = torch.diagonal(
        covariance
    )

    # ----------------------------------------------------------------------------------------------
    # Standard deviations
    # ----------------------------------------------------------------------------------------------

    standard_deviation = torch.sqrt(
        torch.clamp(
            variance,
            min=DEPENDENCY_TOLERANCE,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Pearson denominator
    # ----------------------------------------------------------------------------------------------

    denominator = (
        standard_deviation[:, None]
        * standard_deviation[None, :]
    )

    # ----------------------------------------------------------------------------------------------
    # Correlation matrix
    # ----------------------------------------------------------------------------------------------

    correlation = covariance / (
        denominator
        + DEPENDENCY_TOLERANCE
    )

    # ----------------------------------------------------------------------------------------------
    # Handle constant features
    # ----------------------------------------------------------------------------------------------

    constant_mask = (
        variance
        <= DEPENDENCY_TOLERANCE
    )

    if constant_mask.any():

        correlation = correlation.clone()

        correlation[
            constant_mask,
            :,
        ] = 0.0

        correlation[
            :,
            constant_mask,
        ] = 0.0

    # ----------------------------------------------------------------------------------------------
    # Numerical stabilization
    # ----------------------------------------------------------------------------------------------

    correlation = torch.clamp(
        correlation,
        min=-1.0,
        max=1.0,
    )

    # ----------------------------------------------------------------------------------------------
    # Final validation
    # ----------------------------------------------------------------------------------------------

    if not torch.isfinite(
        correlation
    ).all():

        raise FloatingPointError(
            "Differentiable Pearson correlation produced "
            "non-finite values."
        )

    return correlation


# --------------------------------------------------------------------------------------------------
# 9.3 Dependency Frobenius Guidance
# --------------------------------------------------------------------------------------------------

def dependency_frobenius_loss(
    real_features,
    synthetic_features,
):
    """
    Differentiable dependency-preservation loss.

    Dependency representation:
        Pearson correlation matrix.

    Let:

        D_real = Pearson(real_features)
        D_syn  = Pearson(synthetic_features)

    The dependency discrepancy is:

        ||D_real - D_syn||_F

    The loss is normalized by the number of features:

        L_dep =
            ||D_real - D_syn||_F / F

    where F is the number of dependency features.

    The synthetic dependency matrix remains differentiable
    with respect to synthetic_features.
    """

    # ----------------------------------------------------------------------------------------------
    # Validate input tensors
    # ----------------------------------------------------------------------------------------------

    _validate_dependency_batch(
        real_features,
        "real_features",
    )

    _validate_dependency_batch(
        synthetic_features,
        "synthetic_features",
    )

    # ----------------------------------------------------------------------------------------------
    # Validate feature dimensions
    # ----------------------------------------------------------------------------------------------

    if (
        real_features.shape[1]
        != synthetic_features.shape[1]
    ):

        raise ValueError(
            "Real and synthetic feature dimensions must match.\n"
            f"Real      : {real_features.shape[1]}\n"
            f"Synthetic : {synthetic_features.shape[1]}"
        )

    n_features = real_features.shape[1]

    # ----------------------------------------------------------------------------------------------
    # No dependency features
    # ----------------------------------------------------------------------------------------------

    if n_features == 0:

        return synthetic_features.new_zeros(
            ()
        )

    # ----------------------------------------------------------------------------------------------
    # Align device and dtype
    # ----------------------------------------------------------------------------------------------

    real_features = real_features.to(
        device=synthetic_features.device,
        dtype=synthetic_features.dtype,
    )

    # ----------------------------------------------------------------------------------------------
    # Compute dependency matrices
    # ----------------------------------------------------------------------------------------------

    real_dependency = differentiable_pearson_matrix(
        real_features
    )

    synthetic_dependency = differentiable_pearson_matrix(
        synthetic_features
    )

    # ----------------------------------------------------------------------------------------------
    # Dependency difference
    # ----------------------------------------------------------------------------------------------

    difference = (
        synthetic_dependency
        - real_dependency
    )

    # ----------------------------------------------------------------------------------------------
    # Frobenius discrepancy
    #
    # torch.linalg.matrix_norm(..., ord="fro")
    # is used because matrix_norm supports the "fro"
    # norm specification.
    # ----------------------------------------------------------------------------------------------

    frobenius_norm = torch.linalg.matrix_norm(
        difference,
        ord="fro",
    )

    # ----------------------------------------------------------------------------------------------
    # Feature-normalized dependency loss
    # ----------------------------------------------------------------------------------------------

    dependency_loss = (
        frobenius_norm
        / float(n_features)
    )

    # ----------------------------------------------------------------------------------------------
    # Final scalar validation
    # ----------------------------------------------------------------------------------------------

    if dependency_loss.ndim != 0:

        raise RuntimeError(
            "Dependency guidance must return a scalar."
        )

    if not torch.isfinite(
        dependency_loss
    ):

        raise FloatingPointError(
            "Dependency guidance produced a non-finite loss."
        )

    return dependency_loss


# --------------------------------------------------------------------------------------------------
# 9.4 Differentiability Smoke Test
# --------------------------------------------------------------------------------------------------

_test_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ----------------------------------------------------------------------------------------------
# Real dependency test data
# ----------------------------------------------------------------------------------------------

_test_real_features = torch.tensor(
    [
        [0.10, 0.20, 0.30],
        [0.20, 0.30, 0.40],
        [0.30, 0.40, 0.50],
        [0.40, 0.50, 0.60],
        [0.50, 0.60, 0.70],
        [0.60, 0.70, 0.80],
    ],
    device=_test_device,
    dtype=torch.float32,
)


# ----------------------------------------------------------------------------------------------
# Synthetic dependency test data
# ----------------------------------------------------------------------------------------------

_test_synthetic_features = torch.tensor(
    [
        [0.15, 0.18, 0.32],
        [0.22, 0.31, 0.39],
        [0.28, 0.42, 0.51],
        [0.37, 0.47, 0.59],
        [0.53, 0.58, 0.68],
        [0.61, 0.73, 0.77],
    ],
    device=_test_device,
    dtype=torch.float32,
    requires_grad=True,
)


# --------------------------------------------------------------------------------------------------
# 9.5 Forward Test
# --------------------------------------------------------------------------------------------------

_test_dependency_loss = dependency_frobenius_loss(
    _test_real_features,
    _test_synthetic_features,
)


# ----------------------------------------------------------------------------------------------
# Scalar validation
# ----------------------------------------------------------------------------------------------

if _test_dependency_loss.ndim != 0:

    raise RuntimeError(
        "Dependency guidance did not return a scalar."
    )


# ----------------------------------------------------------------------------------------------
# Finite loss validation
# ----------------------------------------------------------------------------------------------

if not torch.isfinite(
    _test_dependency_loss
):

    raise RuntimeError(
        "Dependency guidance produced a non-finite "
        "smoke-test loss."
    )


# ----------------------------------------------------------------------------------------------
# Differentiability validation
# ----------------------------------------------------------------------------------------------

if not _test_dependency_loss.requires_grad:

    raise RuntimeError(
        "Dependency guidance is not differentiable "
        "with respect to synthetic features."
    )


# --------------------------------------------------------------------------------------------------
# 9.6 Backward Test
# --------------------------------------------------------------------------------------------------

_test_dependency_loss.backward()


# ----------------------------------------------------------------------------------------------
# Gradient existence
# ----------------------------------------------------------------------------------------------

if _test_synthetic_features.grad is None:

    raise RuntimeError(
        "Dependency guidance did not propagate gradient "
        "to synthetic features."
    )


# ----------------------------------------------------------------------------------------------
# Gradient finiteness
# ----------------------------------------------------------------------------------------------

if not torch.isfinite(
    _test_synthetic_features.grad
).all():

    raise RuntimeError(
        "Dependency guidance produced non-finite "
        "synthetic gradients."
    )


# ----------------------------------------------------------------------------------------------
# Aggregate gradient magnitude
# ----------------------------------------------------------------------------------------------

_test_gradient_norm = float(
    _test_synthetic_features.grad.abs().sum().item()
)

if (
    _test_gradient_norm
    <= DEPENDENCY_GRADIENT_TOLERANCE
):

    raise RuntimeError(
        "Dependency guidance produced zero aggregate "
        "gradient for synthetic features."
    )


# --------------------------------------------------------------------------------------------------
# 9.7 Constant-Feature Stability Test
# --------------------------------------------------------------------------------------------------

_test_constant_real = torch.tensor(
    [
        [1.0, 0.10],
        [1.0, 0.20],
        [1.0, 0.30],
        [1.0, 0.40],
    ],
    device=_test_device,
    dtype=torch.float32,
)

_test_constant_synthetic = torch.tensor(
    [
        [1.0, 0.15],
        [1.0, 0.25],
        [1.0, 0.35],
        [1.0, 0.45],
    ],
    device=_test_device,
    dtype=torch.float32,
    requires_grad=True,
)


_test_constant_loss = dependency_frobenius_loss(
    _test_constant_real,
    _test_constant_synthetic,
)


if not torch.isfinite(
    _test_constant_loss
):

    raise RuntimeError(
        "Dependency guidance failed the constant-feature "
        "stability test."
    )

if _test_constant_loss.ndim != 0:

    raise RuntimeError(
        "Constant-feature dependency test did not "
        "return a scalar."
    )


# --------------------------------------------------------------------------------------------------
# 9.8 Cleanup
# --------------------------------------------------------------------------------------------------

del _test_real_features
del _test_synthetic_features
del _test_dependency_loss
del _test_constant_real
del _test_constant_synthetic
del _test_constant_loss


# --------------------------------------------------------------------------------------------------
# 9.9 Final Output
# --------------------------------------------------------------------------------------------------

print(
    "✓ Dependency guidance defined."
)

print(
    "  Dependency representation : Pearson correlation matrix"
)

print(
    "  Loss : feature-normalized Frobenius discrepancy"
)

print(
    "  Feature alignment : explicit dimension validation"
)

print(
    "  Constant features : safely handled"
)

print(
    "  Non-finite handling : explicit rejection"
)

print(
    "  Gradient propagation : synthetic features ✓"
)

print(
    "  Statistical reference : Notebook 03 Pearson dependencies"
)

print(
    "  Differentiability : validated"
)

print(
    "  Constant-feature stability : validated"
)


9. DEFINE DEPENDENCY GUIDANCE
✓ Dependency guidance defined.
  Dependency representation : Pearson correlation matrix
  Loss : feature-normalized Frobenius discrepancy
  Feature alignment : explicit dimension validation
  Constant features : safely handled
  Non-finite handling : explicit rejection
  Gradient propagation : synthetic features ✓
  Statistical reference : Notebook 03 Pearson dependencies
  Differentiability : validated
  Constant-feature stability : validated


In [35]:
# ==================================================================================================
# 10. DEFINE CORRELATION VALIDATION
# ==================================================================================================

print("\n" + "=" * 100)
print("10. DEFINE CORRELATION VALIDATION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 10.0 Configuration
# --------------------------------------------------------------------------------------------------

CORRELATION_TOLERANCE = 1e-8


# --------------------------------------------------------------------------------------------------
# 10.1 Correlation Matrix Validation
# --------------------------------------------------------------------------------------------------

def validate_correlation_matrix(
    correlation,
    name="correlation",
):
    """
    Validate a Pearson correlation matrix.

    Required properties:
        - square matrix
        - finite values
        - diagonal approximately equal to 1 for
          non-constant features
        - symmetric structure
        - values bounded within [-1, 1]
    """

    if not isinstance(
        correlation,
        torch.Tensor,
    ):
        raise TypeError(
            f"{name} must be a torch.Tensor."
        )

    if correlation.ndim != 2:
        raise ValueError(
            f"{name} must be two-dimensional. "
            f"Received shape: {tuple(correlation.shape)}"
        )

    rows, columns = correlation.shape

    if rows != columns:
        raise ValueError(
            f"{name} must be square. "
            f"Received shape: {tuple(correlation.shape)}"
        )

    if not torch.isfinite(
        correlation
    ).all():

        raise ValueError(
            f"{name} contains non-finite values."
        )

    if correlation.numel() == 0:
        return True

    # ----------------------------------------------------------------------------------------------
    # Range validation
    # ----------------------------------------------------------------------------------------------

    if (
        correlation < -1.0 - CORRELATION_TOLERANCE
    ).any() or (
        correlation > 1.0 + CORRELATION_TOLERANCE
    ).any():

        raise ValueError(
            f"{name} contains values outside [-1, 1]."
        )

    # ----------------------------------------------------------------------------------------------
    # Symmetry validation
    # ----------------------------------------------------------------------------------------------

    if not torch.allclose(
        correlation,
        correlation.transpose(0, 1),
        atol=1e-5,
        rtol=1e-5,
    ):

        raise ValueError(
            f"{name} is not symmetric."
        )

    return True


# --------------------------------------------------------------------------------------------------
# 10.2 Differentiable Correlation Matrix
# --------------------------------------------------------------------------------------------------

def correlation_matrix(
    x,
):
    """
    Compute a differentiable Pearson correlation matrix.

    This function is intentionally aligned with the
    dependency representation used in Section 9.

    Input:
        x:
            [batch, features]

    Output:
        [features, features]
    """

    if not isinstance(
        x,
        torch.Tensor,
    ):
        raise TypeError(
            "x must be a torch.Tensor."
        )

    if x.ndim != 2:
        raise ValueError(
            f"x must be two-dimensional. "
            f"Received shape: {tuple(x.shape)}"
        )

    if x.shape[0] < 2:
        raise ValueError(
            "Correlation requires at least two observations."
        )

    if not torch.isfinite(
        x
    ).all():

        raise ValueError(
            "x contains non-finite values."
        )

    n_features = x.shape[1]

    if n_features == 0:

        return x.new_zeros(
            (0, 0)
        )

    centered = (
        x
        - x.mean(
            dim=0,
            keepdim=True,
        )
    )

    variance = torch.mean(
        centered.pow(2),
        dim=0,
    )

    std = torch.sqrt(
        torch.clamp(
            variance,
            min=CORRELATION_TOLERANCE,
        )
    )

    standardized = (
        centered
        / std.unsqueeze(0)
    )

    n = float(
        x.shape[0]
    )

    correlation = (
        standardized.transpose(0, 1)
        @ standardized
    ) / n

    # ----------------------------------------------------------------------------------------------
    # Constant-feature handling
    # ----------------------------------------------------------------------------------------------

    constant_mask = (
        variance
        <= CORRELATION_TOLERANCE
    )

    if constant_mask.any():

        correlation = correlation.clone()

        correlation[
            constant_mask,
            :,
        ] = 0.0

        correlation[
            :,
            constant_mask,
        ] = 0.0

    # ----------------------------------------------------------------------------------------------
    # Numerical stabilization
    # ----------------------------------------------------------------------------------------------

    correlation = torch.clamp(
        correlation,
        min=-1.0,
        max=1.0,
    )

    validate_correlation_matrix(
        correlation,
        name="correlation_matrix",
    )

    return correlation


# --------------------------------------------------------------------------------------------------
# 10.3 Correlation Representation Smoke Test
# --------------------------------------------------------------------------------------------------

_test_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

_test_real_features = torch.tensor(
    [
        [0.10, 0.20, 0.30],
        [0.20, 0.30, 0.40],
        [0.30, 0.40, 0.50],
        [0.40, 0.50, 0.60],
        [0.50, 0.60, 0.70],
        [0.60, 0.70, 0.80],
    ],
    device=_test_device,
    dtype=torch.float32,
)

_test_synthetic_features = torch.tensor(
    [
        [0.15, 0.18, 0.32],
        [0.22, 0.31, 0.39],
        [0.28, 0.42, 0.51],
        [0.37, 0.47, 0.59],
        [0.53, 0.58, 0.68],
        [0.61, 0.73, 0.77],
    ],
    device=_test_device,
    dtype=torch.float32,
    requires_grad=True,
)


# --------------------------------------------------------------------------------------------------
# 10.4 Forward Validation
# --------------------------------------------------------------------------------------------------

_test_real_corr = correlation_matrix(
    _test_real_features
)

_test_synthetic_corr = correlation_matrix(
    _test_synthetic_features
)

validate_correlation_matrix(
    _test_real_corr,
    name="test_real_correlation",
)

validate_correlation_matrix(
    _test_synthetic_corr,
    name="test_synthetic_correlation",
)


# --------------------------------------------------------------------------------------------------
# 10.5 Dimensional Validation
# --------------------------------------------------------------------------------------------------

if (
    _test_real_corr.shape
    != _test_synthetic_corr.shape
):

    raise RuntimeError(
        "Real and synthetic correlation matrices "
        "have different shapes."
    )


if (
    _test_real_corr.shape[0]
    != _test_real_features.shape[1]
):

    raise RuntimeError(
        "Correlation matrix dimension does not match "
        "the feature dimension."
    )


# --------------------------------------------------------------------------------------------------
# 10.6 Differentiability Validation
# --------------------------------------------------------------------------------------------------

_test_correlation_probe = (
    _test_synthetic_corr
    * _test_real_corr.detach()
).sum()


if not _test_correlation_probe.requires_grad:

    raise RuntimeError(
        "Correlation representation is not differentiable "
        "with respect to synthetic features."
    )


_test_correlation_probe.backward()


if _test_synthetic_features.grad is None:

    raise RuntimeError(
        "Correlation representation did not propagate "
        "gradient to synthetic features."
    )


if not torch.isfinite(
    _test_synthetic_features.grad
).all():

    raise RuntimeError(
        "Correlation representation produced "
        "non-finite gradients."
    )


if float(
    _test_synthetic_features.grad.abs().sum().item()
) <= 1e-12:

    raise RuntimeError(
        "Correlation representation produced zero "
        "aggregate gradient."
    )


# --------------------------------------------------------------------------------------------------
# 10.7 Constant-Feature Validation
# --------------------------------------------------------------------------------------------------

_test_constant_features = torch.tensor(
    [
        [1.0, 0.10],
        [1.0, 0.20],
        [1.0, 0.30],
        [1.0, 0.40],
    ],
    device=_test_device,
    dtype=torch.float32,
)

_test_constant_corr = correlation_matrix(
    _test_constant_features
)

validate_correlation_matrix(
    _test_constant_corr,
    name="constant_feature_correlation",
)


# --------------------------------------------------------------------------------------------------
# 10.8 Cleanup
# --------------------------------------------------------------------------------------------------

del _test_real_features
del _test_synthetic_features
del _test_real_corr
del _test_synthetic_corr
del _test_correlation_probe
del _test_constant_features
del _test_constant_corr


# --------------------------------------------------------------------------------------------------
# 10.9 Final Output
# --------------------------------------------------------------------------------------------------

print(
    "✓ Correlation representation validated."
)

print(
    "  Representation : differentiable Pearson correlation matrix"
)

print(
    "  Role : dependency representation used by L_dep"
)

print(
    "  Duplicate loss : not introduced"
)

print(
    "  Matrix properties : finite, symmetric, bounded"
)

print(
    "  Feature alignment : validated"
)

print(
    "  Gradient propagation : synthetic features ✓"
)

print(
    "  Constant features : safely handled"
)

print(
    "  Training objective : no independent L_corr term"
)

print(
    "  Dependency loss : implemented in Section 9"
)


10. DEFINE CORRELATION VALIDATION
✓ Correlation representation validated.
  Representation : differentiable Pearson correlation matrix
  Role : dependency representation used by L_dep
  Duplicate loss : not introduced
  Matrix properties : finite, symmetric, bounded
  Feature alignment : validated
  Gradient propagation : synthetic features ✓
  Constant features : safely handled
  Training objective : no independent L_corr term
  Dependency loss : implemented in Section 9


In [37]:
# ==================================================================================================
# 11. NORMALIZE GUIDANCE COMPONENTS
# ==================================================================================================

print("\n" + "=" * 100)
print("11. NORMALIZE GUIDANCE COMPONENTS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 11.0 Configuration
# --------------------------------------------------------------------------------------------------

GUIDANCE_SCALE_EPSILON = 1e-8
GUIDANCE_FINITE_TOLERANCE = 1e-12


# --------------------------------------------------------------------------------------------------
# 11.1 Validate Guidance Loss
# --------------------------------------------------------------------------------------------------

def _validate_guidance_loss(
    loss,
    name="guidance_loss",
):
    """
    Validate a guidance loss before normalization.

    Requirements:
        - torch.Tensor
        - scalar
        - finite
    """

    if not isinstance(
        loss,
        torch.Tensor,
    ):
        raise TypeError(
            f"{name} must be a torch.Tensor."
        )

    if loss.ndim != 0:
        raise ValueError(
            f"{name} must be a scalar tensor. "
            f"Received shape: {tuple(loss.shape)}"
        )

    if not torch.isfinite(
        loss
    ):

        raise FloatingPointError(
            f"{name} contains a non-finite value."
        )

    return True


# --------------------------------------------------------------------------------------------------
# 11.2 Validate Reference Scale
# --------------------------------------------------------------------------------------------------

def _validate_reference_scale(
    reference_scale,
    name="reference_scale",
):
    """
    Validate a fixed normalization scale.

    The scale must be:
        - finite
        - strictly positive
    """

    if isinstance(
        reference_scale,
        torch.Tensor,
    ):

        if reference_scale.numel() != 1:
            raise ValueError(
                f"{name} must contain exactly one value."
            )

        reference_scale = (
            reference_scale.detach()
            .cpu()
            .item()
        )

    try:
        reference_scale = float(
            reference_scale
        )

    except (
        TypeError,
        ValueError,
    ) as exc:

        raise TypeError(
            f"{name} must be numeric."
        ) from exc

    if not np.isfinite(
        reference_scale
    ):

        raise ValueError(
            f"{name} must be finite."
        )

    if (
        reference_scale
        <= 0.0
    ):

        raise ValueError(
            f"{name} must be strictly positive."
        )

    return reference_scale


# --------------------------------------------------------------------------------------------------
# 11.3 Normalize Guidance Loss
# --------------------------------------------------------------------------------------------------

def normalize_guidance_loss(
    loss,
    reference_scale=1.0,
):
    """
    Normalize a scalar guidance loss using a fixed,
    externally supplied reference scale.

    Formula:

        L_normalized =
            L / s

    where:
        L = guidance loss
        s = fixed positive reference scale

    The reference scale is detached from the computational
    graph and therefore does not introduce an additional
    gradient path.

    Important:
        The scale must be determined independently of the
        current synthetic batch.
    """

    _validate_guidance_loss(
        loss,
        name="guidance_loss",
    )

    scale = _validate_reference_scale(
        reference_scale,
        name="reference_scale",
    )

    normalized_loss = (
        loss
        / scale
    )

    if normalized_loss.ndim != 0:

        raise RuntimeError(
            "Normalized guidance loss must be scalar."
        )

    if not torch.isfinite(
        normalized_loss
    ):

        raise FloatingPointError(
            "Normalized guidance loss is non-finite."
        )

    return normalized_loss


# --------------------------------------------------------------------------------------------------
# 11.4 Safe Scalar Conversion
# --------------------------------------------------------------------------------------------------

def safe_float(
    value,
):
    """
    Convert a scalar/tensor to a finite Python float.

    Non-finite values raise an explicit error instead of
    being silently replaced with zero.
    """

    if isinstance(
        value,
        torch.Tensor,
    ):

        if value.numel() != 1:
            raise ValueError(
                "safe_float requires a scalar tensor."
            )

        value = (
            value.detach()
            .cpu()
            .item()
        )

    try:
        value = float(
            value
        )

    except (
        TypeError,
        ValueError,
    ) as exc:

        raise TypeError(
            "Value cannot be converted to float."
        ) from exc

    if not np.isfinite(
        value
    ):

        raise FloatingPointError(
            f"Non-finite scalar encountered: {value}"
        )

    return value


# --------------------------------------------------------------------------------------------------
# 11.5 Normalization Smoke Test
# --------------------------------------------------------------------------------------------------

_test_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

_test_loss = torch.tensor(
    0.40,
    device=_test_device,
    dtype=torch.float32,
    requires_grad=True,
)

_test_scale = 2.0


# --------------------------------------------------------------------------------------------------
# 11.6 Forward Test
# --------------------------------------------------------------------------------------------------

_validate_guidance_loss(
    _test_loss,
    name="test_loss",
)

_validate_reference_scale(
    _test_scale,
    name="test_scale",
)

_test_normalized_loss = normalize_guidance_loss(
    _test_loss,
    reference_scale=_test_scale,
)


if not torch.isfinite(
    _test_normalized_loss
):

    raise RuntimeError(
        "Guidance normalization produced "
        "a non-finite value."
    )


if _test_normalized_loss.ndim != 0:

    raise RuntimeError(
        "Guidance normalization did not return "
        "a scalar tensor."
    )


# --------------------------------------------------------------------------------------------------
# 11.7 Backward Test
# --------------------------------------------------------------------------------------------------

_test_normalized_loss.backward()


if _test_loss.grad is None:

    raise RuntimeError(
        "Guidance normalization did not preserve "
        "the gradient path."
    )


if not torch.isfinite(
    _test_loss.grad
).all():

    raise RuntimeError(
        "Guidance normalization produced "
        "a non-finite gradient."
    )


if float(
    _test_loss.grad.abs().sum().item()
) <= GUIDANCE_FINITE_TOLERANCE:

    raise RuntimeError(
        "Guidance normalization produced "
        "zero gradient."
    )


# --------------------------------------------------------------------------------------------------
# 11.8 Numerical Conversion Test
# --------------------------------------------------------------------------------------------------

_test_float = safe_float(
    _test_normalized_loss
)

if not isinstance(
    _test_float,
    float,
):

    raise RuntimeError(
        "safe_float did not return a Python float."
    )

if not np.isfinite(
    _test_float
):

    raise RuntimeError(
        "safe_float returned a non-finite value."
    )


# --------------------------------------------------------------------------------------------------
# 11.9 Invalid-Value Protection Test
# --------------------------------------------------------------------------------------------------

_invalid_value_rejected = False

try:

    safe_float(
        float("nan")
    )

except FloatingPointError:

    _invalid_value_rejected = True


if not _invalid_value_rejected:

    raise RuntimeError(
        "safe_float failed to reject a non-finite value."
    )


# --------------------------------------------------------------------------------------------------
# 11.10 Cleanup
# --------------------------------------------------------------------------------------------------

del _test_loss
del _test_normalized_loss


# --------------------------------------------------------------------------------------------------
# 11.11 Final Output
# --------------------------------------------------------------------------------------------------

print(
    "✓ Guidance normalization defined."
)

print(
    "  Method : fixed reference-scale normalization"
)

print(
    "  Scale source : externally supplied, fixed quantity"
)

print(
    "  Gradient path : preserved ✓"
)

print(
    "  Scalar validation : enforced ✓"
)

print(
    "  Non-finite handling : explicit rejection ✓"
)

print(
    "  Silent zero replacement : disabled ✓"
)

print(
    "  Synthetic-batch-dependent scaling : avoided ✓"
)

print(
    f"  ε = {GUIDANCE_SCALE_EPSILON}"
)


11. NORMALIZE GUIDANCE COMPONENTS
✓ Guidance normalization defined.
  Method : fixed reference-scale normalization
  Scale source : externally supplied, fixed quantity
  Gradient path : preserved ✓
  Scalar validation : enforced ✓
  Non-finite handling : explicit rejection ✓
  Silent zero replacement : disabled ✓
  Synthetic-batch-dependent scaling : avoided ✓
  ε = 1e-08


In [39]:
# ==================================================================================================
# 12. DEFINE STATISTICAL LOSS
# ==================================================================================================

print("\n" + "=" * 100)
print("12. DEFINE STATISTICAL LOSS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 12.0 Canonical SPP-GAN Statistical Objective
# --------------------------------------------------------------------------------------------------

#
# Canonical statistical objective:
#
# L_stat =
#     λ_m   L_marg
#   + λ_mom L_mom
#   + λ_d   L_dep
#   + λ_c   L_cat
#
# where:
#
#   L_marg : marginal/distribution discrepancy
#   L_mom  : numerical first/second-moment discrepancy
#   L_dep  : differentiable Pearson dependency discrepancy
#   L_cat  : categorical marginal probability discrepancy
#
# Pearson correlation is the dependency representation used
# inside L_dep. No independent L_corr term is introduced.
#


# --------------------------------------------------------------------------------------------------
# 12.1 Load Canonical Guidance Weights
# --------------------------------------------------------------------------------------------------

_REQUIRED_GUIDANCE_WEIGHTS = [
    "lambda_m",
    "lambda_mom",
    "lambda_d",
    "lambda_c",
]


if "GUIDANCE_CONFIG" not in globals():

    raise RuntimeError(
        "GUIDANCE_CONFIG is not available."
    )


if "loss" not in GUIDANCE_CONFIG:

    raise KeyError(
        "GUIDANCE_CONFIG['loss'] is not available."
    )


GUIDANCE_WEIGHTS = {}

for _weight_name in _REQUIRED_GUIDANCE_WEIGHTS:

    if _weight_name not in GUIDANCE_CONFIG["loss"]:

        raise KeyError(
            f"Missing guidance weight: {_weight_name}"
        )

    _weight_value = float(
        GUIDANCE_CONFIG["loss"][_weight_name]
    )

    if not np.isfinite(
        _weight_value
    ):

        raise ValueError(
            f"Guidance weight '{_weight_name}' is non-finite."
        )

    if _weight_value < 0.0:

        raise ValueError(
            f"Guidance weight '{_weight_name}' must be "
            "non-negative."
        )

    GUIDANCE_WEIGHTS[
        _weight_name
    ] = _weight_value


# --------------------------------------------------------------------------------------------------
# 12.2 Validate Guidance Component
# --------------------------------------------------------------------------------------------------

def _validate_statistical_component(
    component,
    name,
):
    """
    Validate an individual statistical guidance component.

    Requirements:
        - torch.Tensor
        - scalar
        - finite
    """

    if not isinstance(
        component,
        torch.Tensor,
    ):

        raise TypeError(
            f"{name} must be a torch.Tensor."
        )

    if component.ndim != 0:

        raise ValueError(
            f"{name} must be a scalar tensor. "
            f"Received shape: {tuple(component.shape)}"
        )

    if not torch.isfinite(
        component
    ):

        raise FloatingPointError(
            f"{name} is non-finite."
        )

    return True


# --------------------------------------------------------------------------------------------------
# 12.3 Validate Statistical Weights
# --------------------------------------------------------------------------------------------------

def _validate_statistical_weights(
    weights,
):
    """
    Validate the four canonical statistical-guidance weights.
    """

    if not isinstance(
        weights,
        dict,
    ):

        raise TypeError(
            "weights must be a dictionary."
        )

    missing = [
        key
        for key in _REQUIRED_GUIDANCE_WEIGHTS
        if key not in weights
    ]

    if missing:

        raise KeyError(
            "Missing statistical guidance weights: "
            + ", ".join(missing)
        )

    validated = {}

    for key in _REQUIRED_GUIDANCE_WEIGHTS:

        value = float(
            weights[key]
        )

        if not np.isfinite(
            value
        ):

            raise ValueError(
                f"Weight '{key}' is non-finite."
            )

        if value < 0.0:

            raise ValueError(
                f"Weight '{key}' must be non-negative."
            )

        validated[key] = value

    if not any(
        value > 0.0
        for value in validated.values()
    ):

        raise ValueError(
            "At least one statistical guidance weight "
            "must be positive."
        )

    return validated


# --------------------------------------------------------------------------------------------------
# 12.4 Complete Statistical Guidance Loss
# --------------------------------------------------------------------------------------------------

def statistical_guidance_loss(
    real_numeric,
    synthetic_numeric,
    real_distribution_features,
    synthetic_distribution_features,
    real_categorical_probabilities=None,
    synthetic_categorical_probabilities=None,
    weights=None,
    return_components=False,
):
    """
    Complete differentiable SPP-GAN statistical guidance.

    Components:

        L_marg :
            differentiable marginal/distribution discrepancy

        L_mom :
            numerical mean/std discrepancy

        L_dep :
            differentiable Pearson dependency discrepancy

        L_cat :
            categorical marginal probability discrepancy

    Canonical objective:

        L_stat =
            λ_m L_marg
            + λ_mom L_mom
            + λ_d L_dep
            + λ_c L_cat

    No independent correlation loss is included.

    Parameters
    ----------
    real_numeric :
        Real numerical representation.

    synthetic_numeric :
        Synthetic numerical representation.

    real_distribution_features :
        Real representation used for marginal/distribution
        and dependency guidance.

    synthetic_distribution_features :
        Synthetic differentiable representation corresponding
        to the real distribution representation.

    real_categorical_probabilities :
        Optional list of real categorical probability vectors.

    synthetic_categorical_probabilities :
        Optional list of differentiable synthetic categorical
        probability vectors.

    weights :
        Optional dictionary containing the four canonical
        statistical-guidance weights.

    return_components :
        If True, return total loss and component dictionary.

    Returns
    -------
    total_loss
        Scalar differentiable statistical guidance loss.

    components
        Dictionary of L_marg, L_mom, L_dep, and L_cat when
        return_components=True.
    """

    # ----------------------------------------------------------------------------------------------
    # A. Resolve and validate weights
    # ----------------------------------------------------------------------------------------------

    if weights is None:

        weights = GUIDANCE_WEIGHTS

    weights = _validate_statistical_weights(
        weights
    )

    # ----------------------------------------------------------------------------------------------
    # B. Distribution / marginal guidance
    # ----------------------------------------------------------------------------------------------

    l_marg = distribution_guidance(
        real_distribution_features,
        synthetic_distribution_features,
    )

    _validate_statistical_component(
        l_marg,
        "L_marg",
    )

    # ----------------------------------------------------------------------------------------------
    # C. Numerical moment guidance
    # ----------------------------------------------------------------------------------------------

    l_mom = numerical_moment_guidance(
        real_numeric,
        synthetic_numeric,
    )

    _validate_statistical_component(
        l_mom,
        "L_mom",
    )

    # ----------------------------------------------------------------------------------------------
    # D. Dependency guidance
    #
    # IMPORTANT:
    # Section 9 owns the dependency loss.
    # Do not call the obsolete covariance_frobenius_loss().
    # ----------------------------------------------------------------------------------------------

    l_dep = dependency_frobenius_loss(
        real_distribution_features,
        synthetic_distribution_features,
    )

    _validate_statistical_component(
        l_dep,
        "L_dep",
    )

    # ----------------------------------------------------------------------------------------------
    # E. Categorical guidance
    # ----------------------------------------------------------------------------------------------

    if (
        real_categorical_probabilities is None
        and
        synthetic_categorical_probabilities is None
    ):

        l_cat = l_marg.new_zeros(
            ()
        )

    elif (
        real_categorical_probabilities is not None
        and
        synthetic_categorical_probabilities is not None
    ):

        l_cat = categorical_probability_guidance(
            real_categorical_probabilities,
            synthetic_categorical_probabilities,
        )

    else:

        raise ValueError(
            "real_categorical_probabilities and "
            "synthetic_categorical_probabilities must "
            "either both be provided or both be None."
        )

    _validate_statistical_component(
        l_cat,
        "L_cat",
    )

    # ----------------------------------------------------------------------------------------------
    # F. Weighted Statistical Objective
    # ----------------------------------------------------------------------------------------------

    total = (
        weights["lambda_m"] * l_marg
        +
        weights["lambda_mom"] * l_mom
        +
        weights["lambda_d"] * l_dep
        +
        weights["lambda_c"] * l_cat
    )

    # ----------------------------------------------------------------------------------------------
    # G. Final Loss Validation
    # ----------------------------------------------------------------------------------------------

    _validate_statistical_component(
        total,
        "L_stat",
    )

    # ----------------------------------------------------------------------------------------------
    # H. Return Components
    # ----------------------------------------------------------------------------------------------

    if return_components:

        return total, {
            "L_marg": l_marg,
            "L_mom": l_mom,
            "L_dep": l_dep,
            "L_cat": l_cat,
        }

    return total


# --------------------------------------------------------------------------------------------------
# 12.5 Statistical Objective Smoke Test
# --------------------------------------------------------------------------------------------------

_test_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ----------------------------------------------------------------------------------------------
# Numerical test tensors
# ----------------------------------------------------------------------------------------------

_test_real_numeric = torch.tensor(
    [
        [0.10, 0.20],
        [0.20, 0.30],
        [0.30, 0.40],
        [0.40, 0.50],
        [0.50, 0.60],
        [0.60, 0.70],
    ],
    device=_test_device,
    dtype=torch.float32,
)

_test_synthetic_numeric = torch.tensor(
    [
        [0.12, 0.18],
        [0.21, 0.32],
        [0.29, 0.39],
        [0.42, 0.48],
        [0.52, 0.58],
        [0.63, 0.72],
    ],
    device=_test_device,
    dtype=torch.float32,
    requires_grad=True,
)


# ----------------------------------------------------------------------------------------------
# Distribution representation test tensors
# ----------------------------------------------------------------------------------------------

_test_real_distribution = torch.tensor(
    [
        [0.10, 0.20, 0.30],
        [0.20, 0.30, 0.40],
        [0.30, 0.40, 0.50],
        [0.40, 0.50, 0.60],
        [0.50, 0.60, 0.70],
        [0.60, 0.70, 0.80],
    ],
    device=_test_device,
    dtype=torch.float32,
)

_test_synthetic_distribution = torch.tensor(
    [
        [0.12, 0.18, 0.32],
        [0.21, 0.31, 0.39],
        [0.29, 0.42, 0.51],
        [0.42, 0.48, 0.59],
        [0.52, 0.58, 0.68],
        [0.63, 0.72, 0.77],
    ],
    device=_test_device,
    dtype=torch.float32,
    requires_grad=True,
)


# ----------------------------------------------------------------------------------------------
# Categorical probability test tensors
# ----------------------------------------------------------------------------------------------

_test_real_categorical = [
    torch.tensor(
        [0.20, 0.50, 0.30],
        device=_test_device,
        dtype=torch.float32,
    ),
    torch.tensor(
        [0.70, 0.30],
        device=_test_device,
        dtype=torch.float32,
    ),
]

_test_synthetic_categorical = [
    torch.tensor(
        [0.25, 0.45, 0.30],
        device=_test_device,
        dtype=torch.float32,
        requires_grad=True,
    ),
    torch.tensor(
        [0.60, 0.40],
        device=_test_device,
        dtype=torch.float32,
        requires_grad=True,
    ),
]


# --------------------------------------------------------------------------------------------------
# 12.6 Execute Smoke Test
# --------------------------------------------------------------------------------------------------

_test_total, _test_components = statistical_guidance_loss(
    real_numeric=_test_real_numeric,
    synthetic_numeric=_test_synthetic_numeric,
    real_distribution_features=_test_real_distribution,
    synthetic_distribution_features=_test_synthetic_distribution,
    real_categorical_probabilities=_test_real_categorical,
    synthetic_categorical_probabilities=_test_synthetic_categorical,
    weights={
        "lambda_m": 1.0,
        "lambda_mom": 1.0,
        "lambda_d": 1.0,
        "lambda_c": 1.0,
    },
    return_components=True,
)


# --------------------------------------------------------------------------------------------------
# 12.7 Component Validation
# --------------------------------------------------------------------------------------------------

_expected_components = [
    "L_marg",
    "L_mom",
    "L_dep",
    "L_cat",
]

if set(
    _test_components.keys()
) != set(
    _expected_components
):

    raise RuntimeError(
        "Statistical loss component set is incorrect."
    )


for _component_name in _expected_components:

    _validate_statistical_component(
        _test_components[_component_name],
        _component_name,
    )


# --------------------------------------------------------------------------------------------------
# 12.8 Total Loss Validation
# --------------------------------------------------------------------------------------------------

_validate_statistical_component(
    _test_total,
    "L_stat",
)


if not _test_total.requires_grad:

    raise RuntimeError(
        "L_stat is not differentiable."
    )


# --------------------------------------------------------------------------------------------------
# 12.9 Backward Test
# --------------------------------------------------------------------------------------------------

_test_total.backward()


# ----------------------------------------------------------------------------------------------
# Numerical gradient
# ----------------------------------------------------------------------------------------------

if _test_synthetic_numeric.grad is None:

    raise RuntimeError(
        "L_stat did not propagate gradient to "
        "synthetic numerical features."
    )


if not torch.isfinite(
    _test_synthetic_numeric.grad
).all():

    raise RuntimeError(
        "L_stat produced non-finite numerical gradients."
    )


if float(
    _test_synthetic_numeric.grad.abs().sum().item()
) <= 1e-12:

    raise RuntimeError(
        "L_stat produced zero aggregate numerical gradient."
    )


# ----------------------------------------------------------------------------------------------
# Distribution gradient
# ----------------------------------------------------------------------------------------------

if _test_synthetic_distribution.grad is None:

    raise RuntimeError(
        "L_stat did not propagate gradient to "
        "synthetic distribution features."
    )


if not torch.isfinite(
    _test_synthetic_distribution.grad
).all():

    raise RuntimeError(
        "L_stat produced non-finite distribution gradients."
    )


if float(
    _test_synthetic_distribution.grad.abs().sum().item()
) <= 1e-12:

    raise RuntimeError(
        "L_stat produced zero aggregate distribution gradient."
    )


# ----------------------------------------------------------------------------------------------
# Categorical gradients
# ----------------------------------------------------------------------------------------------

for _feature_index, _synthetic_probability in enumerate(
    _test_synthetic_categorical
):

    if _synthetic_probability.grad is None:

        raise RuntimeError(
            "L_stat did not propagate gradient to "
            f"categorical feature {_feature_index}."
        )

    if not torch.isfinite(
        _synthetic_probability.grad
    ).all():

        raise RuntimeError(
            "L_stat produced non-finite categorical "
            f"gradients for feature {_feature_index}."
        )


# --------------------------------------------------------------------------------------------------
# 12.10 Zero-Categorical-Feature Test
# --------------------------------------------------------------------------------------------------

_test_total_no_categories = statistical_guidance_loss(
    real_numeric=_test_real_numeric,
    synthetic_numeric=_test_synthetic_numeric.detach().requires_grad_(True),
    real_distribution_features=_test_real_distribution,
    synthetic_distribution_features=_test_synthetic_distribution.detach().requires_grad_(True),
    real_categorical_probabilities=None,
    synthetic_categorical_probabilities=None,
    weights={
        "lambda_m": 1.0,
        "lambda_mom": 1.0,
        "lambda_d": 1.0,
        "lambda_c": 1.0,
    },
)


_validate_statistical_component(
    _test_total_no_categories,
    "L_stat_no_categories",
)


# --------------------------------------------------------------------------------------------------
# 12.11 Cleanup
# --------------------------------------------------------------------------------------------------

del _test_real_numeric
del _test_synthetic_numeric
del _test_real_distribution
del _test_synthetic_distribution
del _test_real_categorical
del _test_synthetic_categorical
del _test_total
del _test_components
del _test_total_no_categories


# --------------------------------------------------------------------------------------------------
# 12.12 Final Output
# --------------------------------------------------------------------------------------------------

print(
    "✓ Statistical loss defined."
)

print()

print(
    "L_stat = λ_m L_marg + λ_mom L_mom + "
    "λ_d L_dep + λ_c L_cat"
)

print()

print(
    "  L_marg : differentiable distribution guidance ✓"
)

print(
    "  L_mom  : numerical mean/std guidance ✓"
)

print(
    "  L_dep  : Pearson dependency Frobenius guidance ✓"
)

print(
    "  L_cat  : categorical probability guidance ✓"
)

print(
    "  Component validation : finite scalar losses ✓"
)

print(
    "  Weight validation : finite and non-negative ✓"
)

print(
    "  Total differentiability : validated ✓"
)

print(
    "  Numerical gradient propagation : validated ✓"
)

print(
    "  Dependency gradient propagation : validated ✓"
)

print(
    "  Categorical gradient propagation : validated ✓"
)

print(
    "  Independent L_corr : not included ✓"
)

print(
    "  Silent NaN/Inf replacement : disabled ✓"
)


12. DEFINE STATISTICAL LOSS
✓ Statistical loss defined.

L_stat = λ_m L_marg + λ_mom L_mom + λ_d L_dep + λ_c L_cat

  L_marg : differentiable distribution guidance ✓
  L_mom  : numerical mean/std guidance ✓
  L_dep  : Pearson dependency Frobenius guidance ✓
  L_cat  : categorical probability guidance ✓
  Component validation : finite scalar losses ✓
  Weight validation : finite and non-negative ✓
  Total differentiability : validated ✓
  Numerical gradient propagation : validated ✓
  Dependency gradient propagation : validated ✓
  Categorical gradient propagation : validated ✓
  Independent L_corr : not included ✓
  Silent NaN/Inf replacement : disabled ✓


In [40]:
# ==================================================================================================
# 13. DEFINE GUIDANCE WEIGHTS
# ==================================================================================================

print("\n" + "=" * 100)
print("13. DEFINE GUIDANCE WEIGHTS")
print("=" * 100)

WEIGHT_SUM = sum(
    GUIDANCE_WEIGHTS.values()
)

if WEIGHT_SUM <= 0:
    raise ValueError(
        "Statistical guidance weights must have a positive sum."
    )

print("Guidance weights:")

for name, value in GUIDANCE_WEIGHTS.items():
    print(
        f"  {name:<12}: {value:.6f}"
    )

print(
    f"\nWeight sum     : {WEIGHT_SUM:.6f}"
)

print(
    f"Generator λ_stat: "
    f"{SPPGAN_CONFIG['lambda_stat']:.6f}"
)


13. DEFINE GUIDANCE WEIGHTS
Guidance weights:
  lambda_m    : 0.250000
  lambda_mom  : 0.250000
  lambda_d    : 0.250000
  lambda_c    : 0.250000

Weight sum     : 1.000000
Generator λ_stat: 1.000000


In [42]:
# ==================================================================================================
# 14. INTEGRATE GUIDANCE INTO GENERATOR OBJECTIVE
# ==================================================================================================

print("\n" + "=" * 100)
print("14. INTEGRATE GUIDANCE INTO GENERATOR OBJECTIVE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 14.1 Configuration
# --------------------------------------------------------------------------------------------------

GENERATOR_OBJECTIVE_TOLERANCE = 1e-12


# --------------------------------------------------------------------------------------------------
# 14.2 Validate Objective Component
# --------------------------------------------------------------------------------------------------

def _validate_generator_objective_component(
    value,
    name,
):
    """
    Validate a scalar generator-objective component.
    """

    if not isinstance(
        value,
        torch.Tensor,
    ):
        raise TypeError(
            f"{name} must be a torch.Tensor."
        )

    if value.ndim != 0:
        raise ValueError(
            f"{name} must be a scalar tensor. "
            f"Received shape: {tuple(value.shape)}"
        )

    if not torch.isfinite(
        value
    ):
        raise FloatingPointError(
            f"{name} is non-finite."
        )

    return True


# --------------------------------------------------------------------------------------------------
# 14.3 Validate lambda_stat
# --------------------------------------------------------------------------------------------------

def _validate_lambda_stat(
    lambda_stat,
):
    """
    Validate the generator statistical-guidance coefficient.

    lambda_stat must be:
        - numeric
        - finite
        - non-negative
    """

    if isinstance(
        lambda_stat,
        torch.Tensor,
    ):

        if lambda_stat.numel() != 1:
            raise ValueError(
                "lambda_stat must contain exactly one value."
            )

        lambda_stat = (
            lambda_stat.detach()
            .cpu()
            .item()
        )

    try:

        lambda_stat = float(
            lambda_stat
        )

    except (
        TypeError,
        ValueError,
    ) as exc:

        raise TypeError(
            "lambda_stat must be numeric."
        ) from exc

    if not np.isfinite(
        lambda_stat
    ):

        raise ValueError(
            "lambda_stat must be finite."
        )

    if lambda_stat < 0.0:

        raise ValueError(
            "lambda_stat must be non-negative."
        )

    return lambda_stat


# --------------------------------------------------------------------------------------------------
# 14.4 SPP-GAN Generator Objective
# --------------------------------------------------------------------------------------------------

def sppgan_generator_objective(
    adversarial_loss,
    statistical_loss,
    lambda_stat=None,
):
    """
    Canonical SPP-GAN generator objective:

        L_G = L_adv + lambda_stat * L_stat

    where:

        L_adv  = adversarial generator objective
        L_stat = statistical guidance objective

    Statistical guidance is incorporated as an objective
    term rather than as a privacy term.

    Privacy mechanisms and privacy accounting are handled
    by their designated downstream notebooks.
    """

    # ----------------------------------------------------------------------------------------------
    # Resolve configured lambda_stat
    # ----------------------------------------------------------------------------------------------

    if lambda_stat is None:

        if "lambda_stat" not in SPPGAN_CONFIG:

            raise KeyError(
                "SPPGAN_CONFIG['lambda_stat'] is not available."
            )

        lambda_stat = SPPGAN_CONFIG[
            "lambda_stat"
        ]

    lambda_stat = _validate_lambda_stat(
        lambda_stat
    )

    # ----------------------------------------------------------------------------------------------
    # Validate objective components
    # ----------------------------------------------------------------------------------------------

    _validate_generator_objective_component(
        adversarial_loss,
        "adversarial_loss",
    )

    _validate_generator_objective_component(
        statistical_loss,
        "statistical_loss",
    )

    # ----------------------------------------------------------------------------------------------
    # Device consistency
    # ----------------------------------------------------------------------------------------------

    if (
        adversarial_loss.device
        != statistical_loss.device
    ):

        raise ValueError(
            "Adversarial and statistical losses must "
            "reside on the same device."
        )

    # ----------------------------------------------------------------------------------------------
    # Dtype consistency
    # ----------------------------------------------------------------------------------------------

    if (
        adversarial_loss.dtype
        != statistical_loss.dtype
    ):

        raise ValueError(
            "Adversarial and statistical losses must "
            "use the same dtype."
        )

    # ----------------------------------------------------------------------------------------------
    # Combined generator objective
    # ----------------------------------------------------------------------------------------------

    total_generator_loss = (
        adversarial_loss
        +
        lambda_stat
        *
        statistical_loss
    )

    # ----------------------------------------------------------------------------------------------
    # Validate combined objective
    # ----------------------------------------------------------------------------------------------

    _validate_generator_objective_component(
        total_generator_loss,
        "total_generator_loss",
    )

    return total_generator_loss


# --------------------------------------------------------------------------------------------------
# 14.5 Differentiability Smoke Test
# --------------------------------------------------------------------------------------------------

_test_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


_test_adversarial_loss = torch.tensor(
    0.40,
    device=_test_device,
    dtype=torch.float32,
    requires_grad=True,
)

_test_statistical_loss = torch.tensor(
    0.20,
    device=_test_device,
    dtype=torch.float32,
    requires_grad=True,
)

_test_lambda_stat = 1.0


# --------------------------------------------------------------------------------------------------
# 14.6 Forward Test
# --------------------------------------------------------------------------------------------------

_test_generator_loss = sppgan_generator_objective(
    adversarial_loss=_test_adversarial_loss,
    statistical_loss=_test_statistical_loss,
    lambda_stat=_test_lambda_stat,
)


_validate_generator_objective_component(
    _test_generator_loss,
    "test_generator_loss",
)


if not _test_generator_loss.requires_grad:

    raise RuntimeError(
        "Combined generator objective is not differentiable."
    )


# --------------------------------------------------------------------------------------------------
# 14.7 Backward Test
# --------------------------------------------------------------------------------------------------

_test_generator_loss.backward()


if _test_adversarial_loss.grad is None:

    raise RuntimeError(
        "Generator objective did not preserve the "
        "adversarial gradient path."
    )


if _test_statistical_loss.grad is None:

    raise RuntimeError(
        "Generator objective did not preserve the "
        "statistical-guidance gradient path."
    )


if not torch.isfinite(
    _test_adversarial_loss.grad
).all():

    raise RuntimeError(
        "Adversarial gradient is non-finite."
    )


if not torch.isfinite(
    _test_statistical_loss.grad
).all():

    raise RuntimeError(
        "Statistical-guidance gradient is non-finite."
    )


if float(
    _test_adversarial_loss.grad.abs().sum().item()
) <= GENERATOR_OBJECTIVE_TOLERANCE:

    raise RuntimeError(
        "Adversarial gradient path is zero."
    )


if float(
    _test_statistical_loss.grad.abs().sum().item()
) <= GENERATOR_OBJECTIVE_TOLERANCE:

    raise RuntimeError(
        "Statistical-guidance gradient path is zero."
    )


# --------------------------------------------------------------------------------------------------
# 14.8 Mathematical Consistency Test
# --------------------------------------------------------------------------------------------------

_expected_generator_loss = (
    0.40
    +
    1.0
    *
    0.20
)

_observed_generator_loss = float(
    _test_generator_loss.detach()
    .cpu()
    .item()
)

if not np.isclose(
    _observed_generator_loss,
    _expected_generator_loss,
    atol=1e-7,
    rtol=1e-7,
):

    raise RuntimeError(
        "Generator objective does not match the canonical "
        "L_G = L_adv + lambda_stat * L_stat formulation."
    )


# --------------------------------------------------------------------------------------------------
# 14.9 Cleanup
# --------------------------------------------------------------------------------------------------

del _test_adversarial_loss
del _test_statistical_loss
del _test_generator_loss


# --------------------------------------------------------------------------------------------------
# 14.10 Final Output
# --------------------------------------------------------------------------------------------------

print(
    "✓ Generator objective defined."
)

print()

print(
    "L_G = L_adv + λ_stat × L_stat"
)

print()

print(
    f"λ_stat = "
    f"{SPPGAN_CONFIG['lambda_stat']:.6f}"
)

print()

print(
    "  Adversarial objective : validated ✓"
)

print(
    "  Statistical objective : validated ✓"
)

print(
    "  Combined objective : finite scalar ✓"
)

print(
    "  Adversarial gradient path : preserved ✓"
)

print(
    "  Statistical gradient path : preserved ✓"
)

print(
    "  Mathematical formulation : validated ✓"
)

print(
    "  Privacy term in Notebook 09 : NONE"
)

print(
    "  Privacy mechanism : handled in downstream DP notebooks"
)

print(
    "  End-to-end privacy claim : NOT established by this section"
)


14. INTEGRATE GUIDANCE INTO GENERATOR OBJECTIVE
✓ Generator objective defined.

L_G = L_adv + λ_stat × L_stat

λ_stat = 1.000000

  Adversarial objective : validated ✓
  Statistical objective : validated ✓
  Combined objective : finite scalar ✓
  Adversarial gradient path : preserved ✓
  Statistical gradient path : preserved ✓
  Mathematical formulation : validated ✓
  Privacy term in Notebook 09 : NONE
  Privacy mechanism : handled in downstream DP notebooks
  End-to-end privacy claim : NOT established by this section


In [48]:
# ==================================================================================================
# 15. TEST GUIDANCE COMPUTATION
# ==================================================================================================

print("\n" + "=" * 100)
print("15. TEST GUIDANCE COMPUTATION")
print("=" * 100)

GUIDANCE_COMPUTATION_RESULTS = []

TEST_BATCH_SIZE = int(
    GUIDANCE_CONFIG["testing"]["batch_size"]
)

if TEST_BATCH_SIZE < 2:
    raise ValueError(
        "Testing batch size must be at least 2."
    )


# --------------------------------------------------------------------------------------------------
# 15.1 Validate Notebook 08 Architecture Summary
# --------------------------------------------------------------------------------------------------

REQUIRED_ARCHITECTURE_COLUMNS = [
    "dataset",
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
]

missing_columns = [
    column
    for column in REQUIRED_ARCHITECTURE_COLUMNS
    if column not in ARCHITECTURE_SUMMARY_DF.columns
]

if missing_columns:

    raise RuntimeError(
        "Notebook 08 architecture summary is missing required columns: "
        + ", ".join(missing_columns)
    )


# --------------------------------------------------------------------------------------------------
# 15.2 Validate Dataset Coverage
# --------------------------------------------------------------------------------------------------

architecture_dataset_ids = set(
    ARCHITECTURE_SUMMARY_DF["dataset"].astype(str)
)

expected_dataset_ids = set(
    str(dataset_id)
    for dataset_id in DATASET_IDS
)

if architecture_dataset_ids != expected_dataset_ids:

    raise RuntimeError(
        "Notebook 08 architecture summary dataset coverage "
        "does not match DATASET_IDS."
    )


if (
    ARCHITECTURE_SUMMARY_DF["dataset"]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate dataset entries detected in Notebook 08 "
        "architecture summary."
    )


# --------------------------------------------------------------------------------------------------
# 15.3 Test Guidance Computation for Each Dataset
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    dataset_id = str(dataset_id)

    rows = ARCHITECTURE_SUMMARY_DF[
        ARCHITECTURE_SUMMARY_DF["dataset"].astype(str)
        == dataset_id
    ]

    if len(rows) != 1:

        raise RuntimeError(
            f"Expected exactly one architecture row for "
            f"{dataset_id}; found {len(rows)}."
        )

    architecture = rows.iloc[0]

    generative_dimension = int(
        architecture["generative_dimension"]
    )

    transformed_dimension = int(
        architecture["transformed_dimension"]
    )

    numerical_features = int(
        architecture["numerical_features"]
    )

    categorical_features = int(
        architecture["categorical_features"]
    )

    if generative_dimension <= 0:

        raise ValueError(
            f"{dataset_id}: invalid generative dimension."
        )

    if transformed_dimension <= 0:

        raise ValueError(
            f"{dataset_id}: invalid transformed dimension."
        )

    if numerical_features < 0:

        raise ValueError(
            f"{dataset_id}: invalid numerical feature count."
        )

    if categorical_features < 0:

        raise ValueError(
            f"{dataset_id}: invalid categorical feature count."
        )

    # ----------------------------------------------------------------------------------------------
    # Deterministic test tensors
    #
    # The transformed dimension is used because statistical guidance
    # operates on differentiable tensor representations.
    # ----------------------------------------------------------------------------------------------

    torch.manual_seed(
        MASTER_SEED
    )

    real = torch.randn(
        TEST_BATCH_SIZE,
        transformed_dimension,
        device=DEVICE,
    )

    synthetic = torch.randn(
        TEST_BATCH_SIZE,
        transformed_dimension,
        device=DEVICE,
    )

    synthetic.requires_grad_(True)

    # ----------------------------------------------------------------------------------------------
    # Numerical representation
    # ----------------------------------------------------------------------------------------------

    if numerical_features > 0:

        numeric_real = real[
            :,
            :numerical_features,
        ]

        numeric_synthetic = synthetic[
            :,
            :numerical_features,
        ]

    else:

        numeric_real = real[
            :,
            :0,
        ]

        numeric_synthetic = synthetic[
            :,
            :0,
        ]

    # ----------------------------------------------------------------------------------------------
    # Statistical guidance computation
    #
    # Categorical probability inputs are intentionally omitted.
    # This section verifies the core objective and its
    # zero-categorical execution path.
    # ----------------------------------------------------------------------------------------------

    total_loss, components = statistical_guidance_loss(
        real_numeric=numeric_real,
        synthetic_numeric=numeric_synthetic,
        real_distribution_features=real,
        synthetic_distribution_features=synthetic,
        real_categorical_probabilities=None,
        synthetic_categorical_probabilities=None,
        return_components=True,
    )

    # ----------------------------------------------------------------------------------------------
    # Validate component structure
    # ----------------------------------------------------------------------------------------------

    expected_components = {
        "L_marg",
        "L_mom",
        "L_dep",
        "L_cat",
    }

    if set(components.keys()) != expected_components:

        raise RuntimeError(
            f"{dataset_id}: unexpected statistical guidance "
            f"components: {set(components.keys())}"
        )

    # ----------------------------------------------------------------------------------------------
    # Validate individual components
    # ----------------------------------------------------------------------------------------------

    component_finite = True

    for component_name, component_value in components.items():

        if not isinstance(
            component_value,
            torch.Tensor
        ):

            raise TypeError(
                f"{dataset_id}: {component_name} is not a tensor."
            )

        if component_value.ndim != 0:

            raise ValueError(
                f"{dataset_id}: {component_name} is not scalar."
            )

        if not torch.isfinite(
            component_value
        ).item():

            component_finite = False

            raise FloatingPointError(
                f"{dataset_id}: {component_name} is non-finite."
            )

    # ----------------------------------------------------------------------------------------------
    # Validate total statistical loss
    # ----------------------------------------------------------------------------------------------

    if total_loss.ndim != 0:

        raise ValueError(
            f"{dataset_id}: L_stat is not scalar."
        )

    finite_total = bool(
        torch.isfinite(
            total_loss
        ).item()
    )

    if not finite_total:

        raise FloatingPointError(
            f"{dataset_id}: L_stat is non-finite."
        )

    if not total_loss.requires_grad:

        raise RuntimeError(
            f"{dataset_id}: L_stat is not differentiable."
        )

    # ----------------------------------------------------------------------------------------------
    # Gradient propagation test
    # ----------------------------------------------------------------------------------------------

    total_loss.backward()

    if synthetic.grad is None:

        raise RuntimeError(
            f"{dataset_id}: synthetic gradient is missing."
        )

    gradient_finite = bool(
        torch.isfinite(
            synthetic.grad
        ).all().item()
    )

    if not gradient_finite:

        raise FloatingPointError(
            f"{dataset_id}: synthetic gradient contains "
            "non-finite values."
        )

    gradient_norm = float(
        synthetic.grad.abs().sum().item()
    )

    if gradient_norm <= 1e-12:

        raise RuntimeError(
            f"{dataset_id}: synthetic guidance gradient "
            "is effectively zero."
        )

    # ----------------------------------------------------------------------------------------------
    # Record result
    # ----------------------------------------------------------------------------------------------

    GUIDANCE_COMPUTATION_RESULTS.append(
        {
            "dataset_id": dataset_id,
            "batch_size": TEST_BATCH_SIZE,
            "generative_dimension": generative_dimension,
            "transformed_dimension": transformed_dimension,
            "numerical_features": numerical_features,
            "categorical_features": categorical_features,
            "L_marg": safe_float(
                components["L_marg"]
            ),
            "L_mom": safe_float(
                components["L_mom"]
            ),
            "L_dep": safe_float(
                components["L_dep"]
            ),
            "L_cat": safe_float(
                components["L_cat"]
            ),
            "L_stat": safe_float(
                total_loss
            ),
            "finite_components": component_finite,
            "finite_total": finite_total,
            "gradient_finite": gradient_finite,
            "gradient_norm": gradient_norm,
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id}"
    )

    print(
        f"  Generative dimension : {generative_dimension}"
    )

    print(
        f"  Transformed dimension: {transformed_dimension}"
    )

    print(
        f"  Numerical features  : {numerical_features}"
    )

    print(
        f"  Categorical features: {categorical_features}"
    )

    print(
        f"  L_stat              : "
        f"{safe_float(total_loss):.8f}"
    )

    print(
        f"  Gradient norm       : "
        f"{gradient_norm:.8e}"
    )


# --------------------------------------------------------------------------------------------------
# 15.4 Create Results DataFrame
# --------------------------------------------------------------------------------------------------

GUIDANCE_COMPUTATION_DF = pd.DataFrame(
    GUIDANCE_COMPUTATION_RESULTS
)


# --------------------------------------------------------------------------------------------------
# 15.5 Validate Results
# --------------------------------------------------------------------------------------------------

if len(GUIDANCE_COMPUTATION_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "Guidance computation did not produce exactly one "
        "result per dataset."
    )


result_dataset_ids = set(
    GUIDANCE_COMPUTATION_DF["dataset_id"].astype(str)
)

if result_dataset_ids != expected_dataset_ids:

    raise RuntimeError(
        "Guidance computation result dataset coverage "
        "does not match DATASET_IDS."
    )


if (
    GUIDANCE_COMPUTATION_DF["dataset_id"]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate guidance computation results detected."
    )


if not (
    GUIDANCE_COMPUTATION_DF["status"]
    .eq("PASS")
    .all()
):

    raise RuntimeError(
        "Statistical guidance computation test failed."
    )


# --------------------------------------------------------------------------------------------------
# 15.6 Display Results
# --------------------------------------------------------------------------------------------------

display(
    GUIDANCE_COMPUTATION_DF
)


# --------------------------------------------------------------------------------------------------
# 15.7 Final Output
# --------------------------------------------------------------------------------------------------

print()
print("✓ Guidance computation tests passed.")

print(
    f"  Datasets tested       : "
    f"{len(GUIDANCE_COMPUTATION_DF)}"
)

print(
    f"  Successful            : "
    f"{GUIDANCE_COMPUTATION_DF['status'].eq('PASS').sum()}"
)

print("  Component finiteness  : PASS")
print("  Total-loss finiteness : PASS")
print("  Gradient propagation  : PASS")
print("  Dataset coverage      : PASS")


15. TEST GUIDANCE COMPUTATION
✓ adult_income
  Generative dimension : 15
  Transformed dimension: 105
  Numerical features  : 6
  Categorical features: 8
  L_stat              : 0.07419859
  Gradient norm       : 1.94971487e-01
✓ bank_marketing
  Generative dimension : 17
  Transformed dimension: 51
  Numerical features  : 7
  Categorical features: 9
  L_stat              : 0.07908409
  Gradient norm       : 2.03683749e-01
✓ diabetes_130us
  Generative dimension : 48
  Transformed dimension: 2329
  Numerical features  : 11
  Categorical features: 36
  L_stat              : 0.08205149
  Gradient norm       : 1.71841308e-01


,dataset_id,batch_size,generative_dimension,transformed_dimension,numerical_features,categorical_features,L_marg,L_mom,L_dep,L_cat,L_stat,finite_components,finite_total,gradient_finite,gradient_norm,status
0,adult_income,64,15,105,6,8,0.018785,0.100839,0.177170,0.0,0.074199,True,True,True,0.194971,PASS
1,bank_marketing,64,17,51,7,9,0.020923,0.117413,0.178000,0.0,0.079084,True,True,True,0.203684,PASS
2,diabetes_130us,64,48,2329,11,36,0.019149,0.130951,0.178106,0.0,0.082051,True,True,True,0.171841,PASS



✓ Guidance computation tests passed.
  Datasets tested       : 3
  Successful            : 3
  Component finiteness  : PASS
  Total-loss finiteness : PASS
  Gradient propagation  : PASS
  Dataset coverage      : PASS


In [50]:
# ==================================================================================================
# 16. TEST GRADIENT FLOW
# ==================================================================================================

print("\n" + "=" * 100)
print("16. TEST GRADIENT FLOW")
print("=" * 100)

GRADIENT_RESULTS = []


# --------------------------------------------------------------------------------------------------
# 16.1 Validate Architecture Summary
# --------------------------------------------------------------------------------------------------

REQUIRED_ARCHITECTURE_COLUMNS = [
    "dataset",
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
]

missing_columns = [
    column
    for column in REQUIRED_ARCHITECTURE_COLUMNS
    if column not in ARCHITECTURE_SUMMARY_DF.columns
]

if missing_columns:

    raise RuntimeError(
        "Architecture summary is missing required columns: "
        + ", ".join(missing_columns)
    )


# --------------------------------------------------------------------------------------------------
# 16.2 Test Gradient Flow for Each Dataset
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    dataset_id = str(dataset_id)

    rows = ARCHITECTURE_SUMMARY_DF[
        ARCHITECTURE_SUMMARY_DF["dataset"].astype(str)
        == dataset_id
    ]

    if len(rows) != 1:

        raise RuntimeError(
            f"Expected exactly one architecture row for "
            f"{dataset_id}; found {len(rows)}."
        )

    architecture = rows.iloc[0]

    generative_dimension = int(
        architecture["generative_dimension"]
    )

    transformed_dimension = int(
        architecture["transformed_dimension"]
    )

    numerical_features = int(
        architecture["numerical_features"]
    )

    categorical_features = int(
        architecture["categorical_features"]
    )

    if transformed_dimension <= 0:

        raise ValueError(
            f"{dataset_id}: transformed dimension must be positive."
        )

    if numerical_features < 0:

        raise ValueError(
            f"{dataset_id}: invalid numerical feature count."
        )


    # ----------------------------------------------------------------------------------------------
    # Deterministic test tensors
    # ----------------------------------------------------------------------------------------------

    torch.manual_seed(
        MASTER_SEED + 1
    )

    real = torch.randn(
        TEST_BATCH_SIZE,
        transformed_dimension,
        device=DEVICE,
    )

    synthetic = torch.randn(
        TEST_BATCH_SIZE,
        transformed_dimension,
        device=DEVICE,
        requires_grad=True,
    )


    # ----------------------------------------------------------------------------------------------
    # Numerical representation
    # ----------------------------------------------------------------------------------------------

    numeric_real = real[
        :,
        :numerical_features,
    ]

    numeric_synthetic = synthetic[
        :,
        :numerical_features,
    ]


    # ----------------------------------------------------------------------------------------------
    # Statistical guidance computation
    # ----------------------------------------------------------------------------------------------

    statistical_loss = statistical_guidance_loss(
        real_numeric=numeric_real,
        synthetic_numeric=numeric_synthetic,
        real_distribution_features=real,
        synthetic_distribution_features=synthetic,
        real_categorical_probabilities=None,
        synthetic_categorical_probabilities=None,
        return_components=False,
    )


    # ----------------------------------------------------------------------------------------------
    # Validate scalar loss
    # ----------------------------------------------------------------------------------------------

    if not isinstance(
        statistical_loss,
        torch.Tensor
    ):

        raise TypeError(
            f"{dataset_id}: statistical guidance loss "
            "must return a tensor."
        )

    if statistical_loss.ndim != 0:

        raise ValueError(
            f"{dataset_id}: statistical guidance loss "
            "must be scalar."
        )

    if not torch.isfinite(
        statistical_loss
    ).item():

        raise FloatingPointError(
            f"{dataset_id}: statistical guidance loss "
            "is non-finite."
        )

    if not statistical_loss.requires_grad:

        raise RuntimeError(
            f"{dataset_id}: statistical guidance loss "
            "does not require gradients."
        )


    # ----------------------------------------------------------------------------------------------
    # Backward pass
    # ----------------------------------------------------------------------------------------------

    if synthetic.grad is not None:

        synthetic.grad.zero_()

    statistical_loss.backward()


    # ----------------------------------------------------------------------------------------------
    # Gradient validation
    # ----------------------------------------------------------------------------------------------

    gradient = synthetic.grad

    gradient_exists = (
        gradient is not None
    )

    gradient_finite = (
        gradient_exists
        and
        bool(
            torch.isfinite(
                gradient
            ).all().item()
        )
    )

    gradient_norm = (
        float(
            torch.linalg.vector_norm(
                gradient
            ).detach().cpu()
        )
        if gradient_exists
        else float("nan")
    )

    gradient_nonzero = (
        gradient_exists
        and
        gradient_norm > 1e-12
    )

    passed = (
        gradient_exists
        and
        gradient_finite
        and
        gradient_nonzero
    )


    # ----------------------------------------------------------------------------------------------
    # Store result
    # ----------------------------------------------------------------------------------------------

    GRADIENT_RESULTS.append(
        {
            "dataset_id": dataset_id,
            "batch_size": TEST_BATCH_SIZE,
            "generative_dimension": generative_dimension,
            "transformed_dimension": transformed_dimension,
            "numerical_features": numerical_features,
            "categorical_features": categorical_features,
            "statistical_loss": safe_float(
                statistical_loss
            ),
            "gradient_exists": gradient_exists,
            "gradient_finite": gradient_finite,
            "gradient_nonzero": gradient_nonzero,
            "gradient_norm": gradient_norm,
            "status": (
                "PASS"
                if passed
                else "FAIL"
            ),
        }
    )

    print(
        f"✓ {dataset_id}"
    )

    print(
        f"  Transformed dimension : "
        f"{transformed_dimension}"
    )

    print(
        f"  Numerical features   : "
        f"{numerical_features}"
    )

    print(
        f"  Statistical loss     : "
        f"{safe_float(statistical_loss):.8f}"
    )

    print(
        f"  Gradient norm        : "
        f"{gradient_norm:.8e}"
    )


# --------------------------------------------------------------------------------------------------
# 16.3 Results DataFrame
# --------------------------------------------------------------------------------------------------

GRADIENT_TEST_DF = pd.DataFrame(
    GRADIENT_RESULTS
)


# --------------------------------------------------------------------------------------------------
# 16.4 Validate Dataset Coverage
# --------------------------------------------------------------------------------------------------

expected_dataset_ids = set(
    str(dataset_id)
    for dataset_id in DATASET_IDS
)

result_dataset_ids = set(
    GRADIENT_TEST_DF["dataset_id"].astype(str)
)

if result_dataset_ids != expected_dataset_ids:

    raise RuntimeError(
        "Gradient-flow results do not cover exactly "
        "the registered datasets."
    )


if (
    GRADIENT_TEST_DF["dataset_id"]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate gradient-flow results detected."
    )


# --------------------------------------------------------------------------------------------------
# 16.5 Final Validation
# --------------------------------------------------------------------------------------------------

if not (
    GRADIENT_TEST_DF["gradient_exists"]
    .all()
):

    raise RuntimeError(
        "Gradient tensors were not produced for all datasets."
    )


if not (
    GRADIENT_TEST_DF["gradient_finite"]
    .all()
):

    raise RuntimeError(
        "Non-finite gradients detected."
    )


if not (
    GRADIENT_TEST_DF["gradient_nonzero"]
    .all()
):

    raise RuntimeError(
        "Zero gradients detected."
    )


if not (
    GRADIENT_TEST_DF["status"]
    .eq("PASS")
    .all()
):

    raise RuntimeError(
        "Statistical guidance gradient-flow test failed."
    )


# --------------------------------------------------------------------------------------------------
# 16.6 Display Results
# --------------------------------------------------------------------------------------------------

display(
    GRADIENT_TEST_DF
)


# --------------------------------------------------------------------------------------------------
# 16.7 Final Output
# --------------------------------------------------------------------------------------------------

print()
print("✓ Gradient-flow tests passed.")

print(
    f"  Datasets tested      : "
    f"{len(GRADIENT_TEST_DF)}"
)

print(
    f"  Successful           : "
    f"{GRADIENT_TEST_DF['status'].eq('PASS').sum()}"
)

print("  Gradient existence   : PASS")
print("  Gradient finiteness  : PASS")
print("  Non-zero gradients   : PASS")
print("  Dataset coverage     : PASS")


16. TEST GRADIENT FLOW
✓ adult_income
  Transformed dimension : 105
  Numerical features   : 6
  Statistical loss     : 0.08149929
  Gradient norm        : 9.09693260e-03
✓ bank_marketing
  Transformed dimension : 51
  Numerical features   : 7
  Statistical loss     : 0.07557134
  Gradient norm        : 8.58618878e-03
✓ diabetes_130us
  Transformed dimension : 2329
  Numerical features   : 11
  Statistical loss     : 0.06827363
  Gradient norm        : 6.66473201e-03


,dataset_id,batch_size,generative_dimension,transformed_dimension,numerical_features,categorical_features,statistical_loss,gradient_exists,gradient_finite,gradient_nonzero,gradient_norm,status
0,adult_income,64,15,105,6,8,0.081499,True,True,True,0.009097,PASS
1,bank_marketing,64,17,51,7,9,0.075571,True,True,True,0.008586,PASS
2,diabetes_130us,64,48,2329,11,36,0.068274,True,True,True,0.006665,PASS



✓ Gradient-flow tests passed.
  Datasets tested      : 3
  Successful           : 3
  Gradient existence   : PASS
  Gradient finiteness  : PASS
  Non-zero gradients   : PASS
  Dataset coverage     : PASS


In [52]:
# ==================================================================================================
# 17. VALIDATE NUMERICAL STABILITY
# ==================================================================================================

print("\n" + "=" * 100)
print("17. VALIDATE NUMERICAL STABILITY")
print("=" * 100)

NUMERICAL_STABILITY_RESULTS = []

STABILITY_TESTS = {
    "normal": 1.0,
    "large": 1e3,
    "small": 1e-6,
    "mixed": 1e2,
}


# --------------------------------------------------------------------------------------------------
# 17.1 Validate Architecture Summary
# --------------------------------------------------------------------------------------------------

REQUIRED_ARCHITECTURE_COLUMNS = [
    "dataset",
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
]

missing_columns = [
    column
    for column in REQUIRED_ARCHITECTURE_COLUMNS
    if column not in ARCHITECTURE_SUMMARY_DF.columns
]

if missing_columns:

    raise RuntimeError(
        "Architecture summary is missing required columns: "
        + ", ".join(missing_columns)
    )


# --------------------------------------------------------------------------------------------------
# 17.2 Validate Dataset Coverage
# --------------------------------------------------------------------------------------------------

expected_dataset_ids = set(
    str(dataset_id)
    for dataset_id in DATASET_IDS
)

architecture_dataset_ids = set(
    ARCHITECTURE_SUMMARY_DF["dataset"].astype(str)
)

if architecture_dataset_ids != expected_dataset_ids:

    raise RuntimeError(
        "Architecture summary dataset coverage does not "
        "match DATASET_IDS."
    )


# --------------------------------------------------------------------------------------------------
# 17.3 Numerical Stability Tests
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    dataset_id = str(dataset_id)

    rows = ARCHITECTURE_SUMMARY_DF[
        ARCHITECTURE_SUMMARY_DF["dataset"].astype(str)
        == dataset_id
    ]

    if len(rows) != 1:

        raise RuntimeError(
            f"Expected exactly one architecture row for "
            f"{dataset_id}; found {len(rows)}."
        )

    architecture = rows.iloc[0]

    transformed_dimension = int(
        architecture["transformed_dimension"]
    )

    numerical_features = int(
        architecture["numerical_features"]
    )

    categorical_features = int(
        architecture["categorical_features"]
    )

    generative_dimension = int(
        architecture["generative_dimension"]
    )

    if transformed_dimension <= 0:

        raise ValueError(
            f"{dataset_id}: transformed dimension must be positive."
        )

    if numerical_features < 0:

        raise ValueError(
            f"{dataset_id}: numerical feature count cannot be negative."
        )


    for test_name, scale in STABILITY_TESTS.items():

        # ------------------------------------------------------------------------------------------
        # Deterministic test tensors
        # ------------------------------------------------------------------------------------------

        torch.manual_seed(
            MASTER_SEED + 2
        )

        real = (
            torch.randn(
                TEST_BATCH_SIZE,
                transformed_dimension,
                device=DEVICE,
            )
            * scale
        )

        synthetic = (
            torch.randn(
                TEST_BATCH_SIZE,
                transformed_dimension,
                device=DEVICE,
            )
            * scale
        )

        synthetic.requires_grad_(True)


        try:

            # --------------------------------------------------------------------------------------
            # Numerical feature representation
            # --------------------------------------------------------------------------------------

            numeric_real = real[
                :,
                :numerical_features,
            ]

            numeric_synthetic = synthetic[
                :,
                :numerical_features,
            ]


            # --------------------------------------------------------------------------------------
            # Statistical guidance computation
            # --------------------------------------------------------------------------------------

            loss = statistical_guidance_loss(
                real_numeric=numeric_real,
                synthetic_numeric=numeric_synthetic,
                real_distribution_features=real,
                synthetic_distribution_features=synthetic,
                real_categorical_probabilities=None,
                synthetic_categorical_probabilities=None,
                return_components=False,
            )


            # --------------------------------------------------------------------------------------
            # Loss validation
            # --------------------------------------------------------------------------------------

            loss_is_tensor = isinstance(
                loss,
                torch.Tensor
            )

            loss_scalar = (
                loss_is_tensor
                and loss.ndim == 0
            )

            loss_finite = (
                loss_scalar
                and
                bool(
                    torch.isfinite(
                        loss
                    ).item()
                )
            )

            loss_requires_grad = (
                loss_scalar
                and
                loss.requires_grad
            )

            if not loss_finite:

                raise FloatingPointError(
                    "Statistical guidance loss is non-finite."
                )

            if not loss_requires_grad:

                raise RuntimeError(
                    "Statistical guidance loss does not require gradients."
                )


            # --------------------------------------------------------------------------------------
            # Backward pass
            # --------------------------------------------------------------------------------------

            loss.backward()


            # --------------------------------------------------------------------------------------
            # Gradient validation
            # --------------------------------------------------------------------------------------

            gradient_exists = (
                synthetic.grad is not None
            )

            gradient_finite = (
                gradient_exists
                and
                bool(
                    torch.isfinite(
                        synthetic.grad
                    ).all().item()
                )
            )

            gradient_norm = (
                float(
                    torch.linalg.vector_norm(
                        synthetic.grad
                    ).detach().cpu()
                )
                if gradient_exists
                else float("nan")
            )

            gradient_nonzero = (
                gradient_exists
                and
                gradient_norm > 1e-12
            )

            passed = (
                loss_finite
                and
                loss_requires_grad
                and
                gradient_finite
                and
                gradient_nonzero
            )


            # --------------------------------------------------------------------------------------
            # Store successful result
            # --------------------------------------------------------------------------------------

            NUMERICAL_STABILITY_RESULTS.append(
                {
                    "dataset_id": dataset_id,
                    "test_case": test_name,
                    "scale": scale,
                    "generative_dimension": generative_dimension,
                    "transformed_dimension": transformed_dimension,
                    "numerical_features": numerical_features,
                    "categorical_features": categorical_features,
                    "loss": safe_float(loss),
                    "loss_finite": loss_finite,
                    "gradient_exists": gradient_exists,
                    "gradient_finite": gradient_finite,
                    "gradient_nonzero": gradient_nonzero,
                    "gradient_norm": gradient_norm,
                    "status": (
                        "PASS"
                        if passed
                        else "FAIL"
                    ),
                }
            )


        except Exception as exc:

            NUMERICAL_STABILITY_RESULTS.append(
                {
                    "dataset_id": dataset_id,
                    "test_case": test_name,
                    "scale": scale,
                    "generative_dimension": generative_dimension,
                    "transformed_dimension": transformed_dimension,
                    "numerical_features": numerical_features,
                    "categorical_features": categorical_features,
                    "loss": np.nan,
                    "loss_finite": False,
                    "gradient_exists": False,
                    "gradient_finite": False,
                    "gradient_nonzero": False,
                    "gradient_norm": np.nan,
                    "status": "FAIL",
                    "error": str(exc),
                }
            )


# --------------------------------------------------------------------------------------------------
# 17.4 Results DataFrame
# --------------------------------------------------------------------------------------------------

NUMERICAL_STABILITY_DF = pd.DataFrame(
    NUMERICAL_STABILITY_RESULTS
)


# --------------------------------------------------------------------------------------------------
# 17.5 Validate Expected Number of Tests
# --------------------------------------------------------------------------------------------------

expected_test_count = (
    len(DATASET_IDS)
    * len(STABILITY_TESTS)
)

if len(NUMERICAL_STABILITY_DF) != expected_test_count:

    raise RuntimeError(
        "Unexpected number of numerical stability test results. "
        f"Expected {expected_test_count}, "
        f"received {len(NUMERICAL_STABILITY_DF)}."
    )


# --------------------------------------------------------------------------------------------------
# 17.6 Validate Dataset and Test-Case Coverage
# --------------------------------------------------------------------------------------------------

if set(
    NUMERICAL_STABILITY_DF["dataset_id"].astype(str)
) != expected_dataset_ids:

    raise RuntimeError(
        "Numerical stability results do not cover "
        "exactly the registered datasets."
    )


if set(
    NUMERICAL_STABILITY_DF["test_case"]
) != set(
    STABILITY_TESTS.keys()
):

    raise RuntimeError(
        "Numerical stability results do not contain "
        "all required test cases."
    )


# --------------------------------------------------------------------------------------------------
# 17.7 Validate Final Status
# --------------------------------------------------------------------------------------------------

if not (
    NUMERICAL_STABILITY_DF["status"]
    .eq("PASS")
    .all()
):

    display(
        NUMERICAL_STABILITY_DF
    )

    raise RuntimeError(
        "Numerical stability validation failed."
    )


# --------------------------------------------------------------------------------------------------
# 17.8 Display Results
# --------------------------------------------------------------------------------------------------

display(
    NUMERICAL_STABILITY_DF
)


# --------------------------------------------------------------------------------------------------
# 17.9 Final Output
# --------------------------------------------------------------------------------------------------

print()
print("✓ Numerical stability validation passed.")

print(
    f"  Datasets tested : "
    f"{NUMERICAL_STABILITY_DF['dataset_id'].nunique()}"
)

print(
    f"  Test cases      : "
    f"{NUMERICAL_STABILITY_DF['test_case'].nunique()}"
)

print(
    f"  Total tests     : "
    f"{len(NUMERICAL_STABILITY_DF)}"
)

print("  Loss finiteness : PASS")
print("  Gradient finite : PASS")
print("  Non-zero gradient: PASS")
print("  Dataset coverage: PASS")


17. VALIDATE NUMERICAL STABILITY


,dataset_id,test_case,scale,generative_dimension,transformed_dimension,numerical_features,categorical_features,loss,loss_finite,gradient_exists,gradient_finite,gradient_nonzero,gradient_norm,status
0,adult_income,normal,1.000000,15,105,6,8,0.071867,True,True,True,True,0.009077,PASS
1,adult_income,large,1000.000000,15,105,6,8,23.669487,True,True,True,True,0.009021,PASS
2,adult_income,small,0.000001,15,105,6,8,0.000057,True,True,True,True,9.355218,PASS
3,adult_income,mixed,100.000000,15,105,6,8,2.410370,True,True,True,True,0.009021,PASS
4,bank_marketing,normal,1.000000,17,51,7,9,0.072440,True,True,True,True,0.008609,PASS
5,bank_marketing,large,1000.000000,17,51,7,9,24.089931,True,True,True,True,0.008352,PASS
6,bank_marketing,small,0.000001,17,51,7,9,0.000058,True,True,True,True,9.472369,PASS
7,bank_marketing,mixed,100.000000,17,51,7,9,2.452551,True,True,True,True,0.008353,PASS
8,diabetes_130us,normal,1.000000,48,2329,11,36,0.079772,True,True,True,True,0.006666,PASS
9,diabetes_130us,large,1000.000000,48,2329,11,36,30.429407,True,True,True,True,0.006663,PASS



✓ Numerical stability validation passed.
  Datasets tested : 3
  Test cases      : 4
  Total tests     : 12
  Loss finiteness : PASS
  Gradient finite : PASS
  Non-zero gradient: PASS
  Dataset coverage: PASS


In [55]:
# ==================================================================================================
# 18. SAVE GUIDANCE MODULE
# ==================================================================================================

print("\n" + "=" * 100)
print("18. SAVE GUIDANCE MODULE")
print("=" * 100)

GUIDANCE_MODULE_PATH = (
    DIRS["models"]
    / "sppgan_statistical_guidance.py"
)


# --------------------------------------------------------------------------------------------------
# 18.1 Validated Statistical Guidance Module
# --------------------------------------------------------------------------------------------------

GUIDANCE_MODULE_CODE = r'''
"""
SPP-GAN Statistical Guidance Module

Validated in Notebook 09, Sections 6-17.

Statistical objective:
    L_stat = λ_m L_marg + λ_mom L_mom + λ_d L_dep + λ_c L_cat

Dependency representation:
    Differentiable Pearson correlation matrix.

Moment representation:
    Mean and standard deviation.

Categorical representation:
    Marginal category probability discrepancy.

Privacy:
    No differential-privacy mechanism is implemented in this module.
    Privacy mechanisms and accounting are handled by downstream notebooks.
"""

import torch

EPSILON = 1e-8


# --------------------------------------------------------------------------------------------------
# Pairwise squared Euclidean distance
# --------------------------------------------------------------------------------------------------

def pairwise_squared_distance(
    x,
    y,
):

    if x.ndim != 2 or y.ndim != 2:
        raise ValueError(
            "Inputs must be two-dimensional tensors."
        )

    if x.shape[1] != y.shape[1]:
        raise ValueError(
            "Input feature dimensions must match."
        )

    if not torch.isfinite(x).all().item():
        raise ValueError(
            "x contains non-finite values."
        )

    if not torch.isfinite(y).all().item():
        raise ValueError(
            "y contains non-finite values."
        )

    x_norm = x.pow(2).sum(
        dim=1,
        keepdim=True,
    )

    y_norm = y.pow(2).sum(
        dim=1,
        keepdim=True,
    ).transpose(
        0,
        1,
    )

    distance = (
        x_norm
        + y_norm
        - 2.0 * torch.matmul(
            x,
            y.transpose(
                0,
                1,
            ),
        )
    )

    return torch.clamp(
        distance,
        min=0.0,
    )


# --------------------------------------------------------------------------------------------------
# RBF kernel
# --------------------------------------------------------------------------------------------------

def rbf_kernel(
    x,
    y,
    sigma=1.0,
):

    sigma = float(sigma)

    if not torch.isfinite(
        torch.tensor(sigma)
    ).item():

        raise ValueError(
            "sigma must be finite."
        )

    if sigma <= 0.0:

        raise ValueError(
            "sigma must be positive."
        )

    distance = pairwise_squared_distance(
        x,
        y,
    )

    denominator = (
        2.0
        * max(
            sigma ** 2,
            EPSILON,
        )
    )

    return torch.exp(
        -distance / denominator
    )


# --------------------------------------------------------------------------------------------------
# Differentiable RBF-MMD²
# --------------------------------------------------------------------------------------------------

def differentiable_mmd(
    real,
    synthetic,
    sigma=1.0,
):

    if real.ndim != 2 or synthetic.ndim != 2:
        raise ValueError(
            "MMD inputs must be two-dimensional tensors."
        )

    if real.shape[1] != synthetic.shape[1]:
        raise ValueError(
            "MMD feature dimensions must match."
        )

    if real.shape[0] < 2 or synthetic.shape[0] < 2:
        raise ValueError(
            "MMD requires at least two observations per batch."
        )

    k_rr = rbf_kernel(
        real,
        real,
        sigma,
    )

    k_ss = rbf_kernel(
        synthetic,
        synthetic,
        sigma,
    )

    k_rs = rbf_kernel(
        real,
        synthetic,
        sigma,
    )

    return (
        k_rr.mean()
        + k_ss.mean()
        - 2.0 * k_rs.mean()
    )


# --------------------------------------------------------------------------------------------------
# Differentiable Pearson correlation matrix
# --------------------------------------------------------------------------------------------------

def differentiable_pearson_matrix(
    x,
):

    if x.ndim != 2:
        raise ValueError(
            "Input must be a two-dimensional tensor."
        )

    if x.shape[0] < 2:
        raise ValueError(
            "Pearson correlation requires at least two observations."
        )

    if not torch.isfinite(x).all().item():
        raise ValueError(
            "Input contains non-finite values."
        )

    if x.shape[1] == 0:

        return x.new_zeros(
            (0, 0)
        )

    centered = (
        x
        - x.mean(
            dim=0,
            keepdim=True,
        )
    )

    variance = torch.mean(
        centered.pow(2),
        dim=0,
    )

    std = torch.sqrt(
        variance
        + EPSILON
    )

    standardized = (
        centered
        / std.unsqueeze(0)
    )

    correlation = (
        standardized.transpose(0, 1)
        @ standardized
    ) / x.shape[0]

    return torch.clamp(
        correlation,
        min=-1.0,
        max=1.0,
    )


# --------------------------------------------------------------------------------------------------
# Numerical moment guidance
# --------------------------------------------------------------------------------------------------

def numerical_moment_guidance(
    real_numeric,
    synthetic_numeric,
):

    if real_numeric.ndim != 2:
        raise ValueError(
            "real_numeric must be two-dimensional."
        )

    if synthetic_numeric.ndim != 2:
        raise ValueError(
            "synthetic_numeric must be two-dimensional."
        )

    if real_numeric.shape != synthetic_numeric.shape:
        raise ValueError(
            "Real and synthetic numerical dimensions must match."
        )

    if not torch.isfinite(
        real_numeric
    ).all().item():

        raise ValueError(
            "real_numeric contains non-finite values."
        )

    if not torch.isfinite(
        synthetic_numeric
    ).all().item():

        raise ValueError(
            "synthetic_numeric contains non-finite values."
        )

    if real_numeric.shape[1] == 0:

        return synthetic_numeric.new_zeros(())

    real_mean = real_numeric.mean(
        dim=0
    )

    synthetic_mean = synthetic_numeric.mean(
        dim=0
    )

    real_std = real_numeric.std(
        dim=0,
        unbiased=False,
    )

    synthetic_std = synthetic_numeric.std(
        dim=0,
        unbiased=False,
    )

    mean_loss = torch.mean(
        torch.abs(
            synthetic_mean
            - real_mean
        )
    )

    std_loss = torch.mean(
        torch.abs(
            synthetic_std
            - real_std
        )
    )

    return (
        mean_loss
        + std_loss
    ) / 2.0


# --------------------------------------------------------------------------------------------------
# Dependency guidance
# --------------------------------------------------------------------------------------------------

def dependency_frobenius_loss(
    real_features,
    synthetic_features,
):

    if real_features.ndim != 2:
        raise ValueError(
            "real_features must be two-dimensional."
        )

    if synthetic_features.ndim != 2:
        raise ValueError(
            "synthetic_features must be two-dimensional."
        )

    if real_features.shape != synthetic_features.shape:
        raise ValueError(
            "Real and synthetic feature dimensions must match."
        )

    if real_features.shape[1] == 0:

        return synthetic_features.new_zeros(())

    real_corr = differentiable_pearson_matrix(
        real_features
    )

    synthetic_corr = differentiable_pearson_matrix(
        synthetic_features
    )

    difference = (
        synthetic_corr
        - real_corr
    )

    n_features = real_features.shape[1]

    return (
        torch.linalg.matrix_norm(
            difference,
            ord="fro",
        )
        / n_features
    )


# --------------------------------------------------------------------------------------------------
# Categorical probability guidance
# --------------------------------------------------------------------------------------------------

def categorical_probability_guidance(
    real_probabilities,
    synthetic_probabilities,
):

    if not isinstance(
        real_probabilities,
        (list, tuple),
    ):

        raise TypeError(
            "real_probabilities must be a list or tuple."
        )

    if not isinstance(
        synthetic_probabilities,
        (list, tuple),
    ):

        raise TypeError(
            "synthetic_probabilities must be a list or tuple."
        )

    if len(real_probabilities) != len(
        synthetic_probabilities
    ):

        raise ValueError(
            "Real and synthetic categorical probability "
            "lists must have equal length."
        )

    if len(real_probabilities) == 0:

        raise ValueError(
            "Categorical probability lists cannot both be empty "
            "when categorical guidance is requested."
        )

    losses = []

    for index, (
        real_prob,
        synthetic_prob,
    ) in enumerate(
        zip(
            real_probabilities,
            synthetic_probabilities,
        )
    ):

        real_prob = torch.as_tensor(
            real_prob,
            dtype=synthetic_prob.dtype,
            device=synthetic_prob.device,
        )

        if real_prob.ndim != 1:
            raise ValueError(
                f"Categorical feature {index}: "
                "real probability vector must be one-dimensional."
            )

        if synthetic_prob.ndim != 1:
            raise ValueError(
                f"Categorical feature {index}: "
                "synthetic probability vector must be one-dimensional."
            )

        if real_prob.shape != synthetic_prob.shape:
            raise ValueError(
                f"Categorical feature {index}: "
                "probability cardinalities do not match."
            )

        if not torch.isfinite(
            real_prob
        ).all().item():

            raise ValueError(
                f"Categorical feature {index}: "
                "real probabilities contain non-finite values."
            )

        if not torch.isfinite(
            synthetic_prob
        ).all().item():

            raise ValueError(
                f"Categorical feature {index}: "
                "synthetic probabilities contain non-finite values."
            )

        if (real_prob < 0).any().item():

            raise ValueError(
                f"Categorical feature {index}: "
                "real probabilities contain negative values."
            )

        if (synthetic_prob < 0).any().item():

            raise ValueError(
                f"Categorical feature {index}: "
                "synthetic probabilities contain negative values."
            )

        if not torch.isclose(
            real_prob.sum(),
            torch.ones(
                (),
                dtype=real_prob.dtype,
                device=real_prob.device,
            ),
            atol=1e-5,
            rtol=1e-5,
        ).item():

            raise ValueError(
                f"Categorical feature {index}: "
                "real probabilities do not form a simplex."
            )

        if not torch.isclose(
            synthetic_prob.sum(),
            torch.ones(
                (),
                dtype=synthetic_prob.dtype,
                device=synthetic_prob.device,
            ),
            atol=1e-5,
            rtol=1e-5,
        ).item():

            raise ValueError(
                f"Categorical feature {index}: "
                "synthetic probabilities do not form a simplex."
            )

        losses.append(
            torch.mean(
                torch.abs(
                    synthetic_prob
                    - real_prob
                )
            )
        )

    return torch.stack(
        losses
    ).mean()


# --------------------------------------------------------------------------------------------------
# Complete statistical guidance objective
# --------------------------------------------------------------------------------------------------

def statistical_guidance_loss(
    real_numeric,
    synthetic_numeric,
    real_distribution_features,
    synthetic_distribution_features,
    real_categorical_probabilities=None,
    synthetic_categorical_probabilities=None,
    lambda_m=0.25,
    lambda_mom=0.25,
    lambda_d=0.25,
    lambda_c=0.25,
    mmd_sigma=1.0,
    return_components=False,
):

    weights = [
        lambda_m,
        lambda_mom,
        lambda_d,
        lambda_c,
    ]

    for weight in weights:

        if not torch.isfinite(
            torch.tensor(float(weight))
        ).item():

            raise ValueError(
                "Statistical guidance weights must be finite."
            )

        if float(weight) < 0.0:

            raise ValueError(
                "Statistical guidance weights cannot be negative."
            )

    l_marg = differentiable_mmd(
        real_distribution_features,
        synthetic_distribution_features,
        sigma=mmd_sigma,
    )

    l_mom = numerical_moment_guidance(
        real_numeric,
        synthetic_numeric,
    )

    l_dep = dependency_frobenius_loss(
        real_distribution_features,
        synthetic_distribution_features,
    )

    categorical_requested = (
        real_categorical_probabilities is not None
        or
        synthetic_categorical_probabilities is not None
    )

    if categorical_requested:

        if (
            real_categorical_probabilities is None
            or
            synthetic_categorical_probabilities is None
        ):

            raise ValueError(
                "Real and synthetic categorical probabilities "
                "must both be provided."
            )

        l_cat = categorical_probability_guidance(
            real_categorical_probabilities,
            synthetic_categorical_probabilities,
        )

    else:

        l_cat = l_marg.new_zeros(())

    component_values = {
        "L_marg": l_marg,
        "L_mom": l_mom,
        "L_dep": l_dep,
        "L_cat": l_cat,
    }

    for name, value in component_values.items():

        if value.ndim != 0:

            raise ValueError(
                f"{name} must be scalar."
            )

        if not torch.isfinite(
            value
        ).item():

            raise FloatingPointError(
                f"{name} is non-finite."
            )

    total = (
        lambda_m * l_marg
        +
        lambda_mom * l_mom
        +
        lambda_d * l_dep
        +
        lambda_c * l_cat
    )

    if not torch.isfinite(
        total
    ).item():

        raise FloatingPointError(
            "Statistical guidance loss is non-finite."
        )

    if return_components:

        return total, component_values

    return total


# --------------------------------------------------------------------------------------------------
# Generator objective
# --------------------------------------------------------------------------------------------------

def sppgan_generator_objective(
    adversarial_loss,
    statistical_loss,
    lambda_stat=1.0,
):

    if adversarial_loss.ndim != 0:
        raise ValueError(
            "Adversarial loss must be scalar."
        )

    if statistical_loss.ndim != 0:
        raise ValueError(
            "Statistical loss must be scalar."
        )

    if not torch.isfinite(
        adversarial_loss
    ).item():

        raise FloatingPointError(
            "Adversarial loss is non-finite."
        )

    if not torch.isfinite(
        statistical_loss
    ).item():

        raise FloatingPointError(
            "Statistical loss is non-finite."
        )

    lambda_stat_tensor = torch.tensor(
        float(lambda_stat),
        dtype=adversarial_loss.dtype,
        device=adversarial_loss.device,
    )

    if not torch.isfinite(
        lambda_stat_tensor
    ).item():

        raise ValueError(
            "lambda_stat must be finite."
        )

    if float(lambda_stat) < 0.0:

        raise ValueError(
            "lambda_stat cannot be negative."
        )

    if (
        adversarial_loss.device
        != statistical_loss.device
        or
        adversarial_loss.dtype
        != statistical_loss.dtype
    ):

        raise ValueError(
            "Adversarial and statistical losses must have "
            "the same dtype and device."
        )

    return (
        adversarial_loss
        +
        lambda_stat * statistical_loss
    )
'''

# --------------------------------------------------------------------------------------------------
# 18.2 Persist Module
# --------------------------------------------------------------------------------------------------

GUIDANCE_MODULE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(
    GUIDANCE_MODULE_PATH,
    "w",
    encoding="utf-8",
) as f:

    f.write(
        GUIDANCE_MODULE_CODE
    )


# --------------------------------------------------------------------------------------------------
# 18.3 Syntax Validation
# --------------------------------------------------------------------------------------------------

import py_compile

py_compile.compile(
    str(GUIDANCE_MODULE_PATH),
    doraise=True,
)


# --------------------------------------------------------------------------------------------------
# 18.4 Validate Persisted Module Content
# --------------------------------------------------------------------------------------------------

with open(
    GUIDANCE_MODULE_PATH,
    "r",
    encoding="utf-8",
) as f:

    persisted_code = f.read()


REQUIRED_FUNCTIONS = [
    "pairwise_squared_distance",
    "rbf_kernel",
    "differentiable_mmd",
    "differentiable_pearson_matrix",
    "numerical_moment_guidance",
    "dependency_frobenius_loss",
    "categorical_probability_guidance",
    "statistical_guidance_loss",
    "sppgan_generator_objective",
]

for function_name in REQUIRED_FUNCTIONS:

    if (
        f"def {function_name}("
        not in persisted_code
    ):

        raise RuntimeError(
            f"Persisted guidance module is missing: "
            f"{function_name}"
        )


# --------------------------------------------------------------------------------------------------
# 18.5 Critical Contract Validation
# --------------------------------------------------------------------------------------------------

if "covariance_frobenius_loss" in persisted_code:

    raise RuntimeError(
        "Obsolete covariance-based dependency function detected."
    )

if "covariance_matrix" in persisted_code:

    raise RuntimeError(
        "Obsolete covariance matrix implementation detected."
    )

if "torch.nan_to_num" in persisted_code:

    raise RuntimeError(
        "Forbidden silent NaN/Inf replacement detected."
    )

if "synthetic_max" in persisted_code:

    raise RuntimeError(
        "Obsolete min/max moment guidance detected."
    )

if "synthetic_min" in persisted_code:

    raise RuntimeError(
        "Obsolete min/max moment guidance detected."
    )

if "return_components=False" not in persisted_code:

    raise RuntimeError(
        "Validated return_components interface is missing."
    )

if "differentiable_pearson_matrix" not in persisted_code:

    raise RuntimeError(
        "Differentiable Pearson dependency representation is missing."
    )


# --------------------------------------------------------------------------------------------------
# 18.6 Final Output
# --------------------------------------------------------------------------------------------------

print(
    f"✓ Guidance module saved:\n"
    f"  {GUIDANCE_MODULE_PATH}"
)

print()
print("✓ Python syntax validation       : PASS")
print("✓ Required functions             : PASS")
print("✓ Pearson dependency guidance   : PASS")
print("✓ Mean/std moment guidance      : PASS")
print("✓ Categorical simplex validation: PASS")
print("✓ Silent NaN/Inf replacement    : DISABLED")
print("✓ Obsolete covariance guidance  : ABSENT")
print("✓ Obsolete min/max guidance     : ABSENT")
print("✓ Privacy mechanism             : NOT INCLUDED")


18. SAVE GUIDANCE MODULE
✓ Guidance module saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09/models/sppgan_statistical_guidance.py

✓ Python syntax validation       : PASS
✓ Required functions             : PASS
✓ Pearson dependency guidance   : PASS
✓ Mean/std moment guidance      : PASS
✓ Categorical simplex validation: PASS
✓ Silent NaN/Inf replacement    : DISABLED
✓ Obsolete covariance guidance  : ABSENT
✓ Obsolete min/max guidance     : ABSENT
✓ Privacy mechanism             : NOT INCLUDED


In [58]:
# ==================================================================================================
# 19. SAVE GUIDANCE CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("19. SAVE GUIDANCE CONFIGURATION")
print("=" * 100)

from pathlib import Path
import json
from datetime import datetime, timezone

# --------------------------------------------------------------------------------------------------
# 1. Verify required configuration
# --------------------------------------------------------------------------------------------------

if "GUIDANCE_CONFIG" not in globals():
    raise RuntimeError(
        "GUIDANCE_CONFIG is not available. "
        "Run the guidance configuration sections before Section 19."
    )

if not isinstance(GUIDANCE_CONFIG, dict):
    raise TypeError("GUIDANCE_CONFIG must be a dictionary.")

# --------------------------------------------------------------------------------------------------
# 2. Resolve lambda_stat from the actual configuration structure
# --------------------------------------------------------------------------------------------------

lambda_stat = 1.0

if isinstance(GUIDANCE_CONFIG.get("lambda_stat"), (int, float)):
    lambda_stat = float(GUIDANCE_CONFIG["lambda_stat"])

if lambda_stat < 0:
    raise ValueError("lambda_stat must be non-negative.")

# --------------------------------------------------------------------------------------------------
# 3. Define publication-aligned statistical guidance configuration
# --------------------------------------------------------------------------------------------------

PERSISTED_GUIDANCE_CONFIG = {
    "configuration_version": "1.0",
    "notebook": "09",
    "notebook_name": "SPP-GAN Statistical Guidance",

    "objective": {
        "formula": "L_stat = lambda_m * L_marg + lambda_mom * L_mom + lambda_d * L_dep + lambda_c * L_cat",
        "generator_formula": "L_G = L_adv + lambda_stat * L_stat",
        "lambda_stat": lambda_stat
    },

    "weights": {
        "lambda_m": 0.25,
        "lambda_mom": 0.25,
        "lambda_d": 0.25,
        "lambda_c": 0.25,
        "weight_sum": 1.0
    },

    "distribution_guidance": {
        "component": "L_marg",
        "method": "differentiable_rbf_mmd_squared",
        "kernel": "RBF",
        "sigma": "configured_or_real_batch_median_nonzero_distance",
        "differentiable": True
    },

    "moment_guidance": {
        "component": "L_mom",
        "method": "mean_absolute_mean_and_standard_deviation_discrepancy",
        "components": [
            "mean",
            "std"
        ],
        "std_estimator": "population",
        "unbiased": False,
        "min_max_guidance": False,
        "differentiable": True
    },

    "categorical_guidance": {
        "component": "L_cat",
        "method": "marginal_category_probability_discrepancy",
        "distance": "mean_absolute_probability_discrepancy",
        "probability_requirements": {
            "non_negative": True,
            "finite": True,
            "simplex": True,
            "silent_normalization": False
        },
        "differentiable": True
    },

    "dependency_guidance": {
        "component": "L_dep",
        "representation": "differentiable_pearson_correlation_matrix",
        "method": "feature_normalized_frobenius_discrepancy",
        "normalization": "divide_by_number_of_features",
        "independent_correlation_loss": False,
        "differentiable": True
    },

    "correlation_representation": {
        "method": "Pearson_correlation_matrix",
        "role": "dependency_representation_used_by_L_dep",
        "independent_loss": False
    },

    "statistical_characterization": {
        "source": "Notebook 03",
        "source_role": "training_statistical_reference_and_characterization",
        "pearson": True,
        "spearman": True,
        "cramers_v": True,
        "training_dependency_representation": "Pearson",
        "spearman_training_loss": False,
        "cramers_v_training_loss": False
    },

    "privacy": {
        "enabled_in_notebook_09": False,
        "privacy_mechanism": "handled_in_downstream_DP_notebooks",
        "downstream_notebooks": [
            "10",
            "11"
        ],
        "end_to_end_privacy_claim_established": False
    },

    "execution_status": {
        "training_performed": False,
        "training_notebook": "12",
        "synthetic_generation_performed": False,
        "synthetic_generation_notebook": "13"
    },

    "provenance": {
        "statistical_source": "Notebook 03",
        "architecture_source": "Notebook 08",
        "guidance_implementation": "Notebook 09",
        "created_utc": datetime.now(timezone.utc).isoformat()
    }
}

# --------------------------------------------------------------------------------------------------
# 4. Validate weights
# --------------------------------------------------------------------------------------------------

weights = PERSISTED_GUIDANCE_CONFIG["weights"]

if abs(
    weights["lambda_m"]
    + weights["lambda_mom"]
    + weights["lambda_d"]
    + weights["lambda_c"]
    - 1.0
) > 1e-12:
    raise ValueError("Statistical guidance weights must sum to 1.0.")

# --------------------------------------------------------------------------------------------------
# 5. Validate objective configuration
# --------------------------------------------------------------------------------------------------

objective = PERSISTED_GUIDANCE_CONFIG["objective"]

if objective["lambda_stat"] < 0:
    raise ValueError("Generator lambda_stat must be non-negative.")

if objective["lambda_stat"] != lambda_stat:
    raise ValueError("Persisted lambda_stat does not match resolved lambda_stat.")

# --------------------------------------------------------------------------------------------------
# 6. Persist configuration
# --------------------------------------------------------------------------------------------------

CONFIG_ROOT = (
    Path("/content/drive/MyDrive/SPP_GAN_Research")
    / "results"
    / "notebooks"
    / "notebook_09"
    / "configuration"
)

CONFIG_ROOT.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = CONFIG_ROOT / "sppgan_statistical_guidance_configuration.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(
        PERSISTED_GUIDANCE_CONFIG,
        f,
        indent=2,
        ensure_ascii=False
    )

# --------------------------------------------------------------------------------------------------
# 7. Reload persisted configuration
# --------------------------------------------------------------------------------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    RELOADED_GUIDANCE_CONFIG = json.load(f)

# --------------------------------------------------------------------------------------------------
# 8. Persistence validation
# --------------------------------------------------------------------------------------------------

if not isinstance(RELOADED_GUIDANCE_CONFIG, dict):
    raise TypeError("Reloaded configuration is not a dictionary.")

required_top_level = {
    "configuration_version",
    "objective",
    "weights",
    "distribution_guidance",
    "moment_guidance",
    "categorical_guidance",
    "dependency_guidance",
    "correlation_representation",
    "statistical_characterization",
    "privacy",
    "execution_status",
    "provenance"
}

missing_keys = required_top_level - set(RELOADED_GUIDANCE_CONFIG.keys())

if missing_keys:
    raise ValueError(
        f"Persisted configuration is missing keys: {sorted(missing_keys)}"
    )

# --------------------------------------------------------------------------------------------------
# 9. Critical semantic validation
# --------------------------------------------------------------------------------------------------

assert (
    RELOADED_GUIDANCE_CONFIG["objective"]["lambda_stat"]
    == lambda_stat
)

assert (
    RELOADED_GUIDANCE_CONFIG["moment_guidance"]["components"]
    == ["mean", "std"]
)

assert (
    RELOADED_GUIDANCE_CONFIG["moment_guidance"]["min_max_guidance"]
    is False
)

assert (
    RELOADED_GUIDANCE_CONFIG["dependency_guidance"]["representation"]
    == "differentiable_pearson_correlation_matrix"
)

assert (
    RELOADED_GUIDANCE_CONFIG["dependency_guidance"]
    ["independent_correlation_loss"]
    is False
)

assert (
    RELOADED_GUIDANCE_CONFIG["categorical_guidance"]
    ["probability_requirements"]["silent_normalization"]
    is False
)

assert (
    RELOADED_GUIDANCE_CONFIG["privacy"]
    ["end_to_end_privacy_claim_established"]
    is False
)

# --------------------------------------------------------------------------------------------------
# 10. Final status
# --------------------------------------------------------------------------------------------------

print()
print("✓ Guidance configuration saved:")
print(f"  {CONFIG_PATH}")

print()
print("✓ Configuration persistence validation : PASS")
print("✓ Objective formulation               : PASS")
print("✓ lambda_stat                         :", lambda_stat)
print("✓ Statistical weights                 : PASS")
print("✓ Distribution guidance               : RBF-MMD²")
print("✓ Moment guidance                     : mean + std")
print("✓ Categorical guidance                : probability discrepancy")
print("✓ Dependency guidance                 : Pearson Frobenius")
print("✓ Independent correlation loss        : DISABLED")
print("✓ Privacy mechanism                    : NOT INCLUDED")
print("✓ End-to-end privacy claim             : NOT ESTABLISHED")
print()
print("SECTION 19 STATUS: PASS")

19. SAVE GUIDANCE CONFIGURATION

✓ Guidance configuration saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09/configuration/sppgan_statistical_guidance_configuration.json

✓ Configuration persistence validation : PASS
✓ Objective formulation               : PASS
✓ lambda_stat                         : 1.0
✓ Statistical weights                 : PASS
✓ Distribution guidance               : RBF-MMD²
✓ Moment guidance                     : mean + std
✓ Categorical guidance                : probability discrepancy
✓ Dependency guidance                 : Pearson Frobenius
✓ Independent correlation loss        : DISABLED
✓ Privacy mechanism                    : NOT INCLUDED
✓ End-to-end privacy claim             : NOT ESTABLISHED

SECTION 19 STATUS: PASS


In [59]:
# ==================================================================================================
# 20. SAVE GUIDANCE MANIFEST
# ==================================================================================================

print("\n" + "=" * 100)
print("20. SAVE GUIDANCE MANIFEST")
print("=" * 100)


def sha256_file(
    path,
):
    """
    Compute SHA-256 without loading the complete file into RAM.
    """

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


GUIDANCE_ARTIFACTS = [
    GUIDANCE_MODULE_PATH,
    GUIDANCE_CONFIGURATION_PATH,
]

for dataset_id in DATASET_IDS:

    GUIDANCE_ARTIFACTS.extend([
        GUIDANCE_PATHS[dataset_id],
        REFERENCE_PATHS[dataset_id],
    ])

GUIDANCE_ARTIFACT_REGISTRY = []

for path in GUIDANCE_ARTIFACTS:

    path = Path(path)

    GUIDANCE_ARTIFACT_REGISTRY.append({
        "artifact": path.name,
        "absolute_path": str(path),
        "exists": path.exists(),
        "size_bytes": (
            path.stat().st_size
            if path.exists()
            else 0
        ),
        "sha256": (
            sha256_file(path)
            if path.exists()
            else None
        ),
    })

GUIDANCE_ARTIFACT_REGISTRY_DF = pd.DataFrame(
    GUIDANCE_ARTIFACT_REGISTRY
)

GUIDANCE_ARTIFACT_REGISTRY_PATH = (
    DIRS["metadata"] /
    "sppgan_statistical_guidance_artifact_registry.csv"
)

GUIDANCE_ARTIFACT_REGISTRY_DF.to_csv(
    GUIDANCE_ARTIFACT_REGISTRY_PATH,
    index=False,
)

print(
    f"✓ Artifact registry saved:\n"
    f"  {GUIDANCE_ARTIFACT_REGISTRY_PATH}"
)

display(
    GUIDANCE_ARTIFACT_REGISTRY_DF
)


20. SAVE GUIDANCE MANIFEST
✓ Artifact registry saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09/metadata/sppgan_statistical_guidance_artifact_registry.csv


,artifact,absolute_path,exists,size_bytes,sha256
0,sppgan_statistical_guidance.py,/content/drive/MyDrive/SPP_GAN_Research/result...,True,16476,5e05725f0e6d9ee73cc5a08195b3439616396fb1209fca...
1,sppgan_statistical_guidance_configuration.json,/content/drive/MyDrive/SPP_GAN_Research/result...,True,2746,9aab6e7e4fdfa55d83d785f94006ade0569c82d58b621b...
2,adult_income_spp_gan_statistical_guidance.json,/content/drive/MyDrive/SPP_GAN_Research/data/p...,True,15491,fe8eae5bf6521eb20c437abcabbac03b771e68e8459a40...
3,adult_income_spp_gan_statistical_reference.json,/content/drive/MyDrive/SPP_GAN_Research/data/p...,True,30209,ee672bb4292985c778397970976f04ff8b9498946ca6f9...
4,bank_marketing_spp_gan_statistical_guidance.json,/content/drive/MyDrive/SPP_GAN_Research/data/p...,True,17280,32ffb00dec670b69cd6a642df4c663b9dcadcc61447286...
5,bank_marketing_spp_gan_statistical_reference.json,/content/drive/MyDrive/SPP_GAN_Research/data/p...,True,34274,12d39af9a0a0f8e225419a9bb4055951cc2f873528ea73...
6,diabetes_130us_spp_gan_statistical_guidance.json,/content/drive/MyDrive/SPP_GAN_Research/data/p...,True,30303,897ba6a575975a37dfb64fbd5a9e72ec4f3f5ec8e60fa7...
7,diabetes_130us_spp_gan_statistical_reference.json,/content/drive/MyDrive/SPP_GAN_Research/data/p...,True,134852,ae3e5b049ab2eda05d330d10027fcb0b6f8e4b11f1bde5...


In [60]:
# ==================================================================================================
# 21. COMPLETION SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("21. COMPLETION SUMMARY")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------------------------

input_validation_pass = (
    STATISTICAL_INPUT_VALIDATION_DF[
        "status"
    ]
    .eq("PASS")
    .all()
)

computation_pass = (
    GUIDANCE_COMPUTATION_DF[
        "status"
    ]
    .eq("PASS")
    .all()
)

gradient_pass = (
    GRADIENT_TEST_DF[
        "status"
    ]
    .eq("PASS")
    .all()
)

stability_pass = (
    NUMERICAL_STABILITY_DF[
        "status"
    ]
    .eq("PASS")
    .all()
)

artifacts_pass = all(
    GUIDANCE_ARTIFACT_REGISTRY_DF[
        "exists"
    ]
    .tolist()
)

ALL_VALIDATION_PASS = all([
    input_validation_pass,
    computation_pass,
    gradient_pass,
    stability_pass,
    artifacts_pass,
])

if not ALL_VALIDATION_PASS:

    raise RuntimeError(
        "Notebook 09 validation failed."
    )

# -----------------------------------------------------------------------------------------------
# Save validation tables
# -----------------------------------------------------------------------------------------------

INPUT_VALIDATION_PATH = (
    DIRS["validation"] /
    "statistical_input_validation.csv"
)

COMPUTATION_PATH = (
    DIRS["validation"] /
    "statistical_guidance_computation_tests.csv"
)

GRADIENT_PATH = (
    DIRS["validation"] /
    "statistical_guidance_gradient_tests.csv"
)

STABILITY_PATH = (
    DIRS["validation"] /
    "statistical_guidance_numerical_stability.csv"
)

STATISTICAL_INPUT_VALIDATION_DF.to_csv(
    INPUT_VALIDATION_PATH,
    index=False,
)

GUIDANCE_COMPUTATION_DF.to_csv(
    COMPUTATION_PATH,
    index=False,
)

GRADIENT_TEST_DF.to_csv(
    GRADIENT_PATH,
    index=False,
)

NUMERICAL_STABILITY_DF.to_csv(
    STABILITY_PATH,
    index=False,
)

# -----------------------------------------------------------------------------------------------
# Completion manifest
# -----------------------------------------------------------------------------------------------

COMPLETION_MANIFEST = {

    "notebook": NOTEBOOK_ID,

    "name": NOTEBOOK_NAME,

    "framework": FRAMEWORK_NAME,

    "status": "PASS",

    "project_root": str(
        PROJECT_ROOT
    ),

    "datasets_registered": len(
        DATASET_IDS
    ),

    "datasets": DATASET_IDS,

    "statistical_guidance": {
        "distribution": "PASS",
        "numerical": "PASS",
        "categorical": "PASS",
        "dependency": "PASS",
        "correlation": "PASS",
        "normalization": "PASS",
        "statistical_loss": "PASS",
    },

    "validation": {
        "statistical_inputs": "PASS",
        "guidance_computation": "PASS",
        "gradient_flow": "PASS",
        "numerical_stability": "PASS",
    },

    "training_performed": False,

    "privacy_enabled": False,

    "synthetic_generation_performed": False,

    "objective": {
        "formula": (
            "L_G = L_adv + lambda_stat * L_stat"
        ),
        "lambda_stat": (
            SPPGAN_CONFIG[
                "lambda_stat"
            ]
        ),
    },

    "statistical_loss": {
        "formula": (
            "L_stat = "
            "lambda_m * L_marg + "
            "lambda_mom * L_mom + "
            "lambda_d * L_dep + "
            "lambda_c * L_cat"
        ),
        "lambda_m": GUIDANCE_WEIGHTS[
            "lambda_m"
        ],
        "lambda_mom": GUIDANCE_WEIGHTS[
            "lambda_mom"
        ],
        "lambda_d": GUIDANCE_WEIGHTS[
            "lambda_d"
        ],
        "lambda_c": GUIDANCE_WEIGHTS[
            "lambda_c"
        ],
    },

    "artifacts": {
        "guidance_module": str(
            GUIDANCE_MODULE_PATH
        ),
        "guidance_configuration": str(
            GUIDANCE_CONFIGURATION_PATH
        ),
        "artifact_registry": str(
            GUIDANCE_ARTIFACT_REGISTRY_PATH
        ),
        "input_validation": str(
            INPUT_VALIDATION_PATH
        ),
        "computation_tests": str(
            COMPUTATION_PATH
        ),
        "gradient_tests": str(
            GRADIENT_PATH
        ),
        "numerical_stability": str(
            STABILITY_PATH
        ),
    },

    "source_dependencies": {
        "notebook_00": (
            PROJECT_ROOT /
            "config"
        ).as_posix(),
        "notebook_02": str(
            NB02_ROOT
        ),
        "notebook_03": str(
            NB03_ROOT
        ),
        "notebook_08": str(
            NB08_ROOT
        ),
    },

    "downstream": {
        "next_notebook": "10",
        "next_notebook_name": (
            "SPP-GAN Differential Privacy"
        ),
        "training_notebook": "12",
    },

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

COMPLETION_PATH = (
    DIRS["validation"] /
    "sppgan_notebook_09_completion.json"
)

with open(
    COMPLETION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        COMPLETION_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

# -----------------------------------------------------------------------------------------------
# Final display
# -----------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 09 — FINAL STATUS")
print("=" * 100)

print(
    f"Framework                       : "
    f"{FRAMEWORK_NAME}"
)

print(
    f"Datasets                        : "
    f"{len(DATASET_IDS)}"
)

print(
    f"Statistical input validation    : PASS"
)

print(
    f"Distribution guidance           : PASS"
)

print(
    f"Numerical guidance              : PASS"
)

print(
    f"Categorical guidance            : PASS"
)

print(
    f"Dependency guidance             : PASS"
)

print(
    f"Correlation guidance            : PASS"
)

print(
    f"Statistical loss                : PASS"
)

print(
    f"Gradient flow                   : PASS"
)

print(
    f"Numerical stability             : PASS"
)

print(
    f"Training                        : NOT PERFORMED"
)

print(
    f"Privacy                         : NOT ENABLED"
)

print(
    f"Synthetic generation            : NOT PERFORMED"
)

print(
    f"Overall status                  : PASS"
)

print("\nArtifacts:")

print(
    f"  Guidance module : "
    f"{DIRS['models']}"
)

print(
    f"  Configuration   : "
    f"{DIRS['configuration']}"
)

print(
    f"  Loss            : "
    f"{DIRS['loss']}"
)

print(
    f"  Metadata        : "
    f"{DIRS['metadata']}"
)

print(
    f"  Validation      : "
    f"{DIRS['validation']}"
)

print("\nNext:")

print(
    "  Notebook 10 — SPP-GAN Differential Privacy"
)

print("=" * 100)

print(
    "\n✓ NOTEBOOK 09 COMPLETED SUCCESSFULLY."
)


21. COMPLETION SUMMARY

NOTEBOOK 09 — FINAL STATUS
Framework                       : SPP-GAN
Datasets                        : 3
Statistical input validation    : PASS
Distribution guidance           : PASS
Numerical guidance              : PASS
Categorical guidance            : PASS
Dependency guidance             : PASS
Correlation guidance            : PASS
Statistical loss                : PASS
Gradient flow                   : PASS
Numerical stability             : PASS
Training                        : NOT PERFORMED
Privacy                         : NOT ENABLED
Synthetic generation            : NOT PERFORMED
Overall status                  : PASS

Artifacts:
  Guidance module : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09/models
  Configuration   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09/configuration
  Loss            : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09/loss
  Metadata        : /content/d